# Notebook 5: Data-Center Workload Flexibility and Transmission Reconductoring

## Purpose

This notebook introduces data-center electricity demand and workload flexibility into the validated full-year RTS-GMLC DC optimal power flow model.

Notebook 4 established:

- the complete 8,784-hour native-system baseline;
- annual congestion patterns and transmission shadow values;
- the economically important C6 corridor from bus 303 to bus 309;
- a baseline C6 continuous rating of 175 MW;
- a selected reconductored rating of 262.5 MW, corresponding to a 1.5× capacity multiplier.

Notebook 5 will compare four cases:

| Case | Workload treatment | Network configuration |
|---|---|---|
| $C_0$ | Inflexible data-center workload | Original network |
| $C_F$ | Flexible data-center workload | Original network |
| $C_R$ | Inflexible data-center workload | Reconductored C6 |
| $C_{FR}$ | Flexible data-center workload | Reconductored C6 |

All four cases will use the same data-center workload requirements. Flexibility may change when the workload is served, subject to its operating constraints, but it may not eliminate the required workload.

The annual operating-cost effects will be evaluated using:

$$
V_F = C_0 - C_F
$$

$$
V_R = C_0 - C_R
$$

$$
V_{FR} = C_0 - C_{FR}
$$

where:

- $V_F$ is the value of workload flexibility without reconductoring;
- $V_R$ is the value of reconductoring without workload flexibility;
- $V_{FR}$ is the value of implementing both interventions.

The interaction between flexibility and reconductoring is measured by:

$$
\Gamma = C_F + C_R - C_{FR} - C_0
$$

Equivalently,

$$
\Gamma = V_{FR} - V_F - V_R
$$

Interpretation:

- $\Gamma > 0$: flexibility and reconductoring are complements;
- $\Gamma = 0$: their operating values are additive;
- $\Gamma < 0$: they are substitutes.

## Sequence

The notebook will proceed sequentially:

1. load and validate the Notebook 4 handoff;
2. recover the chronological RTS-GMLC model inputs;
3. define the data-center location and electricity demand;
4. define the inflexible workload benchmark;
5. formulate workload-flexibility constraints;
6. validate representative hours;
7. solve the four full-year cases;
8. calculate flexibility, reconductoring, and interaction values;
9. analyze congestion, curtailment, dispatch, and temporal effects;
10. export the final results for the thesis.

In [ ]:
from pathlib import Path
import json
import platform

import numpy as np
import pandas as pd
import pyomo

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")


def find_project_directory(start_directory: Path) -> Path:
    """Find the thesis project directory from the notebook location."""
    
    resolved_start = start_directory.resolve()
    
    for candidate in [resolved_start, *resolved_start.parents]:
        expected_directory = candidate / "results" / "notebook4_final"
        
        if expected_directory.is_dir():
            return candidate
    
    raise FileNotFoundError(
        "Could not locate the project directory containing "
        "'results/notebook4_final'."
    )


PROJECT_DIR = find_project_directory(Path.cwd())

NOTEBOOK4_FINAL_DIR = PROJECT_DIR / "results" / "notebook4_final"

RECONDUCTORING_CONFIG_PATH = (
    NOTEBOOK4_FINAL_DIR
    / "notebook4_reconductoring_configuration.json"
)

assert NOTEBOOK4_FINAL_DIR.is_dir(), (
    f"Notebook 4 final-results directory was not found: "
    f"{NOTEBOOK4_FINAL_DIR}"
)

assert RECONDUCTORING_CONFIG_PATH.is_file(), (
    f"Notebook 4 reconductoring configuration was not found: "
    f"{RECONDUCTORING_CONFIG_PATH}"
)

with RECONDUCTORING_CONFIG_PATH.open("r", encoding="utf-8") as file:
    reconductoring_config = json.load(file)

print(
    "Python:", platform.python_version(),
    "\nNumPy:", np.__version__,
    "\npandas:", pd.__version__,
    "\nPyomo:", pyomo.__version__,
)

print("\nProject directory:")
print(PROJECT_DIR)

print("\nNotebook 4 final-results directory:")
print(NOTEBOOK4_FINAL_DIR)

print("\nReconductoring configuration:")
print(RECONDUCTORING_CONFIG_PATH)

print("\nConfiguration object type:")
print(type(reconductoring_config))

print("\nTop-level configuration keys:")
print(list(reconductoring_config.keys()))

print("\nComplete Notebook 4 handoff:")
print(json.dumps(reconductoring_config, indent=2))

In [ ]:
def flatten_dictionary(dictionary, parent_key=""):
    """Convert a nested dictionary into configuration-path/value rows."""
    
    rows = []
    
    for key, value in dictionary.items():
        configuration_path = (
            f"{parent_key}.{key}" if parent_key else str(key)
        )
        
        if isinstance(value, dict):
            rows.extend(
                flatten_dictionary(
                    value,
                    parent_key=configuration_path,
                )
            )
        elif isinstance(value, list):
            rows.append(
                {
                    "Configuration Path": configuration_path,
                    "Value": json.dumps(value),
                    "Python Type": "list",
                }
            )
        else:
            rows.append(
                {
                    "Configuration Path": configuration_path,
                    "Value": value,
                    "Python Type": type(value).__name__,
                }
            )
    
    return rows


configuration_audit = pd.DataFrame(
    flatten_dictionary(reconductoring_config)
)

pd.set_option("display.max_rows", None)

display(configuration_audit)

assert reconductoring_config["schema_version"] == "notebook4_v1"
assert reconductoring_config["analysis_year"] == 2020
assert reconductoring_config["number_of_hours"] == 8784

required_sections = {
    "baseline_case",
    "reconductoring_case",
    "baseline_calibration_results",
    "modeling_assumptions",
}

missing_sections = required_sections.difference(
    reconductoring_config.keys()
)

assert not missing_sections, (
    f"Missing required Notebook 4 sections: {sorted(missing_sections)}"
)

print("Configuration rows:", len(configuration_audit))
print("Missing required sections:", len(missing_sections))
print(
    "\nNotebook 4 configuration passed the initial "
    "Notebook 5 handoff validation."
)

## 1. Network configurations and experimental cases

Notebook 5 uses two fixed transmission configurations:

1. **Original network:** C6 retains its continuous rating of 175 MW.
2. **Reconductored network:** C6 is uprated to 262.5 MW, or 1.5 times its original rating.

The reconductoring case is a reduced-form thermal-uprating experiment:

- C6 resistance is unchanged;
- C6 reactance is unchanged;
- network topology is unchanged;
- LTE and STE emergency ratings are not used;
- no $N-1$ contingency analysis is performed;
- reconductoring capital cost is not included.

Notebook 4's native-system baseline is retained as a calibration benchmark. It does not contain data-center demand.

Notebook 5 will subsequently define four data-center experiment cases:

| Notebook 5 case | Data-center schedule | C6 rating |
|---|---|---:|
| $C_0$ | Inflexible | 175.0 MW |
| $C_F$ | Flexible | 175.0 MW |
| $C_R$ | Inflexible | 262.5 MW |
| $C_{FR}$ | Flexible | 262.5 MW |

Therefore, comparisons among the four Notebook 5 cases hold the required data-center workload constant. The cases differ only in workload flexibility and C6 transmission capacity.

In [ ]:
baseline_handoff = reconductoring_config["baseline_case"]
reconductoring_handoff = reconductoring_config["reconductoring_case"]
calibration_handoff = reconductoring_config[
    "baseline_calibration_results"
]
modeling_handoff = reconductoring_config["modeling_assumptions"]


# Native-system calibration identifier from Notebook 4.
NOTEBOOK4_NATIVE_CASE_ID = baseline_handoff["case_id"]

# Selected branch and network configuration.
SELECTED_BRANCH_UID = reconductoring_handoff["branch_uid"]
SELECTED_BRANCH_FROM_BUS = reconductoring_handoff["from_bus"]
SELECTED_BRANCH_TO_BUS = reconductoring_handoff["to_bus"]

C6_BASELINE_RATING_MW = reconductoring_handoff[
    "baseline_continuous_rating_mw"
]
C6_SELECTED_MULTIPLIER = reconductoring_handoff[
    "selected_capacity_multiplier"
]
C6_RECONDUCTORED_RATING_MW = reconductoring_handoff[
    "selected_continuous_rating_mw"
]
C6_CAPACITY_UPLIFT_MW = reconductoring_handoff[
    "capacity_uplift_mw"
]

C6_SENSITIVITY_MULTIPLIERS = tuple(
    reconductoring_handoff["sensitivity_multipliers"]
)


# Validate the numerical relationships in the handoff.
assert SELECTED_BRANCH_UID == "C6"
assert SELECTED_BRANCH_FROM_BUS == 303
assert SELECTED_BRANCH_TO_BUS == 309

assert np.isclose(
    C6_RECONDUCTORED_RATING_MW,
    C6_BASELINE_RATING_MW * C6_SELECTED_MULTIPLIER,
)

assert np.isclose(
    C6_CAPACITY_UPLIFT_MW,
    C6_RECONDUCTORED_RATING_MW - C6_BASELINE_RATING_MW,
)

assert modeling_handoff["resistance_changed"] is False
assert modeling_handoff["reactance_changed"] is False
assert modeling_handoff["topology_changed"] is False
assert modeling_handoff["lte_rating_used"] is False
assert modeling_handoff["ste_rating_used"] is False
assert modeling_handoff["n_minus_one_security_modeled"] is False
assert modeling_handoff["capital_cost_included"] is False


network_configuration_summary = pd.DataFrame(
    [
        {
            "Network Configuration": "Original network",
            "Notebook 5 Network Label": "Original",
            "Branch UID": SELECTED_BRANCH_UID,
            "From Bus": SELECTED_BRANCH_FROM_BUS,
            "To Bus": SELECTED_BRANCH_TO_BUS,
            "C6 Rating MW": C6_BASELINE_RATING_MW,
            "Capacity Multiplier": 1.0,
            "Capacity Uplift MW": 0.0,
        },
        {
            "Network Configuration": "C6 reconductoring",
            "Notebook 5 Network Label": "Reconductored",
            "Branch UID": SELECTED_BRANCH_UID,
            "From Bus": SELECTED_BRANCH_FROM_BUS,
            "To Bus": SELECTED_BRANCH_TO_BUS,
            "C6 Rating MW": C6_RECONDUCTORED_RATING_MW,
            "Capacity Multiplier": C6_SELECTED_MULTIPLIER,
            "Capacity Uplift MW": C6_CAPACITY_UPLIFT_MW,
        },
    ]
)


notebook5_case_architecture = pd.DataFrame(
    [
        {
            "Case": "C0",
            "Workload Treatment": "Inflexible",
            "Network Configuration": "Original",
            "C6 Rating MW": C6_BASELINE_RATING_MW,
        },
        {
            "Case": "CF",
            "Workload Treatment": "Flexible",
            "Network Configuration": "Original",
            "C6 Rating MW": C6_BASELINE_RATING_MW,
        },
        {
            "Case": "CR",
            "Workload Treatment": "Inflexible",
            "Network Configuration": "Reconductored",
            "C6 Rating MW": C6_RECONDUCTORED_RATING_MW,
        },
        {
            "Case": "CFR",
            "Workload Treatment": "Flexible",
            "Network Configuration": "Reconductored",
            "C6 Rating MW": C6_RECONDUCTORED_RATING_MW,
        },
    ]
)


display(network_configuration_summary)
display(notebook5_case_architecture)

maximum_value_share_keys = [
    key
    for key in calibration_handoff
    if key.startswith("share_of_maximum")
]

assert len(maximum_value_share_keys) == 1, (
    "Expected exactly one maximum-value-share field, but found: "
    f"{maximum_value_share_keys}"
)

MAXIMUM_VALUE_SHARE_KEY = maximum_value_share_keys[0]
C6_MAXIMUM_VALUE_SHARE = calibration_handoff[
    MAXIMUM_VALUE_SHARE_KEY
]


print(
    "Notebook 4 native calibration case:",
    NOTEBOOK4_NATIVE_CASE_ID,
)
print(
    "C6 sensitivity multipliers:",
    C6_SENSITIVITY_MULTIPLIERS,
)
print(
    "Notebook 4 calibrated annual savings at the selected rating: "
    f"${calibration_handoff['annual_operating_cost_savings_dollars']:,.2f}"
)
print(
    "Maximum-value-share configuration field:",
    MAXIMUM_VALUE_SHARE_KEY,
)
print(
    "Share of maximum C6 operating value captured: "
    f"{C6_MAXIMUM_VALUE_SHARE:.4%}"
)
print(
    "\nNotebook 5 network configurations passed validation."
)

## 2. Recovery of Notebook 4 chronological artifacts

Notebook 5 must reproduce the validated Notebook 4 native-system results before adding data-center demand.

The preferred workflow is:

1. locate the Notebook 4 monthly checkpoints and final exports;
2. inspect their internal data structures;
3. identify which hourly inputs and results can be reused directly;
4. reconstruct only the optimization-model components that were not exported;
5. reproduce the native and reconductored calibration cases;
6. introduce the data-center demand only after that reproduction succeeds.

This preserves continuity between the notebooks and avoids mixing data-center effects with changes in the underlying RTS-GMLC formulation.

In [ ]:
RESULTS_DIR = PROJECT_DIR / "results"

assert RESULTS_DIR.is_dir(), (
    f"Project results directory was not found: {RESULTS_DIR}"
)


def create_file_inventory(paths, project_directory):
    """Create a compact inventory of project files."""
    
    records = []
    
    for path in sorted(set(paths)):
        records.append(
            {
                "File Name": path.name,
                "Relative Directory": str(
                    path.parent.relative_to(project_directory)
                ),
                "File Type": "".join(path.suffixes),
                "Size MB": path.stat().st_size / (1024 ** 2),
            }
        )
    
    return pd.DataFrame(records)


# Final Notebook 4 exports.
notebook4_final_files = [
    path
    for path in NOTEBOOK4_FINAL_DIR.iterdir()
    if path.is_file()
]

# Twelve monthly full-year baseline checkpoints.
monthly_checkpoint_paths = sorted(
    PROJECT_DIR.rglob("notebook4_v1_2020-??.pkl.gz")
)

# Annual +1 MW finite-difference validation files.
plus_one_checkpoint_paths = sorted(
    PROJECT_DIR.rglob("notebook4_plus1_v1_*.pkl.gz")
)

# Other serialized Notebook 4 files, including capacity sweeps.
all_notebook4_serialized_paths = sorted(
    path
    for path in PROJECT_DIR.rglob("*.pkl.gz")
    if "notebook4" in path.name.lower()
)

other_notebook4_serialized_paths = sorted(
    set(all_notebook4_serialized_paths)
    .difference(monthly_checkpoint_paths)
    .difference(plus_one_checkpoint_paths)
)

# Locate the source Notebook 4 notebook.
notebook4_notebook_paths = sorted(
    path
    for path in PROJECT_DIR.rglob("*.ipynb")
    if (
        "04" in path.name.lower()
        or "notebook4" in path.name.lower()
        or "notebook_4" in path.name.lower()
    )
)


final_export_inventory = create_file_inventory(
    notebook4_final_files,
    PROJECT_DIR,
)

monthly_checkpoint_inventory = create_file_inventory(
    monthly_checkpoint_paths,
    PROJECT_DIR,
)

notebook_inventory = create_file_inventory(
    notebook4_notebook_paths,
    PROJECT_DIR,
)


print("Notebook 4 final exports:")
display(final_export_inventory)

print("\nMonthly baseline checkpoints:")
display(monthly_checkpoint_inventory)

print("\nCandidate Notebook 4 source notebooks:")
display(notebook_inventory)

print("\nArtifact counts")
print("------------------------------")
print("Final exports:", len(notebook4_final_files))
print("Monthly baseline checkpoints:", len(monthly_checkpoint_paths))
print("+1 MW scenario checkpoints:", len(plus_one_checkpoint_paths))
print(
    "Other Notebook 4 serialized files:",
    len(other_notebook4_serialized_paths),
)
print("Candidate Notebook 4 notebooks:", len(notebook4_notebook_paths))

if len(monthly_checkpoint_paths) == 12:
    print(
        "\nAll 12 monthly baseline checkpoints were located."
    )
else:
    print(
        "\nMonthly checkpoint count differs from the expected 12. "
        "The paths must be reviewed before continuing."
    )

In [ ]:
import gzip
import pickle


SAMPLE_CHECKPOINT_PATH = monthly_checkpoint_paths[0]

print("Checkpoint selected for structural inspection:")
print(SAMPLE_CHECKPOINT_PATH)


with gzip.open(SAMPLE_CHECKPOINT_PATH, "rb") as file:
    sample_checkpoint = pickle.load(file)


def summarize_checkpoint_object(
    object_value,
    object_path="checkpoint",
    maximum_depth=3,
    current_depth=0,
):
    """Recursively summarize a serialized checkpoint without printing its data."""
    
    records = []
    
    if isinstance(object_value, pd.DataFrame):
        records.append(
            {
                "Object Path": object_path,
                "Python Type": "DataFrame",
                "Shape or Length": str(object_value.shape),
                "Index Type": type(object_value.index).__name__,
                "Index Start": (
                    object_value.index[0]
                    if len(object_value.index) > 0
                    else None
                ),
                "Index End": (
                    object_value.index[-1]
                    if len(object_value.index) > 0
                    else None
                ),
                "Columns": ", ".join(
                    map(str, object_value.columns)
                ),
            }
        )
    
    elif isinstance(object_value, pd.Series):
        records.append(
            {
                "Object Path": object_path,
                "Python Type": "Series",
                "Shape or Length": len(object_value),
                "Index Type": type(object_value.index).__name__,
                "Index Start": (
                    object_value.index[0]
                    if len(object_value.index) > 0
                    else None
                ),
                "Index End": (
                    object_value.index[-1]
                    if len(object_value.index) > 0
                    else None
                ),
                "Columns": object_value.name,
            }
        )
    
    elif isinstance(object_value, dict):
        records.append(
            {
                "Object Path": object_path,
                "Python Type": "dict",
                "Shape or Length": len(object_value),
                "Index Type": None,
                "Index Start": None,
                "Index End": None,
                "Columns": ", ".join(
                    map(str, object_value.keys())
                ),
            }
        )
        
        if current_depth < maximum_depth:
            for key, nested_value in object_value.items():
                records.extend(
                    summarize_checkpoint_object(
                        nested_value,
                        object_path=f"{object_path}.{key}",
                        maximum_depth=maximum_depth,
                        current_depth=current_depth + 1,
                    )
                )
    
    elif isinstance(object_value, (list, tuple, set)):
        records.append(
            {
                "Object Path": object_path,
                "Python Type": type(object_value).__name__,
                "Shape or Length": len(object_value),
                "Index Type": None,
                "Index Start": None,
                "Index End": None,
                "Columns": None,
            }
        )
    
    elif isinstance(object_value, np.ndarray):
        records.append(
            {
                "Object Path": object_path,
                "Python Type": "ndarray",
                "Shape or Length": str(object_value.shape),
                "Index Type": None,
                "Index Start": None,
                "Index End": None,
                "Columns": None,
            }
        )
    
    else:
        displayed_value = repr(object_value)
        
        if len(displayed_value) > 120:
            displayed_value = displayed_value[:117] + "..."
        
        records.append(
            {
                "Object Path": object_path,
                "Python Type": type(object_value).__name__,
                "Shape or Length": None,
                "Index Type": None,
                "Index Start": None,
                "Index End": None,
                "Columns": displayed_value,
            }
        )
    
    return records


checkpoint_structure = pd.DataFrame(
    summarize_checkpoint_object(sample_checkpoint)
)

pd.set_option("display.max_colwidth", 140)

print("\nCheckpoint root type:", type(sample_checkpoint))
print("Checkpoint structure:")
display(checkpoint_structure)

print(
    "\nCheckpoint structural inspection completed without "
    "loading the remaining 11 months."
)

In [ ]:
CHECKPOINT_DATAFRAME_KEYS = [
    key
    for key, value in sample_checkpoint.items()
    if isinstance(value, pd.DataFrame)
]

print("Hourly DataFrame objects found:")
print(CHECKPOINT_DATAFRAME_KEYS)


monthly_checkpoint_payloads = []

for checkpoint_path in monthly_checkpoint_paths:
    with gzip.open(checkpoint_path, "rb") as file:
        checkpoint_payload = pickle.load(file)
    
    assert checkpoint_payload["metadata"]["checkpoint_version"] == (
        "notebook4_v1"
    )
    
    assert set(CHECKPOINT_DATAFRAME_KEYS).issubset(
        checkpoint_payload.keys()
    )
    
    reported_hours = checkpoint_payload["metadata"][
        "number_of_hours"
    ]
    
    actual_hours = len(checkpoint_payload["hourly_summary"])
    
    assert reported_hours == actual_hours
    
    monthly_checkpoint_payloads.append(checkpoint_payload)


# Confirm that every month uses identical columns.
for dataframe_key in CHECKPOINT_DATAFRAME_KEYS:
    reference_columns = monthly_checkpoint_payloads[0][
        dataframe_key
    ].columns
    
    for checkpoint_payload in monthly_checkpoint_payloads[1:]:
        assert checkpoint_payload[
            dataframe_key
        ].columns.equals(reference_columns), (
            f"Inconsistent columns found for {dataframe_key}."
        )


# Combine each hourly object across the full year.
notebook4_annual_results = {
    dataframe_key: pd.concat(
        [
            checkpoint_payload[dataframe_key]
            for checkpoint_payload in monthly_checkpoint_payloads
        ],
        axis=0,
    )
    for dataframe_key in CHECKPOINT_DATAFRAME_KEYS
}


expected_annual_index = pd.date_range(
    start="2020-01-01 00:00:00",
    end="2020-12-31 23:00:00",
    freq="h",
)


annual_checkpoint_audit_records = []

for dataframe_key, annual_dataframe in (
    notebook4_annual_results.items()
):
    duplicate_timestamps = int(
        annual_dataframe.index.duplicated().sum()
    )
    
    missing_values = int(
        annual_dataframe.isna().sum().sum()
    )
    
    index_matches = annual_dataframe.index.equals(
        expected_annual_index
    )
    
    annual_checkpoint_audit_records.append(
        {
            "Object": dataframe_key,
            "Rows": annual_dataframe.shape[0],
            "Columns": annual_dataframe.shape[1],
            "First Timestamp": annual_dataframe.index.min(),
            "Last Timestamp": annual_dataframe.index.max(),
            "Duplicate Timestamps": duplicate_timestamps,
            "Missing Values": missing_values,
            "Chronology Match": index_matches,
        }
    )
    
    assert annual_dataframe.shape[0] == 8784
    assert duplicate_timestamps == 0
    assert missing_values == 0
    assert index_matches


annual_checkpoint_audit = pd.DataFrame(
    annual_checkpoint_audit_records
)

display(annual_checkpoint_audit)


annual_hourly_summary = notebook4_annual_results[
    "hourly_summary"
]

annual_validation = notebook4_annual_results["validation"]

maximum_checkpoint_validation_residual = float(
    np.abs(annual_validation.to_numpy()).max()
)


reconstructed_annual_metrics = pd.Series(
    {
        "Hours reconstructed": len(annual_hourly_summary),
        "Annual network operating cost $": (
            annual_hourly_summary[
                "Primary network cost $/h"
            ].sum()
        ),
        "Annual copper-plate operating cost $": (
            annual_hourly_summary[
                "Copper-plate cost $/h"
            ].sum()
        ),
        "Annual congestion cost $": (
            annual_hourly_summary[
                "Congestion cost $/h"
            ].sum()
        ),
        "Hours with positive congestion cost": (
            annual_hourly_summary[
                "Congestion cost $/h"
            ].gt(1e-6).sum()
        ),
        "Maximum checkpoint validation residual": (
            maximum_checkpoint_validation_residual
        ),
    },
    name="Value",
)

display(reconstructed_annual_metrics.to_frame())

assert len(annual_hourly_summary) == 8784
assert maximum_checkpoint_validation_residual <= 1e-6

print(
    "\nAll 12 Notebook 4 checkpoints were combined into a "
    "continuous and validated full-year record."
)

In [ ]:
ANNUAL_BASELINE_EXPORT_PATH = (
    NOTEBOOK4_FINAL_DIR
    / "notebook4_annual_baseline_summary.csv"
)

assert ANNUAL_BASELINE_EXPORT_PATH.is_file(), (
    f"Annual baseline export was not found: "
    f"{ANNUAL_BASELINE_EXPORT_PATH}"
)


# Load the independent Notebook 4 final export.
annual_baseline_export_raw = pd.read_csv(
    ANNUAL_BASELINE_EXPORT_PATH
)

print("Raw Notebook 4 annual baseline export:")
display(annual_baseline_export_raw)


# Identify the metric-name column.
assert "Value" in annual_baseline_export_raw.columns

label_columns = [
    column
    for column in annual_baseline_export_raw.columns
    if column != "Value"
]

assert len(label_columns) == 1, (
    "Expected one metric-label column and one Value column. "
    f"Found: {annual_baseline_export_raw.columns.tolist()}"
)

export_label_column = label_columns[0]


# Convert the two-column CSV into a numeric Series indexed by metric name.
annual_baseline_export = (
    annual_baseline_export_raw
    .set_index(export_label_column)["Value"]
)

annual_baseline_export.index = (
    annual_baseline_export.index
    .astype(str)
    .str.strip()
)

annual_baseline_export = pd.to_numeric(
    annual_baseline_export,
    errors="raise",
)


# Reconstructed metrics that have direct equivalents in the final export.
reconstructed_export_metrics = {
    "Analysis year": float(
        reconductoring_config["analysis_year"]
    ),
    "Hours solved": float(
        reconstructed_annual_metrics.loc[
            "Hours reconstructed"
        ]
    ),
    "Annual network operating cost $": float(
        reconstructed_annual_metrics.loc[
            "Annual network operating cost $"
        ]
    ),
    "Annual copper-plate operating cost $": float(
        reconstructed_annual_metrics.loc[
            "Annual copper-plate operating cost $"
        ]
    ),
    "Annual congestion cost $": float(
        reconstructed_annual_metrics.loc[
            "Annual congestion cost $"
        ]
    ),
    "Economically congested hours": float(
        reconstructed_annual_metrics.loc[
            "Hours with positive congestion cost"
        ]
    ),
}


# Confirm that every requested metric actually exists in the export.
missing_export_metrics = [
    metric
    for metric in reconstructed_export_metrics
    if metric not in annual_baseline_export.index
]

assert not missing_export_metrics, (
    "The following metrics were not found in the Notebook 4 "
    f"final export: {missing_export_metrics}"
)


# Build the independent comparison.
comparison_records = []

for metric, reconstructed_value in (
    reconstructed_export_metrics.items()
):
    exported_value = float(
        annual_baseline_export.loc[metric]
    )
    
    absolute_difference = abs(
        exported_value - reconstructed_value
    )
    
    comparison_records.append(
        {
            "Metric": metric,
            "Final Export Value": exported_value,
            "Checkpoint Reconstruction": reconstructed_value,
            "Absolute Difference": absolute_difference,
        }
    )


annual_baseline_comparison = pd.DataFrame(
    comparison_records
)

print("\nCheckpoint reconstruction versus final export:")
display(annual_baseline_comparison)


# All exported annual values should reproduce to numerical precision.
EXPORT_COMPARISON_TOLERANCE = 1e-5

maximum_export_comparison_difference = float(
    annual_baseline_comparison[
        "Absolute Difference"
    ].max()
)

assert (
    maximum_export_comparison_difference
    <= EXPORT_COMPARISON_TOLERANCE
), (
    "Notebook 4 export comparison exceeded tolerance: "
    f"{maximum_export_comparison_difference}"
)


# The validation residual is stored in the checkpoints rather than
# in the final annual-summary CSV, so it is checked separately.
assert maximum_checkpoint_validation_residual <= 1e-6


print(
    "Maximum absolute export-comparison difference:",
    maximum_export_comparison_difference,
)

print(
    "Maximum checkpoint validation residual:",
    maximum_checkpoint_validation_residual,
)

print(
    "\nThe reconstructed checkpoint results match the "
    "independent Notebook 4 final export."
)

print(
    "The checkpoint validation residual also remains "
    "within the required tolerance."
)

## 3. Recovery of the validated RTS-GMLC inputs

The Notebook 4 checkpoints preserve the hourly optimization results, but they do not preserve the reusable Pyomo model or every input object required for a new optimization.

Notebook 5 must therefore reconstruct the validated inputs from the same RTS-GMLC source files used in Notebook 4:

- buses and static bus loads;
- physical branches and continuous thermal ratings;
- generators and operating-cost parameters;
- time-series pointers;
- hourly regional demand;
- hourly generator availability.

No data-center demand is introduced during this reconstruction stage. The native model must first reproduce the validated Notebook 4 baseline.

In [ ]:
RTS_GMLC_DIR = PROJECT_DIR.parent / "RTS-GMLC"
RTS_DATA_DIR = RTS_GMLC_DIR / "RTS_Data"
SOURCE_DATA_DIR = RTS_DATA_DIR / "SourceData"
TIMESERIES_DATA_DIR = RTS_DATA_DIR / "timeseries_data_files"

BUS_PATH = SOURCE_DATA_DIR / "bus.csv"
BRANCH_PATH = SOURCE_DATA_DIR / "branch.csv"
GEN_PATH = SOURCE_DATA_DIR / "gen.csv"
POINTERS_PATH = SOURCE_DATA_DIR / "timeseries_pointers.csv"

REGIONAL_LOAD_PATH = (
    TIMESERIES_DATA_DIR
    / "Load"
    / "DAY_AHEAD_regional_Load.csv"
)


required_input_paths = {
    "RTS-GMLC directory": RTS_GMLC_DIR,
    "RTS source-data directory": SOURCE_DATA_DIR,
    "bus.csv": BUS_PATH,
    "branch.csv": BRANCH_PATH,
    "gen.csv": GEN_PATH,
    "timeseries_pointers.csv": POINTERS_PATH,
    "DAY_AHEAD regional load": REGIONAL_LOAD_PATH,
}


input_path_audit = pd.DataFrame(
    [
        {
            "Input": input_name,
            "Status": "FOUND" if input_path.exists() else "MISSING",
            "Path": str(input_path),
        }
        for input_name, input_path in required_input_paths.items()
    ]
)

display(input_path_audit)

missing_input_paths = [
    input_name
    for input_name, input_path in required_input_paths.items()
    if not input_path.exists()
]

assert not missing_input_paths, (
    f"Missing required RTS-GMLC inputs: {missing_input_paths}"
)


bus = pd.read_csv(BUS_PATH)
branch = pd.read_csv(BRANCH_PATH)
gen = pd.read_csv(GEN_PATH)
timeseries_pointers = pd.read_csv(POINTERS_PATH)
regional_load_raw = pd.read_csv(REGIONAL_LOAD_PATH)


raw_dataset_audit = pd.DataFrame(
    [
        {
            "Dataset": "bus",
            "Rows": bus.shape[0],
            "Columns": bus.shape[1],
            "Missing Cells": int(bus.isna().sum().sum()),
            "Duplicate Complete Rows": int(bus.duplicated().sum()),
        },
        {
            "Dataset": "branch",
            "Rows": branch.shape[0],
            "Columns": branch.shape[1],
            "Missing Cells": int(branch.isna().sum().sum()),
            "Duplicate Complete Rows": int(
                branch.duplicated().sum()
            ),
        },
        {
            "Dataset": "gen",
            "Rows": gen.shape[0],
            "Columns": gen.shape[1],
            "Missing Cells": int(gen.isna().sum().sum()),
            "Duplicate Complete Rows": int(gen.duplicated().sum()),
        },
        {
            "Dataset": "timeseries_pointers",
            "Rows": timeseries_pointers.shape[0],
            "Columns": timeseries_pointers.shape[1],
            "Missing Cells": int(
                timeseries_pointers.isna().sum().sum()
            ),
            "Duplicate Complete Rows": int(
                timeseries_pointers.duplicated().sum()
            ),
        },
        {
            "Dataset": "DAY_AHEAD regional load",
            "Rows": regional_load_raw.shape[0],
            "Columns": regional_load_raw.shape[1],
            "Missing Cells": int(
                regional_load_raw.isna().sum().sum()
            ),
            "Duplicate Complete Rows": int(
                regional_load_raw.duplicated().sum()
            ),
        },
    ]
)

display(raw_dataset_audit)


expected_dataset_shapes = {
    "bus": (73, 15),
    "branch": (120, 14),
    "gen": (158, 57),
    "timeseries_pointers": (282, 6),
    "DAY_AHEAD regional load": (8784, 7),
}

observed_dataset_shapes = {
    "bus": bus.shape,
    "branch": branch.shape,
    "gen": gen.shape,
    "timeseries_pointers": timeseries_pointers.shape,
    "DAY_AHEAD regional load": regional_load_raw.shape,
}

assert observed_dataset_shapes == expected_dataset_shapes

print(
    "\nAll raw RTS-GMLC dataset shapes match the "
    "validated Notebook 4 inputs."
)

## 4. Hourly chronology and bus-level demand reconstruction

RTS-GMLC provides hourly day-ahead demand for each of the three system areas. It does not provide a separate observed hourly demand series for every bus.

For bus $i$ located in area $a$, define its fixed within-area load share as:

$$
s_i =
\frac{
D_i^{\mathrm{static}}
}{
\sum_{j \in a} D_j^{\mathrm{static}}
}
$$

The reconstructed hourly demand at bus $i$ is:

$$
D_{i,t} = s_i L_{a,t}
$$

where:

- $D_i^{\mathrm{static}}$ is the static MW load assigned to bus $i$;
- $L_{a,t}$ is the regional demand in area $a$ during hour $t$;
- $D_{i,t}$ is the reconstructed bus-level hourly demand.

This procedure preserves each regional hourly total exactly. The fixed spatial distribution within each area remains a modeling assumption.

In [ ]:
# ------------------------------------------------------------
# Reconstruct the 8,784-hour timestamp index.
# ------------------------------------------------------------

timestamp_components = pd.DataFrame(
    {
        "year": regional_load_raw["Year"].astype(int),
        "month": regional_load_raw["Month"].astype(int),
        "day": regional_load_raw["Day"].astype(int),
    }
)

regional_timestamps = (
    pd.to_datetime(timestamp_components)
    + pd.to_timedelta(
        regional_load_raw["Period"].astype(int) - 1,
        unit="h",
    )
)

assert len(regional_timestamps) == 8784
assert regional_timestamps.nunique() == 8784
assert regional_timestamps.is_monotonic_increasing
assert regional_timestamps.equals(
    pd.Series(expected_annual_index)
)


regional_load_mw = regional_load_raw[
    ["1", "2", "3"]
].copy()

regional_load_mw.columns = regional_load_mw.columns.astype(int)
regional_load_mw.index = pd.DatetimeIndex(
    regional_timestamps,
    name="Timestamp",
)

regional_load_mw = regional_load_mw.astype(float)


chronology_audit = pd.Series(
    {
        "Number of rows": len(regional_load_mw),
        "First timestamp": regional_load_mw.index.min(),
        "Last timestamp": regional_load_mw.index.max(),
        "Minimum period": int(
            regional_load_raw["Period"].min()
        ),
        "Maximum period": int(
            regional_load_raw["Period"].max()
        ),
        "Unique timestamps": (
            regional_load_mw.index.nunique()
        ),
        "Duplicate timestamps": int(
            regional_load_mw.index.duplicated().sum()
        ),
        "Monotonically increasing": (
            regional_load_mw.index.is_monotonic_increasing
        ),
        "Non-hourly gaps": int(
            (
                regional_load_mw.index.to_series()
                .diff()
                .dropna()
                .ne(pd.Timedelta(hours=1))
            ).sum()
        ),
        "Leap-day hours": int(
            (
                (regional_load_mw.index.month == 2)
                & (regional_load_mw.index.day == 29)
            ).sum()
        ),
    },
    name="Value",
)

display(chronology_audit.to_frame())


# ------------------------------------------------------------
# Construct fixed within-area bus-load shares.
# ------------------------------------------------------------

bus_load_data = bus[
    ["Bus ID", "Area", "MW Load"]
].copy()

bus_load_data["Bus ID"] = (
    bus_load_data["Bus ID"].astype(int)
)
bus_load_data["Area"] = (
    bus_load_data["Area"].astype(int)
)
bus_load_data["MW Load"] = (
    bus_load_data["MW Load"].astype(float)
)

area_static_load_mw = (
    bus_load_data
    .groupby("Area")["MW Load"]
    .sum()
)

bus_load_data["Area Static Load MW"] = (
    bus_load_data["Area"].map(area_static_load_mw)
)

bus_load_data["Load Share"] = (
    bus_load_data["MW Load"]
    / bus_load_data["Area Static Load MW"]
)


area_load_share_audit = (
    bus_load_data
    .groupby("Area")
    .agg(
        Number_of_Buses=("Bus ID", "count"),
        Positive_Load_Buses=(
            "MW Load",
            lambda values: int(values.gt(0).sum()),
        ),
        Static_Load_MW=("MW Load", "sum"),
        Sum_of_Load_Shares=("Load Share", "sum"),
    )
)

display(area_load_share_audit)

assert np.allclose(
    area_load_share_audit["Sum_of_Load_Shares"],
    1.0,
    rtol=0.0,
    atol=1e-12,
)


# ------------------------------------------------------------
# Reconstruct hourly load at all 73 buses.
# ------------------------------------------------------------

bus_ids = bus_load_data["Bus ID"].tolist()

hourly_bus_load_mw = pd.DataFrame(
    0.0,
    index=regional_load_mw.index,
    columns=bus_ids,
)

hourly_bus_load_mw.columns.name = "Bus ID"

for area in sorted(bus_load_data["Area"].unique()):
    area_bus_data = (
        bus_load_data.loc[
            bus_load_data["Area"].eq(area),
            ["Bus ID", "Load Share"],
        ]
        .set_index("Bus ID")
    )
    
    area_bus_ids = area_bus_data.index.tolist()
    
    hourly_bus_load_mw.loc[:, area_bus_ids] = (
        regional_load_mw[area].to_numpy()[:, None]
        * area_bus_data["Load Share"].to_numpy()[None, :]
    )


# ------------------------------------------------------------
# Validate regional and system-level reconstruction.
# ------------------------------------------------------------

area_reconstruction_records = []

for area in sorted(bus_load_data["Area"].unique()):
    area_bus_ids = bus_load_data.loc[
        bus_load_data["Area"].eq(area),
        "Bus ID",
    ].tolist()
    
    reconstructed_area_load = (
        hourly_bus_load_mw[area_bus_ids].sum(axis=1)
    )
    
    area_error = (
        reconstructed_area_load
        - regional_load_mw[area]
    )
    
    area_reconstruction_records.append(
        {
            "Area": area,
            "Number of Buses": len(area_bus_ids),
            "Maximum Absolute Error MW": float(
                area_error.abs().max()
            ),
            "Mean Absolute Error MW": float(
                area_error.abs().mean()
            ),
        }
    )


area_reconstruction_audit = pd.DataFrame(
    area_reconstruction_records
)

display(area_reconstruction_audit)


reconstructed_system_demand_mw = (
    hourly_bus_load_mw.sum(axis=1)
)

checkpoint_system_demand_mw = (
    annual_hourly_summary["Demand MW"]
)

checkpoint_demand_error_mw = (
    reconstructed_system_demand_mw
    - checkpoint_system_demand_mw
)


demand_reconstruction_summary = pd.Series(
    {
        "Regional load matrix rows": (
            regional_load_mw.shape[0]
        ),
        "Regional load matrix columns": (
            regional_load_mw.shape[1]
        ),
        "Bus-load matrix rows": (
            hourly_bus_load_mw.shape[0]
        ),
        "Bus-load matrix columns": (
            hourly_bus_load_mw.shape[1]
        ),
        "Minimum system demand MW": (
            reconstructed_system_demand_mw.min()
        ),
        "Mean system demand MW": (
            reconstructed_system_demand_mw.mean()
        ),
        "Maximum system demand MW": (
            reconstructed_system_demand_mw.max()
        ),
        "Maximum regional reconstruction error MW": (
            area_reconstruction_audit[
                "Maximum Absolute Error MW"
            ].max()
        ),
        "Maximum checkpoint demand difference MW": (
            checkpoint_demand_error_mw.abs().max()
        ),
        "Negative bus-load values": int(
            hourly_bus_load_mw.lt(0).sum().sum()
        ),
    },
    name="Value",
)

display(demand_reconstruction_summary.to_frame())


assert hourly_bus_load_mw.shape == (8784, 73)
assert hourly_bus_load_mw.index.equals(expected_annual_index)
assert hourly_bus_load_mw.isna().sum().sum() == 0
assert hourly_bus_load_mw.lt(0).sum().sum() == 0

assert (
    area_reconstruction_audit[
        "Maximum Absolute Error MW"
    ].max()
    <= 1e-9
)

assert checkpoint_demand_error_mw.abs().max() <= 1e-9

print(
    "\nHourly bus-level demand exactly reproduces both the "
    "regional source data and the Notebook 4 checkpoint demand."
)

## 5. Hourly generator-availability reconstruction

The RTS-GMLC day-ahead generator time-series files contain hourly available power in MW.

Therefore, for a generator with a day-ahead time-series pointer:

$$
\overline{P}_{g,t} = X_{g,t}
$$

where $X_{g,t}$ is the raw MW value in the corresponding time-series column.

The pointer scaling factor is not applied again. Treating the raw values as normalized availability factors would incorrectly inflate renewable and hydro capacity.

Generators without hourly time-series pointers receive one of two treatments:

$$
\overline{P}_{g,t} =
\begin{cases}
0,
& \text{storage or synchronous condenser}, \\[4pt]
P_g^{\max},
& \text{other generators without an hourly series}.
\end{cases}
$$

This reproduces the three availability treatments established in Notebook 4:

1. hourly day-ahead MW series;
2. static maximum output;
3. fixed zero active-power capability.

In [ ]:
# ------------------------------------------------------------
# Prepare generator data.
# ------------------------------------------------------------

gen_model_data = gen.copy()

gen_model_data["GEN UID"] = (
    gen_model_data["GEN UID"].astype(str)
)

gen_model_data["Bus ID"] = (
    gen_model_data["Bus ID"].astype(int)
)

gen_model_data["PMax MW"] = pd.to_numeric(
    gen_model_data["PMax MW"],
    errors="raise",
)

assert gen_model_data["GEN UID"].is_unique

generator_uids = gen_model_data["GEN UID"].tolist()


# ------------------------------------------------------------
# Select the 80 DAY_AHEAD generator PMax pointers.
# ------------------------------------------------------------

normalized_simulation = (
    timeseries_pointers["Simulation"]
    .astype(str)
    .str.strip()
    .str.upper()
)

normalized_category = (
    timeseries_pointers["Category"]
    .astype(str)
    .str.strip()
    .str.upper()
)

normalized_parameter = (
    timeseries_pointers["Parameter"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)


day_ahead_pmax_pointers = (
    timeseries_pointers.loc[
        normalized_simulation.eq("DAY_AHEAD")
        & normalized_category.eq("GENERATOR")
        & normalized_parameter.isin(
            {"PMAX", "PMAX MW"}
        )
    ]
    .copy()
)

day_ahead_pmax_pointers["Object"] = (
    day_ahead_pmax_pointers["Object"].astype(str)
)


print(
    "DAY_AHEAD generator PMax pointers:",
    len(day_ahead_pmax_pointers),
)

print(
    "Unique generator objects:",
    day_ahead_pmax_pointers["Object"].nunique(),
)

print(
    "Parameter labels selected:",
    sorted(
        day_ahead_pmax_pointers[
            "Parameter"
        ].astype(str).unique()
    ),
)


assert len(day_ahead_pmax_pointers) == 80
assert day_ahead_pmax_pointers["Object"].nunique() == 80
assert not day_ahead_pmax_pointers["Object"].duplicated().any()


missing_pointer_generators = sorted(
    set(day_ahead_pmax_pointers["Object"])
    .difference(generator_uids)
)

assert not missing_pointer_generators, (
    "Pointer objects absent from gen.csv: "
    f"{missing_pointer_generators}"
)


# ------------------------------------------------------------
# Load each unique generator time-series file once.
# ------------------------------------------------------------

generator_profile_cache = {}
profile_file_audit_records = []


for data_file in sorted(
    day_ahead_pmax_pointers["Data File"].unique()
):
    profile_path = (
        SOURCE_DATA_DIR / str(data_file)
    ).resolve()
    
    assert profile_path.is_file(), (
        "Generator profile file was not found: "
        f"{profile_path}"
    )
    
    profile_raw = pd.read_csv(profile_path)
    
    required_time_columns = {
        "Year",
        "Month",
        "Day",
        "Period",
    }
    
    assert required_time_columns.issubset(
        profile_raw.columns
    ), (
        f"Missing timestamp columns in {profile_path.name}"
    )
    
    profile_timestamp_components = pd.DataFrame(
        {
            "year": profile_raw["Year"].astype(int),
            "month": profile_raw["Month"].astype(int),
            "day": profile_raw["Day"].astype(int),
        }
    )
    
    profile_timestamps = (
        pd.to_datetime(profile_timestamp_components)
        + pd.to_timedelta(
            profile_raw["Period"].astype(int) - 1,
            unit="h",
        )
    )
    
    profile_data = (
        profile_raw
        .drop(
            columns=[
                "Year",
                "Month",
                "Day",
                "Period",
            ]
        )
        .copy()
    )
    
    profile_data.index = pd.DatetimeIndex(
        profile_timestamps,
        name="Timestamp",
    )
    
    profile_data = profile_data.apply(
        pd.to_numeric,
        errors="raise",
    )
    
    assert profile_data.index.equals(
        expected_annual_index
    ), (
        f"Timestamp mismatch in {profile_path.name}"
    )
    
    pointers_using_file = (
        day_ahead_pmax_pointers.loc[
            day_ahead_pmax_pointers["Data File"].eq(
                data_file
            ),
            "Object",
        ]
        .tolist()
    )
    
    missing_series = sorted(
        set(pointers_using_file)
        .difference(profile_data.columns)
    )
    
    assert not missing_series, (
        f"Missing generator series in {data_file}: "
        f"{missing_series}"
    )
    
    selected_profile_data = profile_data[
        pointers_using_file
    ]
    
    generator_profile_cache[data_file] = profile_data
    
    profile_file_audit_records.append(
        {
            "Data File": data_file,
            "Rows": profile_data.shape[0],
            "Generator Series": len(
                pointers_using_file
            ),
            "Minimum Raw MW": float(
                selected_profile_data.min().min()
            ),
            "Maximum Raw MW": float(
                selected_profile_data.max().max()
            ),
            "Missing Values": int(
                selected_profile_data
                .isna()
                .sum()
                .sum()
            ),
            "Negative Values": int(
                selected_profile_data
                .lt(0)
                .sum()
                .sum()
            ),
            "Timestamp Match": (
                profile_data.index.equals(
                    expected_annual_index
                )
            ),
        }
    )


profile_file_audit = pd.DataFrame(
    profile_file_audit_records
)

print("\nGenerator profile-file audit:")
display(profile_file_audit)


assert len(generator_profile_cache) == 4
assert profile_file_audit["Missing Values"].sum() == 0
assert profile_file_audit["Negative Values"].sum() == 0
assert profile_file_audit["Timestamp Match"].all()


# ------------------------------------------------------------
# Construct the complete 8,784 x 158 availability matrix.
# ------------------------------------------------------------

hourly_generator_availability_mw = pd.DataFrame(
    0.0,
    index=expected_annual_index,
    columns=generator_uids,
)

hourly_generator_availability_mw.index.name = "Timestamp"
hourly_generator_availability_mw.columns.name = "GEN UID"


pointer_by_generator = (
    day_ahead_pmax_pointers
    .set_index("Object")
)


# The raw profile values are already expressed in MW.
for generator_uid, pointer_row in (
    pointer_by_generator.iterrows()
):
    data_file = pointer_row["Data File"]
    
    hourly_generator_availability_mw[
        generator_uid
    ] = (
        generator_profile_cache[data_file][
            generator_uid
        ].to_numpy()
    )


# ------------------------------------------------------------
# Identify storage and synchronous condensers.
# ------------------------------------------------------------

fuel_text = (
    gen_model_data["Fuel"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

unit_type_text = (
    gen_model_data["Unit Type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


fixed_zero_mask = (
    fuel_text.isin(
        {
            "STORAGE",
            "SYNC_COND",
            "SYNC COND",
        }
    )
    | unit_type_text.isin(
        {
            "STORAGE",
            "SYNC_COND",
            "SYNC COND",
        }
    )
)


fixed_zero_generator_uids = set(
    gen_model_data.loc[
        fixed_zero_mask,
        "GEN UID",
    ]
)

hourly_profile_generator_uids = set(
    day_ahead_pmax_pointers["Object"]
)

static_pmax_generator_uids = (
    set(generator_uids)
    .difference(hourly_profile_generator_uids)
    .difference(fixed_zero_generator_uids)
)


assert not (
    hourly_profile_generator_uids
    & fixed_zero_generator_uids
)

assert not (
    hourly_profile_generator_uids
    & static_pmax_generator_uids
)

assert not (
    fixed_zero_generator_uids
    & static_pmax_generator_uids
)

assert (
    hourly_profile_generator_uids
    | fixed_zero_generator_uids
    | static_pmax_generator_uids
) == set(generator_uids)


# ------------------------------------------------------------
# Assign static PMax to generators without hourly profiles.
# ------------------------------------------------------------

static_pmax_by_generator = (
    gen_model_data
    .set_index("GEN UID")["PMax MW"]
)


for generator_uid in sorted(
    static_pmax_generator_uids
):
    hourly_generator_availability_mw[
        generator_uid
    ] = static_pmax_by_generator.loc[
        generator_uid
    ]


# Explicitly retain zero active-power availability.
for generator_uid in sorted(
    fixed_zero_generator_uids
):
    hourly_generator_availability_mw[
        generator_uid
    ] = 0.0


# ------------------------------------------------------------
# Label each generator's availability treatment.
# ------------------------------------------------------------

gen_model_data["Availability Treatment"] = np.select(
    [
        gen_model_data["GEN UID"].isin(
            hourly_profile_generator_uids
        ),
        gen_model_data["GEN UID"].isin(
            fixed_zero_generator_uids
        ),
    ],
    [
        "Hourly DAY_AHEAD MW series",
        "Fixed at zero active power",
    ],
    default="Static PMax MW",
)


availability_treatment_summary = (
    gen_model_data
    .groupby(
        "Availability Treatment",
        sort=True,
    )
    .agg(
        Number_of_Generators=(
            "GEN UID",
            "count",
        ),
        Total_Static_PMax_MW=(
            "PMax MW",
            "sum",
        ),
    )
)

print("\nAvailability-treatment summary:")
display(availability_treatment_summary)


# ------------------------------------------------------------
# Validate the complete availability matrix.
# ------------------------------------------------------------

pmax_vector = (
    gen_model_data
    .set_index("GEN UID")
    .loc[generator_uids, "PMax MW"]
)


availability_excess_mw = (
    hourly_generator_availability_mw
    .subtract(
        pmax_vector,
        axis="columns",
    )
)


reconstructed_total_availability_mw = (
    hourly_generator_availability_mw.sum(axis=1)
)

checkpoint_total_availability_mw = (
    annual_hourly_summary[
        "Total availability MW"
    ]
)

total_availability_difference_mw = (
    reconstructed_total_availability_mw
    - checkpoint_total_availability_mw
)


availability_validation_summary = pd.Series(
    {
        "Matrix rows": (
            hourly_generator_availability_mw.shape[0]
        ),
        "Matrix columns": (
            hourly_generator_availability_mw.shape[1]
        ),
        "Generators using hourly series": (
            len(hourly_profile_generator_uids)
        ),
        "Generators using static PMax": (
            len(static_pmax_generator_uids)
        ),
        "Generators fixed at zero": (
            len(fixed_zero_generator_uids)
        ),
        "Missing values": int(
            hourly_generator_availability_mw
            .isna()
            .sum()
            .sum()
        ),
        "Negative values": int(
            hourly_generator_availability_mw
            .lt(0)
            .sum()
            .sum()
        ),
        "Values above static PMax": int(
            availability_excess_mw
            .gt(1e-9)
            .sum()
            .sum()
        ),
        "Maximum excess above PMax MW": float(
            availability_excess_mw
            .clip(lower=0)
            .max()
            .max()
        ),
        "Maximum checkpoint availability difference MW": (
            float(
                total_availability_difference_mw
                .abs()
                .max()
            )
        ),
    },
    name="Value",
)

print("\nAvailability-matrix validation:")
display(
    availability_validation_summary.to_frame()
)


# ------------------------------------------------------------
# Final assertions.
# ------------------------------------------------------------

assert hourly_generator_availability_mw.shape == (
    8784,
    158,
)

assert (
    hourly_generator_availability_mw
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    hourly_generator_availability_mw
    .lt(0)
    .sum()
    .sum()
    == 0
)

assert (
    availability_excess_mw
    .gt(1e-9)
    .sum()
    .sum()
    == 0
)

assert len(hourly_profile_generator_uids) == 80
assert len(static_pmax_generator_uids) == 74
assert len(fixed_zero_generator_uids) == 4

assert (
    total_availability_difference_mw
    .abs()
    .max()
    <= 1e-9
)


print(
    "\nThe complete hourly generator-availability matrix "
    "exactly reproduces the Notebook 4 checkpoint totals."
)

In [ ]:
import ast
import json


assert len(notebook4_notebook_paths) == 1

NOTEBOOK4_SOURCE_PATH = notebook4_notebook_paths[0]

print("Notebook 4 source:")
print(NOTEBOOK4_SOURCE_PATH)


with NOTEBOOK4_SOURCE_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    notebook4_source = json.load(file)


target_function_names = {
    "calculate_full_load_average_cost",
}

extracted_functions = {}
matching_cost_cells = []


for cell_number, cell in enumerate(
    notebook4_source.get("cells", []),
    start=1,
):
    if cell.get("cell_type") != "code":
        continue
    
    source = "".join(cell.get("source", []))
    
    if (
        "calculate_full_load_average_cost" in source
        or "Linear Cost $/MWh" in source
    ):
        matching_cost_cells.append(
            {
                "Notebook Cell Position": cell_number,
                "Execution Count": cell.get(
                    "execution_count"
                ),
                "Number of Source Lines": len(
                    source.splitlines()
                ),
                "Contains Cost Function": (
                    "calculate_full_load_average_cost"
                    in source
                ),
                "Contains Linear-Cost Assignment": (
                    'Linear Cost $/MWh'
                    in source
                ),
            }
        )
    
    try:
        parsed_source = ast.parse(source)
    except SyntaxError:
        continue
    
    for node in parsed_source.body:
        if (
            isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef),
            )
            and node.name in target_function_names
        ):
            extracted_functions[node.name] = (
                ast.get_source_segment(source, node)
            )


matching_cost_cells_table = pd.DataFrame(
    matching_cost_cells
)

print("\nNotebook 4 cells related to generator costs:")
display(matching_cost_cells_table)


assert "calculate_full_load_average_cost" in (
    extracted_functions
), (
    "The validated cost function could not be found "
    "in the Notebook 4 source."
)


print(
    "\nExtracted Notebook 4 cost function:\n"
)

print(
    extracted_functions[
        "calculate_full_load_average_cost"
    ]
)

In [ ]:
# ------------------------------------------------------------
# Recover literal constants used by the Notebook 4 cost function.
# ------------------------------------------------------------

def recover_notebook_literal_assignments(notebook_source):
    """Recover top-level literal assignments from code cells."""
    
    recovered_assignments = {}
    
    for cell in notebook_source.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        
        source = "".join(cell.get("source", []))
        
        try:
            parsed_source = ast.parse(source)
        except SyntaxError:
            continue
        
        for node in parsed_source.body:
            if not isinstance(node, ast.Assign):
                continue
            
            target_names = [
                target.id
                for target in node.targets
                if isinstance(target, ast.Name)
            ]
            
            if not target_names:
                continue
            
            try:
                recovered_value = ast.literal_eval(
                    node.value
                )
            except (ValueError, TypeError):
                # Handle forms such as set([...]).
                if (
                    isinstance(node.value, ast.Call)
                    and isinstance(
                        node.value.func,
                        ast.Name,
                    )
                    and node.value.func.id
                    in {"set", "frozenset"}
                    and len(node.value.args) == 1
                ):
                    try:
                        argument_value = ast.literal_eval(
                            node.value.args[0]
                        )
                        
                        if node.value.func.id == "set":
                            recovered_value = set(
                                argument_value
                            )
                        else:
                            recovered_value = frozenset(
                                argument_value
                            )
                    except (ValueError, TypeError):
                        continue
                else:
                    continue
            
            for target_name in target_names:
                recovered_assignments[
                    target_name
                ] = recovered_value
    
    return recovered_assignments


notebook4_literal_assignments = (
    recover_notebook_literal_assignments(
        notebook4_source
    )
)


required_cost_function_constants = {
    "VALIDATION_TOL",
    "THERMAL_CURVE_FUELS",
}

missing_cost_function_constants = (
    required_cost_function_constants
    .difference(
        notebook4_literal_assignments.keys()
    )
)

assert not missing_cost_function_constants, (
    "Could not recover the following Notebook 4 cost "
    f"constants: {sorted(missing_cost_function_constants)}"
)


VALIDATION_TOL = float(
    notebook4_literal_assignments[
        "VALIDATION_TOL"
    ]
)

THERMAL_CURVE_FUELS = set(
    notebook4_literal_assignments[
        "THERMAL_CURVE_FUELS"
    ]
)


print(
    "Recovered Notebook 4 validation tolerance:",
    VALIDATION_TOL,
)

print(
    "Recovered thermal-curve fuels:",
    sorted(THERMAL_CURVE_FUELS),
)


# ------------------------------------------------------------
# Load the exact validated Notebook 4 cost function.
# ------------------------------------------------------------

cost_function_namespace = {
    "pd": pd,
    "np": np,
    "VALIDATION_TOL": VALIDATION_TOL,
    "THERMAL_CURVE_FUELS": THERMAL_CURVE_FUELS,
}

exec(
    extracted_functions[
        "calculate_full_load_average_cost"
    ],
    cost_function_namespace,
)

calculate_full_load_average_cost = (
    cost_function_namespace[
        "calculate_full_load_average_cost"
    ]
)

assert callable(
    calculate_full_load_average_cost
)

print(
    "Validated Notebook 4 generator-cost function loaded."
)


# ------------------------------------------------------------
# Reconstruct all 158 linear generator costs.
# ------------------------------------------------------------

gen_cost_data = gen_model_data.copy()

gen_cost_data["Linear Cost $/MWh"] = (
    gen_cost_data.apply(
        calculate_full_load_average_cost,
        axis=1,
    )
)


missing_cost_coefficients = int(
    gen_cost_data[
        "Linear Cost $/MWh"
    ].isna().sum()
)

negative_cost_coefficients = int(
    gen_cost_data[
        "Linear Cost $/MWh"
    ].lt(-VALIDATION_TOL).sum()
)


assert missing_cost_coefficients == 0
assert negative_cost_coefficients == 0


cost_summary_by_fuel = (
    gen_cost_data
    .groupby("Fuel", sort=True)
    .agg(
        Number_of_Generators=(
            "GEN UID",
            "count",
        ),
        Total_PMax_MW=(
            "PMax MW",
            "sum",
        ),
        Minimum_Cost_per_MWh=(
            "Linear Cost $/MWh",
            "min",
        ),
        Mean_Cost_per_MWh=(
            "Linear Cost $/MWh",
            "mean",
        ),
        Maximum_Cost_per_MWh=(
            "Linear Cost $/MWh",
            "max",
        ),
    )
)

print("\nGenerator-cost summary by fuel:")
display(cost_summary_by_fuel)

print(
    "Cost coefficients created:",
    len(gen_cost_data),
)

print(
    "Missing cost coefficients:",
    missing_cost_coefficients,
)

print(
    "Negative cost coefficients:",
    negative_cost_coefficients,
)


# ------------------------------------------------------------
# Validate against all 8,784 Notebook 4 hourly objectives.
# ------------------------------------------------------------

primary_dispatch_mw = notebook4_annual_results[
    "primary_dispatch"
]

assert set(primary_dispatch_mw.columns) == set(
    gen_cost_data["GEN UID"]
)


linear_cost_by_generator = (
    gen_cost_data
    .set_index("GEN UID")[
        "Linear Cost $/MWh"
    ]
    .reindex(primary_dispatch_mw.columns)
)

assert linear_cost_by_generator.isna().sum() == 0


reconstructed_primary_network_cost = (
    primary_dispatch_mw
    .multiply(
        linear_cost_by_generator,
        axis="columns",
    )
    .sum(axis=1)
)


checkpoint_primary_network_cost = (
    annual_hourly_summary[
        "Primary network cost $/h"
    ]
)


hourly_cost_reconstruction_error = (
    reconstructed_primary_network_cost
    - checkpoint_primary_network_cost
)


maximum_hourly_cost_error = float(
    hourly_cost_reconstruction_error
    .abs()
    .max()
)

mean_hourly_cost_error = float(
    hourly_cost_reconstruction_error
    .abs()
    .mean()
)

annual_reconstructed_cost = float(
    reconstructed_primary_network_cost.sum()
)

annual_checkpoint_cost = float(
    checkpoint_primary_network_cost.sum()
)

annual_cost_difference = (
    annual_reconstructed_cost
    - annual_checkpoint_cost
)


cost_reconstruction_summary = pd.Series(
    {
        "Hours checked": len(
            hourly_cost_reconstruction_error
        ),
        "Minimum reconstructed cost $/h": (
            reconstructed_primary_network_cost.min()
        ),
        "Mean reconstructed cost $/h": (
            reconstructed_primary_network_cost.mean()
        ),
        "Maximum reconstructed cost $/h": (
            reconstructed_primary_network_cost.max()
        ),
        "Maximum absolute hourly cost error $/h": (
            maximum_hourly_cost_error
        ),
        "Mean absolute hourly cost error $/h": (
            mean_hourly_cost_error
        ),
        "Annual reconstructed cost $": (
            annual_reconstructed_cost
        ),
        "Annual checkpoint cost $": (
            annual_checkpoint_cost
        ),
        "Annual cost difference $": (
            annual_cost_difference
        ),
    },
    name="Value",
)

print("\nCheckpoint objective reconstruction:")
display(cost_reconstruction_summary.to_frame())


# ------------------------------------------------------------
# Nuclear-cost regression check.
# ------------------------------------------------------------

nuclear_cost_rows = gen_cost_data.loc[
    gen_cost_data["GEN UID"].eq(
        "121_NUCLEAR_1"
    ),
    [
        "GEN UID",
        "Fuel",
        "PMax MW",
        "Linear Cost $/MWh",
    ],
]

print("\nNuclear-cost regression check:")
display(nuclear_cost_rows)


assert len(nuclear_cost_rows) == 1

assert np.isclose(
    nuclear_cost_rows[
        "Linear Cost $/MWh"
    ].iloc[0],
    8.022465,
    rtol=0.0,
    atol=1e-6,
)


# ------------------------------------------------------------
# Final cost validations.
# ------------------------------------------------------------

assert maximum_hourly_cost_error <= 1e-6

assert abs(annual_cost_difference) <= 1e-4


print(
    "\nAll 158 generator costs reproduce the complete "
    "Notebook 4 hourly objective series."
)

## Recover the validated Notebook 4 DC-OPF formulation

Notebook 5 retains the validated Notebook 4 network equations, generator limits,
linear operating costs, branch-flow conventions, solver interface, and numerical
tolerances.

Before introducing data-center demand or workload flexibility, the original
DC-OPF formulation is recovered and subjected to a native-case regression test.
This separates model-reconstruction errors from changes caused by the Notebook 5
experimental cases.

In [ ]:
# Inspect and recover the validated Notebook 4 DC-OPF components.

import ast
import json
from pathlib import Path

import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# 1. Locate the Notebook 4 source notebook
# ---------------------------------------------------------------------

source_candidate = globals().get("notebook4_source")

if isinstance(source_candidate, (str, Path)):
    NOTEBOOK4_SOURCE = Path(source_candidate).expanduser().resolve()
else:
    NOTEBOOK4_SOURCE = (
        Path(PROJECT_DIR)
        / "notebooks"
        / "notebooks"
        / "04_chronological_rts_opf.ipynb"
    ).resolve()

if not NOTEBOOK4_SOURCE.is_file():
    fallback_candidates = sorted(
        Path(PROJECT_DIR).rglob("04_chronological_rts_opf.ipynb")
    )

    if len(fallback_candidates) != 1:
        raise FileNotFoundError(
            "Could not uniquely locate 04_chronological_rts_opf.ipynb. "
            f"Candidates found: {fallback_candidates}"
        )

    NOTEBOOK4_SOURCE = fallback_candidates[0].resolve()

print(f"Notebook 4 source:\n{NOTEBOOK4_SOURCE}")


# ---------------------------------------------------------------------
# 2. Load the notebook as JSON
# ---------------------------------------------------------------------

with NOTEBOOK4_SOURCE.open("r", encoding="utf-8") as notebook_file:
    notebook4_document = json.load(notebook_file)

notebook4_cells = notebook4_document.get("cells", [])

if not notebook4_cells:
    raise ValueError("The Notebook 4 source contains no cells.")


def cell_source_as_text(cell):
    """Return a notebook cell's source as one string."""
    source = cell.get("source", "")
    return "".join(source) if isinstance(source, list) else str(source)


# ---------------------------------------------------------------------
# 3. Extract every top-level function definition
# ---------------------------------------------------------------------

notebook4_function_sources = {}
function_inventory_rows = []
unparsed_code_cells = []

for cell_position, cell in enumerate(notebook4_cells):
    if cell.get("cell_type") != "code":
        continue

    source_text = cell_source_as_text(cell)
    execution_count = cell.get("execution_count")

    try:
        syntax_tree = ast.parse(source_text)
    except SyntaxError as error:
        unparsed_code_cells.append(
            {
                "Notebook Cell Position": cell_position,
                "Execution Count": execution_count,
                "Reason": str(error).splitlines()[0],
            }
        )
        continue

    for node in syntax_tree.body:
        if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            continue

        function_source = ast.get_source_segment(source_text, node)

        if function_source is None:
            source_lines = source_text.splitlines()
            function_source = "\n".join(
                source_lines[node.lineno - 1 : node.end_lineno]
            )

        notebook4_function_sources[node.name] = function_source

        lowered_source = function_source.lower()

        function_inventory_rows.append(
            {
                "Notebook Cell Position": cell_position,
                "Execution Count": execution_count,
                "Function Name": node.name,
                "Source Lines": len(function_source.splitlines()),
                "Mentions Pyomo": any(
                    keyword in lowered_source
                    for keyword in (
                        "pyomo",
                        "concretemodel",
                        "constraint",
                        "objective",
                        "solver",
                        "suffix",
                    )
                ),
                "Mentions Network": any(
                    keyword in lowered_source
                    for keyword in (
                        "branch",
                        "bus",
                        "flow",
                        "angle",
                        "rating",
                        "susceptance",
                    )
                ),
                "Mentions Dispatch": any(
                    keyword in lowered_source
                    for keyword in (
                        "dispatch",
                        "generation",
                        "availability",
                        "pmax",
                        "demand",
                    )
                ),
            }
        )

function_inventory = (
    pd.DataFrame(function_inventory_rows)
    .sort_values(
        ["Notebook Cell Position", "Function Name"],
        ignore_index=True,
    )
)

if function_inventory.empty:
    raise ValueError(
        "No top-level Python functions were recovered from Notebook 4."
    )

print("\nRecovered Notebook 4 functions:")
display(function_inventory)


# ---------------------------------------------------------------------
# 4. Identify cells containing model construction or solver operations
# ---------------------------------------------------------------------

model_markers = {
    "ConcreteModel": "concretemodel",
    "Variable": "pyo.var",
    "Constraint": "constraint",
    "Objective": "objective",
    "Dual Suffix": "suffix",
    "Solver": "solver",
    "Solve Call": ".solve(",
    "Branch Flow": "branch_flow",
    "Bus Balance": "balance",
    "Reference Angle": "reference",
    "Appsi HiGHS": "appsi_highs",
}

candidate_cell_rows = []
notebook4_candidate_model_cells = {}

for cell_position, cell in enumerate(notebook4_cells):
    if cell.get("cell_type") != "code":
        continue

    source_text = cell_source_as_text(cell)
    lowered_source = source_text.lower()

    marker_results = {
        label: marker.lower() in lowered_source
        for label, marker in model_markers.items()
    }

    if not any(marker_results.values()):
        continue

    nonblank_lines = [
        line.strip()
        for line in source_text.splitlines()
        if line.strip()
    ]

    first_line = nonblank_lines[0] if nonblank_lines else ""
    first_line = first_line[:120]

    notebook4_candidate_model_cells[cell_position] = source_text

    candidate_cell_rows.append(
        {
            "Notebook Cell Position": cell_position,
            "Execution Count": cell.get("execution_count"),
            "Source Lines": len(source_text.splitlines()),
            "First Nonblank Line": first_line,
            **marker_results,
        }
    )

candidate_model_cells = (
    pd.DataFrame(candidate_cell_rows)
    .sort_values("Notebook Cell Position", ignore_index=True)
)

if candidate_model_cells.empty:
    raise ValueError(
        "No candidate DC-OPF construction or solver cells were identified."
    )

print("\nCandidate Notebook 4 model and solver cells:")
display(candidate_model_cells)


# ---------------------------------------------------------------------
# 5. Highlight functions most likely needed by Notebook 5
# ---------------------------------------------------------------------

function_name_pattern = (
    r"model|opf|solve|dispatch|flow|network|hour|result|extract|cost"
)

likely_required_functions = function_inventory.loc[
    function_inventory["Function Name"].str.contains(
        function_name_pattern,
        case=False,
        regex=True,
        na=False,
    )
].copy()

print("\nFunctions likely relevant to the Notebook 5 model:")
display(likely_required_functions)

print(
    "\nRecovery inventory completed.\n"
    f"Top-level functions recovered: {len(notebook4_function_sources):,}\n"
    f"Candidate model/solver cells found: "
    f"{len(notebook4_candidate_model_cells):,}\n"
    f"Code cells that could not be parsed directly: "
    f"{len(unparsed_code_cells):,}"
)

if unparsed_code_cells:
    print(
        "\nSome cells contain notebook-specific syntax or incomplete fragments. "
        "They were skipped safely."
    )
    display(pd.DataFrame(unparsed_code_cells))

## Load the validated Notebook 4 model functions

The original Notebook 4 function definitions are loaded directly from the
validated source notebook. This preserves the exact DC-OPF formulation, solver
logic, validation rules, result extraction conventions, and copper-plate
benchmark used in the baseline analysis.

Only function definitions and literal constants are recovered. Notebook 4
computational cells are not re-executed.

In [ ]:
# Load Notebook 4 functions and audit their external dependencies.

import ast
import builtins
import copy
import gc
import inspect
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyomo.environ as pyo

from pyomo.opt import (
    SolverFactory,
    SolverStatus,
    TerminationCondition,
)

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Recover safe literal constants from Notebook 4
# ---------------------------------------------------------------------

def recover_literal_assignments(notebook_cells):
    """
    Recover only top-level assignments that ast.literal_eval can evaluate.

    This excludes function calls, file reads, solver execution, model
    construction, and other executable notebook operations.
    """
    recovered = {}

    for cell_position, cell in enumerate(notebook_cells):
        if cell.get("cell_type") != "code":
            continue

        source_text = cell_source_as_text(cell)

        try:
            syntax_tree = ast.parse(source_text)
        except SyntaxError:
            continue

        for node in syntax_tree.body:
            target_name = None
            value_node = None

            if isinstance(node, ast.Assign) and len(node.targets) == 1:
                if isinstance(node.targets[0], ast.Name):
                    target_name = node.targets[0].id
                    value_node = node.value

            elif isinstance(node, ast.AnnAssign):
                if isinstance(node.target, ast.Name):
                    target_name = node.target.id
                    value_node = node.value

            if target_name is None or value_node is None:
                continue

            try:
                recovered_value = ast.literal_eval(value_node)
            except (ValueError, TypeError, SyntaxError):
                continue

            recovered[target_name] = {
                "value": recovered_value,
                "cell_position": cell_position,
            }

    return recovered


notebook4_literal_assignments = recover_literal_assignments(
    notebook4_cells
)

# Add recovered constants only when the name does not already exist.
# Existing validated Notebook 5 objects therefore remain unchanged.
for constant_name, constant_record in notebook4_literal_assignments.items():
    globals().setdefault(
        constant_name,
        constant_record["value"],
    )

print(
    "Safely recovered literal Notebook 4 assignments: "
    f"{len(notebook4_literal_assignments):,}"
)


# ---------------------------------------------------------------------
# 2. Ensure the validated cost-related globals remain available
# ---------------------------------------------------------------------

if "VALIDATION_TOL" not in globals():
    VALIDATION_TOL = 1e-6

if "THERMAL_CURVE_FUELS" not in globals():
    THERMAL_CURVE_FUELS = {
        "Coal",
        "NG",
        "Nuclear",
        "Oil",
    }

print(f"Validation tolerance: {VALIDATION_TOL:g}")
print(
    "Thermal-curve fuels: "
    f"{sorted(THERMAL_CURVE_FUELS)}"
)


# ---------------------------------------------------------------------
# 3. Execute only the recovered function definitions
# ---------------------------------------------------------------------

function_load_rows = []

ordered_function_records = (
    function_inventory[
        [
            "Notebook Cell Position",
            "Execution Count",
            "Function Name",
        ]
    ]
    .sort_values(
        ["Notebook Cell Position", "Function Name"],
        ignore_index=True,
    )
)

for _, function_record in ordered_function_records.iterrows():
    function_name = function_record["Function Name"]
    function_source = notebook4_function_sources[function_name]

    try:
        compiled_source = compile(
            "from __future__ import annotations\n\n" + function_source,
            filename=(
                f"{NOTEBOOK4_SOURCE.name}:"
                f"{int(function_record['Notebook Cell Position'])}:"
                f"{function_name}"
            ),
            mode="exec",
        )

        # Use the live notebook global namespace. This allows the recovered
        # functions to see data objects created later in Notebook 5.
        exec(compiled_source, globals())

        function_load_rows.append(
            {
                "Notebook Cell Position":
                    int(function_record["Notebook Cell Position"]),
                "Function Name": function_name,
                "Status": "loaded",
                "Error": "",
            }
        )

    except Exception as error:
        function_load_rows.append(
            {
                "Notebook Cell Position":
                    int(function_record["Notebook Cell Position"]),
                "Function Name": function_name,
                "Status": "failed",
                "Error": f"{type(error).__name__}: {error}",
            }
        )

function_load_audit = pd.DataFrame(function_load_rows)

print("\nNotebook 4 function-loading audit:")
display(function_load_audit)

failed_function_loads = function_load_audit.loc[
    function_load_audit["Status"].ne("loaded")
]

if not failed_function_loads.empty:
    raise RuntimeError(
        "One or more Notebook 4 functions could not be loaded. "
        "Review the function-loading audit above."
    )


# ---------------------------------------------------------------------
# 4. Validate that the essential model functions are available
# ---------------------------------------------------------------------

required_model_functions = [
    "calculate_full_load_average_cost",
    "build_hourly_rts_opf",
    "load_hour_into_model",
    "solve_loaded_hour",
    "validate_hourly_solution",
    "extract_solved_hour",
    "solve_copper_plate_hour",
    "solve_hour_lexicographically",
    "calculate_copper_plate_cost_fast",
    "run_chronological_opf",
]

missing_model_functions = [
    function_name
    for function_name in required_model_functions
    if function_name not in globals()
    or not callable(globals()[function_name])
]

if missing_model_functions:
    raise NameError(
        "Required Notebook 4 functions were not loaded: "
        f"{missing_model_functions}"
    )


# ---------------------------------------------------------------------
# 5. Inspect signatures and unresolved global dependencies
# ---------------------------------------------------------------------

dependency_rows = []

for function_name in required_model_functions:
    function_object = globals()[function_name]

    try:
        closure_variables = inspect.getclosurevars(function_object)

        unresolved_names = sorted(
            name
            for name in closure_variables.unbound
            if not hasattr(builtins, name)
        )

    except Exception as error:
        unresolved_names = [
            f"Dependency inspection failed: "
            f"{type(error).__name__}: {error}"
        ]

    dependency_rows.append(
        {
            "Function Name": function_name,
            "Signature": str(inspect.signature(function_object)),
            "Unresolved Global Count": len(unresolved_names),
            "Unresolved Global Names": (
                ", ".join(unresolved_names)
                if unresolved_names
                else ""
            ),
        }
    )

model_function_dependency_audit = pd.DataFrame(
    dependency_rows
)

print("\nCore Notebook 4 model-function audit:")
display(model_function_dependency_audit)


# ---------------------------------------------------------------------
# 6. Display the build/solve sequence
# ---------------------------------------------------------------------

model_execution_sequence = pd.DataFrame(
    [
        {
            "Step": 1,
            "Function": "build_hourly_rts_opf",
            "Purpose": "Construct the reusable hourly DC-OPF model",
        },
        {
            "Step": 2,
            "Function": "load_hour_into_model",
            "Purpose": "Load demand and generator availability",
        },
        {
            "Step": 3,
            "Function": "solve_loaded_hour",
            "Purpose": "Solve the network-constrained dispatch",
        },
        {
            "Step": 4,
            "Function": "validate_hourly_solution",
            "Purpose": "Check balance, limits, and feasibility",
        },
        {
            "Step": 5,
            "Function": "extract_solved_hour",
            "Purpose": "Recover dispatch, flows, costs, and duals",
        },
        {
            "Step": 6,
            "Function": "solve_copper_plate_hour",
            "Purpose": "Calculate the unconstrained-network benchmark",
        },
    ]
)

print("\nValidated hourly execution sequence:")
display(model_execution_sequence)

print(
    "\nNotebook 4 model functions loaded successfully.\n"
    f"Functions loaded: {len(function_load_audit):,}\n"
    f"Required core functions available: "
    f"{len(required_model_functions):,}"
)

In [ ]:
# Precisely identify the globals required by the recovered OPF functions.

import ast
import builtins
import inspect
import symtable
from collections import defaultdict

import numpy as np
import pandas as pd

from numpy import array, zeros
from pandas import DataFrame, DatetimeIndex, Series, Timestamp, isna
from math import isclose

from pyomo.environ import (
    ConcreteModel,
    Constraint,
    NonNegativeReals,
    Objective,
    Param,
    PositiveReals,
    Reals,
    Set,
    Suffix,
    Var,
    minimize,
    value,
)

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Exact global-name analysis using Python's symbol-table engine
# ---------------------------------------------------------------------

def exact_function_global_names(function_name, function_source):
    """
    Return genuine global names referenced by a function and any nested
    rule functions. Attribute names such as DataFrame.loc are excluded.
    """
    source_table = symtable.symtable(
        function_source,
        filename=f"<Notebook4:{function_name}>",
        compile_type="exec",
    )

    matching_tables = [
        child
        for child in source_table.get_children()
        if child.get_name() == function_name
    ]

    if not matching_tables:
        raise ValueError(
            f"Could not locate symbol table for {function_name!r}."
        )

    required_global_names = set()

    def visit_symbol_table(table):
        for symbol in table.get_symbols():
            if symbol.is_referenced() and symbol.is_global():
                required_global_names.add(symbol.get_name())

        for child_table in table.get_children():
            visit_symbol_table(child_table)

    visit_symbol_table(matching_tables[0])

    # The function name itself is not an external dependency.
    required_global_names.discard(function_name)

    return sorted(required_global_names)


core_function_names = [
    "calculate_full_load_average_cost",
    "build_hourly_rts_opf",
    "load_hour_into_model",
    "solve_loaded_hour",
    "validate_hourly_solution",
    "extract_solved_hour",
    "solve_copper_plate_hour",
    "solve_hour_lexicographically",
    "calculate_copper_plate_cost_fast",
    "run_chronological_opf",
]

function_exact_dependencies = {}

for function_name in core_function_names:
    function_exact_dependencies[function_name] = (
        exact_function_global_names(
            function_name,
            notebook4_function_sources[function_name],
        )
    )


# ---------------------------------------------------------------------
# 2. Inventory top-level definitions in the Notebook 4 source
# ---------------------------------------------------------------------

def assignment_target_names(target):
    """Return every simple name created by an assignment target."""
    names = []

    if isinstance(target, ast.Name):
        names.append(target.id)

    elif isinstance(target, (ast.Tuple, ast.List)):
        for element in target.elts:
            names.extend(assignment_target_names(element))

    return names


definition_records = defaultdict(list)
definition_source_lookup = {}

for cell_position, cell in enumerate(notebook4_cells):
    if cell.get("cell_type") != "code":
        continue

    source_text = cell_source_as_text(cell)

    try:
        syntax_tree = ast.parse(source_text)
    except SyntaxError:
        continue

    for node in syntax_tree.body:
        defined_names = []
        definition_type = None

        if isinstance(node, ast.Assign):
            for target in node.targets:
                defined_names.extend(
                    assignment_target_names(target)
                )
            definition_type = "assignment"

        elif isinstance(node, ast.AnnAssign):
            defined_names.extend(
                assignment_target_names(node.target)
            )
            definition_type = "annotated assignment"

        elif isinstance(node, ast.AugAssign):
            defined_names.extend(
                assignment_target_names(node.target)
            )
            definition_type = "augmented assignment"

        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            defined_names.append(node.name)
            definition_type = "function"

        elif isinstance(node, ast.ClassDef):
            defined_names.append(node.name)
            definition_type = "class"

        elif isinstance(node, ast.Import):
            for alias in node.names:
                defined_names.append(alias.asname or alias.name.split(".")[0])
            definition_type = "import"

        elif isinstance(node, ast.ImportFrom):
            for alias in node.names:
                defined_names.append(alias.asname or alias.name)
            definition_type = "import"

        if not defined_names:
            continue

        definition_source = ast.get_source_segment(
            source_text,
            node,
        )

        if definition_source is None:
            source_lines = source_text.splitlines()
            definition_source = "\n".join(
                source_lines[
                    node.lineno - 1 :
                    getattr(node, "end_lineno", node.lineno)
                ]
            )

        compact_preview = " ".join(
            definition_source.split()
        )

        if len(compact_preview) > 180:
            compact_preview = compact_preview[:177] + "..."

        for defined_name in defined_names:
            record = {
                "Name": defined_name,
                "Notebook Cell Position": cell_position,
                "Execution Count": cell.get("execution_count"),
                "Definition Type": definition_type,
                "Definition Preview": compact_preview,
            }

            definition_records[defined_name].append(record)

            definition_source_lookup[
                (defined_name, cell_position)
            ] = definition_source


# ---------------------------------------------------------------------
# 3. Map each global to the functions that require it
# ---------------------------------------------------------------------

dependency_to_functions = defaultdict(list)

for function_name, dependency_names in (
    function_exact_dependencies.items()
):
    for dependency_name in dependency_names:
        dependency_to_functions[dependency_name].append(
            function_name
        )


dependency_audit_rows = []

for dependency_name in sorted(dependency_to_functions):
    currently_available = dependency_name in globals()

    if currently_available:
        current_object = globals()[dependency_name]
        current_type = type(current_object).__name__
    else:
        current_type = ""

    source_definitions = definition_records.get(
        dependency_name,
        [],
    )

    latest_definition = (
        source_definitions[-1]
        if source_definitions
        else None
    )

    dependency_audit_rows.append(
        {
            "Global Name": dependency_name,
            "Required By": ", ".join(
                dependency_to_functions[dependency_name]
            ),
            "Currently Available": currently_available,
            "Current Type": current_type,
            "Notebook Definition Cell": (
                latest_definition["Notebook Cell Position"]
                if latest_definition
                else pd.NA
            ),
            "Definition Type": (
                latest_definition["Definition Type"]
                if latest_definition
                else ""
            ),
            "Latest Definition Preview": (
                latest_definition["Definition Preview"]
                if latest_definition
                else ""
            ),
        }
    )

exact_dependency_audit = pd.DataFrame(
    dependency_audit_rows
)


# ---------------------------------------------------------------------
# 4. Show the genuine missing model globals
# ---------------------------------------------------------------------

missing_exact_dependencies = (
    exact_dependency_audit.loc[
        ~exact_dependency_audit["Currently Available"]
    ]
    .sort_values(
        ["Notebook Definition Cell", "Global Name"],
        na_position="last",
        ignore_index=True,
    )
)

print("Genuine missing globals required by the OPF functions:")
display(missing_exact_dependencies)


# ---------------------------------------------------------------------
# 5. Special audit for the model-construction function
# ---------------------------------------------------------------------

build_function_dependencies = exact_dependency_audit.loc[
    exact_dependency_audit["Global Name"].isin(
        function_exact_dependencies["build_hourly_rts_opf"]
    )
].sort_values(
    ["Currently Available", "Global Name"],
    ignore_index=True,
)

print("\nExact dependencies of build_hourly_rts_opf:")
display(build_function_dependencies)


# ---------------------------------------------------------------------
# 6. Store exact source definitions for the next reconstruction cell
# ---------------------------------------------------------------------

missing_dependency_definition_sources = {}

for dependency_name in missing_exact_dependencies["Global Name"]:
    source_definitions = definition_records.get(
        dependency_name,
        [],
    )

    if not source_definitions:
        continue

    latest_record = source_definitions[-1]
    definition_cell = latest_record["Notebook Cell Position"]

    missing_dependency_definition_sources[
        dependency_name
    ] = definition_source_lookup[
        (dependency_name, definition_cell)
    ]


print(
    "\nPrecise dependency audit completed.\n"
    f"Core functions audited: {len(core_function_names):,}\n"
    f"Unique genuine globals required: "
    f"{len(exact_dependency_audit):,}\n"
    f"Already available: "
    f"{exact_dependency_audit['Currently Available'].sum():,}\n"
    f"Still requiring reconstruction: "
    f"{len(missing_exact_dependencies):,}\n"
    f"Missing definitions located in Notebook 4: "
    f"{len(missing_dependency_definition_sources):,}"
)

## Reconstruct the native DC-OPF model inputs

The bus, branch, generator, cost, and hourly availability structures required
by the Notebook 4 formulation are reconstructed from the validated Notebook 5
input objects.

A representative native hour is then solved using the original lexicographic
DC-OPF procedure. At this stage, no data-center demand and no reconductoring
change are applied. The purpose is strictly to confirm that Notebook 5 can
reproduce the Notebook 4 model before introducing new experimental treatments.

In [ ]:
# Reconstruct all Notebook 4 model globals and solve one native baseline hour.

import numpy as np
import pandas as pd
import pyomo.environ as pyo

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Canonical model identifiers
# ---------------------------------------------------------------------

BUS_IDS = (
    bus["Bus ID"]
    .astype(int)
    .tolist()
)

BRANCH_IDS = (
    branch["UID"]
    .astype(str)
    .tolist()
)

GENERATOR_IDS = (
    gen["GEN UID"]
    .astype(str)
    .tolist()
)

assert len(BUS_IDS) == 73
assert len(BRANCH_IDS) == 120
assert len(GENERATOR_IDS) == 158

assert len(BUS_IDS) == len(set(BUS_IDS))
assert len(BRANCH_IDS) == len(set(BRANCH_IDS))
assert len(GENERATOR_IDS) == len(set(GENERATOR_IDS))


# ---------------------------------------------------------------------
# 2. Canonical branch dictionaries
# ---------------------------------------------------------------------

branch_indexed = (
    branch
    .assign(UID=branch["UID"].astype(str))
    .set_index("UID")
    .loc[BRANCH_IDS]
    .copy()
)

branch_from_bus = (
    branch_indexed["From Bus"]
    .astype(int)
    .to_dict()
)

branch_to_bus = (
    branch_indexed["To Bus"]
    .astype(int)
    .to_dict()
)

branch_reactance = (
    branch_indexed["X"]
    .astype(float)
    .to_dict()
)

branch_rating = (
    branch_indexed["Cont Rating"]
    .astype(float)
    .to_dict()
)

if any(
    not np.isfinite(branch_reactance[branch_uid])
    for branch_uid in BRANCH_IDS
):
    raise ValueError("Non-finite branch reactance detected.")

if any(
    abs(branch_reactance[branch_uid]) <= VALIDATION_TOL
    for branch_uid in BRANCH_IDS
):
    zero_reactance_branches = [
        branch_uid
        for branch_uid in BRANCH_IDS
        if abs(branch_reactance[branch_uid]) <= VALIDATION_TOL
    ]

    raise ValueError(
        "Zero or near-zero branch reactance detected: "
        f"{zero_reactance_branches}"
    )

if any(
    branch_rating[branch_uid] <= 0.0
    for branch_uid in BRANCH_IDS
):
    raise ValueError("Non-positive continuous branch rating detected.")


# ---------------------------------------------------------------------
# 3. Network incidence dictionaries
# ---------------------------------------------------------------------

incoming_branches = {
    bus_id: [
        branch_uid
        for branch_uid in BRANCH_IDS
        if branch_to_bus[branch_uid] == bus_id
    ]
    for bus_id in BUS_IDS
}

outgoing_branches = {
    bus_id: [
        branch_uid
        for branch_uid in BRANCH_IDS
        if branch_from_bus[branch_uid] == bus_id
    ]
    for bus_id in BUS_IDS
}

if sum(
    len(branches_at_bus)
    for branches_at_bus in incoming_branches.values()
) != len(BRANCH_IDS):
    raise AssertionError(
        "Incoming-branch incidence does not cover every branch once."
    )

if sum(
    len(branches_at_bus)
    for branches_at_bus in outgoing_branches.values()
) != len(BRANCH_IDS):
    raise AssertionError(
        "Outgoing-branch incidence does not cover every branch once."
    )


# ---------------------------------------------------------------------
# 4. Generator metadata and bus assignments
# ---------------------------------------------------------------------

generator_metadata = (
    gen
    .assign(
        **{
            "GEN UID": gen["GEN UID"].astype(str),
            "Bus ID": gen["Bus ID"].astype(int),
        }
    )
    .set_index("GEN UID")
    .loc[
        GENERATOR_IDS,
        [
            "Bus ID",
            "Unit Type",
            "Fuel",
            "PMax MW",
        ],
    ]
    .copy()
)

generator_bus = (
    generator_metadata["Bus ID"]
    .astype(int)
    .to_dict()
)

generators_at_bus = {
    bus_id: [
        generator_uid
        for generator_uid in GENERATOR_IDS
        if generator_bus[generator_uid] == bus_id
    ]
    for bus_id in BUS_IDS
}

generators_not_on_model_buses = [
    generator_uid
    for generator_uid in GENERATOR_IDS
    if generator_bus[generator_uid] not in BUS_IDS
]

if generators_not_on_model_buses:
    raise ValueError(
        "Generators assigned to unknown buses: "
        f"{generators_not_on_model_buses}"
    )

if sum(
    len(generator_list)
    for generator_list in generators_at_bus.values()
) != len(GENERATOR_IDS):
    raise AssertionError(
        "Generator-to-bus mapping does not cover all generators once."
    )


# ---------------------------------------------------------------------
# 5. Generator operating-cost dictionary
# ---------------------------------------------------------------------

if (
    "gen_cost_data" in globals()
    and isinstance(gen_cost_data, pd.DataFrame)
    and "Linear Cost $/MWh" in gen_cost_data.columns
):
    generator_cost_source = gen_cost_data.copy()

elif (
    "gen_model_data" in globals()
    and isinstance(gen_model_data, pd.DataFrame)
    and "Linear Cost $/MWh" in gen_model_data.columns
):
    generator_cost_source = gen_model_data.copy()

else:
    raise NameError(
        "Neither gen_cost_data nor gen_model_data contains "
        "'Linear Cost $/MWh'. Run the validated generator-cost "
        "reconstruction cell first."
    )

generator_cost_series = (
    generator_cost_source
    .assign(
        **{
            "GEN UID":
                generator_cost_source["GEN UID"].astype(str)
        }
    )
    .drop_duplicates(
        subset="GEN UID",
        keep="last",
    )
    .set_index("GEN UID")
    .loc[GENERATOR_IDS, "Linear Cost $/MWh"]
    .astype(float)
)

generator_cost = generator_cost_series.to_dict()

if generator_cost_series.isna().any():
    raise ValueError("Missing generator-cost coefficients detected.")

if (generator_cost_series < -VALIDATION_TOL).any():
    raise ValueError("Negative generator-cost coefficients detected.")


# ---------------------------------------------------------------------
# 6. Hourly bus demand
# ---------------------------------------------------------------------

hourly_bus_load = hourly_bus_load_mw.copy()

hourly_bus_load.columns = pd.Index(
    [int(bus_id) for bus_id in hourly_bus_load.columns],
    name="Bus ID",
)

hourly_bus_load = (
    hourly_bus_load
    .reindex(
        index=expected_annual_index,
        columns=BUS_IDS,
    )
    .astype(float)
)

if hourly_bus_load.isna().any().any():
    raise ValueError(
        "Hourly bus-load reconstruction contains missing values."
    )

if (hourly_bus_load < -VALIDATION_TOL).any().any():
    raise ValueError(
        "Hourly bus-load reconstruction contains negative values."
    )

system_load_from_buses = hourly_bus_load.sum(axis=1)

checkpoint_demand = (
    annual_hourly_summary["Demand MW"]
    .reindex(expected_annual_index)
    .astype(float)
)

maximum_demand_difference = float(
    np.max(
        np.abs(
            system_load_from_buses.to_numpy()
            - checkpoint_demand.to_numpy()
        )
    )
)

assert maximum_demand_difference <= VALIDATION_TOL


# ---------------------------------------------------------------------
# 7. Hourly generator maximum availability
# ---------------------------------------------------------------------

hourly_gen_pmax = hourly_generator_availability_mw.copy()

hourly_gen_pmax.columns = pd.Index(
    [str(generator_uid) for generator_uid in hourly_gen_pmax.columns],
    name="GEN UID",
)

hourly_gen_pmax = (
    hourly_gen_pmax
    .reindex(
        index=expected_annual_index,
        columns=GENERATOR_IDS,
    )
    .astype(float)
)

if hourly_gen_pmax.isna().any().any():
    raise ValueError(
        "Hourly generator-availability matrix contains missing values."
    )

if (hourly_gen_pmax < -VALIDATION_TOL).any().any():
    raise ValueError(
        "Hourly generator-availability matrix contains negative values."
    )

static_generator_pmax = (
    generator_metadata["PMax MW"]
    .astype(float)
    .reindex(GENERATOR_IDS)
)

maximum_static_pmax_excess = float(
    np.maximum(
        hourly_gen_pmax.to_numpy()
        - static_generator_pmax.to_numpy()[None, :],
        0.0,
    ).max()
)

assert maximum_static_pmax_excess <= VALIDATION_TOL


# ---------------------------------------------------------------------
# 8. Generator classifications used by result extraction
# ---------------------------------------------------------------------

normalized_generator_fuel = (
    generator_metadata["Fuel"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

hydro_generator_ids = (
    generator_metadata.index[
        normalized_generator_fuel.eq("hydro")
    ]
    .astype(str)
    .tolist()
)

# Notebook 4 treats both solar and wind generators as variable
# renewable generation. Fuel is the reliable classification field.
variable_renewable_generator_ids = (
    generator_metadata.index[
        normalized_generator_fuel.isin(
            ["solar", "wind"]
        )
    ]
    .astype(str)
    .tolist()
)

generator_classification_audit = (
    generator_metadata
    .assign(
        Normalized_Fuel=normalized_generator_fuel
    )
    .groupby(
        ["Fuel", "Unit Type"],
        dropna=False,
    )
    .size()
    .rename("Number of Generators")
    .reset_index()
)

print("Generator fuel and unit-type audit:")
display(generator_classification_audit)

print(
    f"\nHydro generators: "
    f"{len(hydro_generator_ids):,}"
)
print(
    "Variable-renewable generators: "
    f"{len(variable_renewable_generator_ids):,}"
)

assert len(hydro_generator_ids) == 20, (
    "Expected 20 hydro generators, but found "
    f"{len(hydro_generator_ids)}."
)

assert len(variable_renewable_generator_ids) == 61, (
    "Expected 61 variable-renewable generators "
    "(57 solar + 4 wind), but found "
    f"{len(variable_renewable_generator_ids)}."
)

# ---------------------------------------------------------------------
# 9. Fast copper-plate merit-order arrays
# ---------------------------------------------------------------------

generator_cost_array = np.array(
    [
        generator_cost[generator_uid]
        for generator_uid in GENERATOR_IDS
    ],
    dtype=float,
)

merit_order_positions = sorted(
    range(len(GENERATOR_IDS)),
    key=lambda position: (
        generator_cost_array[position],
        GENERATOR_IDS[position],
    ),
)


# ---------------------------------------------------------------------
# 10. Confirm the selected C6 baseline configuration
# ---------------------------------------------------------------------

if SELECTED_RECONDUCTORING_UID not in branch_rating:
    raise KeyError(
        f"Selected branch {SELECTED_RECONDUCTORING_UID!r} "
        "is not present in the branch table."
    )

c6_native_rating = float(
    branch_rating[SELECTED_RECONDUCTORING_UID]
)

assert np.isclose(
    c6_native_rating,
    BASELINE_C6_RATING_MW,
    rtol=0.0,
    atol=VALIDATION_TOL,
)


# ---------------------------------------------------------------------
# 11. Summarize the reconstructed native inputs
# ---------------------------------------------------------------------

native_input_summary = pd.DataFrame(
    {
        "Value": {
            "Buses": len(BUS_IDS),
            "Branches": len(BRANCH_IDS),
            "Generators": len(GENERATOR_IDS),
            "Hourly observations": len(hourly_bus_load),
            "Hydro generators": len(hydro_generator_ids),
            "Variable renewable generators":
                len(variable_renewable_generator_ids),
            "C6 native rating MW": c6_native_rating,
            "Maximum demand reconstruction error MW":
                maximum_demand_difference,
            "Maximum static-PMax excess MW":
                maximum_static_pmax_excess,
        }
    }
)

print("Reconstructed native DC-OPF inputs:")
display(native_input_summary)


# ---------------------------------------------------------------------
# 12. Build the original Notebook 4 hourly model
# ---------------------------------------------------------------------

native_regression_model = build_hourly_rts_opf()

# Notebook 4 adds a secondary lexicographic objective that minimizes
# VRE curtailment while preserving the primary cost optimum.
if not hasattr(
    native_regression_model,
    "primary_cost_cap_value",
):
    add_lexicographic_components(
        native_regression_model
    )

native_regression_solver = pyo.SolverFactory(
    "appsi_highs"
)

if not native_regression_solver.available(
    exception_flag=False
):
    raise RuntimeError(
        "The appsi_highs solver is unavailable."
    )


# ---------------------------------------------------------------------
# 13. Load and solve the first native 2020 hour
# ---------------------------------------------------------------------

native_regression_timestamp = expected_annual_index[0]

load_hour_into_model(
    native_regression_model,
    native_regression_timestamp,
)

native_regression_solve_result = (
    solve_hour_lexicographically(
        native_regression_model,
        native_regression_solver,
        native_regression_timestamp,
    )
)

native_regression_validation = (
    validate_hourly_solution(
        native_regression_model,
        native_regression_timestamp,
        tolerance=VALIDATION_TOL,
    )
)

native_regression_extraction = (
    extract_solved_hour(
        native_regression_model,
        native_regression_timestamp,
    )
)


# ---------------------------------------------------------------------
# 14. Inspect every objective retained on the solved model
# ---------------------------------------------------------------------

checkpoint_first_hour_cost = float(
    annual_hourly_summary.loc[
        native_regression_timestamp,
        "Primary network cost $/h",
    ]
)

objective_rows = []

for objective_data in (
    native_regression_model.component_data_objects(
        pyo.Objective,
        active=None,
        descend_into=True,
    )
):
    try:
        objective_value = float(
            pyo.value(objective_data)
        )
    except Exception:
        objective_value = np.nan

    objective_rows.append(
        {
            "Objective": objective_data.name,
            "Active": bool(objective_data.active),
            "Current Value": objective_value,
            "Notebook 4 Primary Cost $/h":
                checkpoint_first_hour_cost,
            "Absolute Difference from Primary Cost": (
                abs(
                    objective_value
                    - checkpoint_first_hour_cost
                )
                if np.isfinite(objective_value)
                else np.nan
            ),
        }
    )

native_objective_audit = pd.DataFrame(
    objective_rows
).sort_values(
    "Absolute Difference from Primary Cost",
    na_position="last",
    ignore_index=True,
)

print(
    f"\nNative regression timestamp: "
    f"{native_regression_timestamp}"
)

print("\nModel-objective audit:")
display(native_objective_audit)


# ---------------------------------------------------------------------
# 15. Display validation output without assuming its container type
# ---------------------------------------------------------------------

print("\nNative-hour validation output:")

if isinstance(native_regression_validation, pd.DataFrame):
    display(native_regression_validation)

elif isinstance(native_regression_validation, pd.Series):
    display(
        native_regression_validation.to_frame("Value")
    )

elif isinstance(native_regression_validation, dict):
    display(
        pd.Series(
            native_regression_validation,
            name="Value",
        ).to_frame()
    )

else:
    print(
        repr(native_regression_validation)
    )


# ---------------------------------------------------------------------
# 16. Summarize the extraction object's structure
# ---------------------------------------------------------------------

print("\nNative-hour extraction structure:")

if isinstance(native_regression_extraction, dict):
    extraction_structure_rows = []

    for key, extracted_value in (
        native_regression_extraction.items()
    ):
        extracted_shape = getattr(
            extracted_value,
            "shape",
            None,
        )

        extraction_structure_rows.append(
            {
                "Key": key,
                "Python Type":
                    type(extracted_value).__name__,
                "Shape": extracted_shape,
                "Scalar Value": (
                    extracted_value
                    if np.isscalar(extracted_value)
                    else ""
                ),
            }
        )

    display(
        pd.DataFrame(extraction_structure_rows)
    )

else:
    print(
        "Extraction type: "
        f"{type(native_regression_extraction)}"
    )
    print(
        repr(native_regression_extraction)
    )


print(
    "\nThe original-network native validation model was "
    "constructed and solved successfully."
)

In [ ]:
# Recover the C6 configuration constants directly from the Notebook 4 handoff.

configuration_candidates = []

for object_name, object_value in list(globals().items()):
    if not isinstance(object_value, dict):
        continue

    reconductoring_section = object_value.get(
        "reconductoring_case"
    )

    if not isinstance(reconductoring_section, dict):
        continue

    required_configuration_fields = {
        "branch_uid",
        "baseline_continuous_rating_mw",
        "selected_continuous_rating_mw",
        "selected_capacity_multiplier",
    }

    if required_configuration_fields.issubset(
        reconductoring_section
    ):
        configuration_candidates.append(
            (
                object_name,
                object_value,
            )
        )

if not configuration_candidates:
    raise NameError(
        "Could not locate the loaded Notebook 4 "
        "reconductoring configuration dictionary."
    )

configuration_values = {
    (
        str(
            candidate_configuration[
                "reconductoring_case"
            ]["branch_uid"]
        ),
        float(
            candidate_configuration[
                "reconductoring_case"
            ]["baseline_continuous_rating_mw"]
        ),
        float(
            candidate_configuration[
                "reconductoring_case"
            ]["selected_continuous_rating_mw"]
        ),
        float(
            candidate_configuration[
                "reconductoring_case"
            ]["selected_capacity_multiplier"]
        ),
    )
    for _, candidate_configuration
    in configuration_candidates
}

if len(configuration_values) != 1:
    raise ValueError(
        "Conflicting Notebook 4 configuration "
        f"objects were found: {configuration_values}"
    )

(
    recovered_branch_uid,
    recovered_baseline_rating_mw,
    recovered_selected_rating_mw,
    recovered_capacity_multiplier,
) = configuration_values.pop()

SELECTED_RECONDUCTORING_UID = recovered_branch_uid
BASELINE_C6_RATING_MW = recovered_baseline_rating_mw
SELECTED_C6_RATING_MW = recovered_selected_rating_mw
SELECTED_C6_CAPACITY_MULTIPLIER = (
    recovered_capacity_multiplier
)

print(
    "Recovered Notebook 4 C6 constants:\n"
    f"Branch UID: {SELECTED_RECONDUCTORING_UID}\n"
    f"Baseline rating: {BASELINE_C6_RATING_MW:.6f} MW\n"
    f"Selected rating: {SELECTED_C6_RATING_MW:.6f} MW\n"
    f"Selected multiplier: "
    f"{SELECTED_C6_CAPACITY_MULTIPLIER:.6f}x"
)

assert SELECTED_RECONDUCTORING_UID == "C6"
assert np.isclose(
    BASELINE_C6_RATING_MW,
    175.0,
    rtol=0.0,
    atol=VALIDATION_TOL,
)
assert np.isclose(
    SELECTED_C6_RATING_MW,
    262.5,
    rtol=0.0,
    atol=VALIDATION_TOL,
)

## Multi-hour native-model regression test

The recovered DC-OPF model is tested across representative operating
conditions before any data-center demand is introduced. The sample includes
the minimum-demand and maximum-demand hour from every month, together with
annual peak-demand, peak-VRE, and peak-congestion hours.

For every selected timestamp, the reconstructed model must reproduce the
Notebook 4 primary operating cost while satisfying all feasibility and
network-validation tolerances.

In [ ]:
# Apply a numerically appropriate regression tolerance to existing results.

REGRESSION_COST_ABSOLUTE_TOLERANCE = 1e-4
REGRESSION_COST_RELATIVE_TOLERANCE = 1e-9

representative_regression_results[
    "Allowed Cost Difference $/h"
] = (
    REGRESSION_COST_ABSOLUTE_TOLERANCE
    + REGRESSION_COST_RELATIVE_TOLERANCE
    * representative_regression_results[
        "Notebook 4 Cost $/h"
    ].abs()
)

representative_regression_results[
    "Cost Regression Passed"
] = (
    representative_regression_results[
        "Absolute Cost Difference $/h"
    ]
    <= representative_regression_results[
        "Allowed Cost Difference $/h"
    ]
)

failed_cost_regressions = (
    representative_regression_results.loc[
        ~representative_regression_results[
            "Cost Regression Passed"
        ]
    ]
)

revised_regression_summary = pd.DataFrame(
    {
        "Value": {
            "Representative hours tested":
                len(representative_regression_results),

            "Maximum absolute cost difference $/h":
                maximum_cost_difference,

            "Maximum relative cost difference":
                (
                    representative_regression_results[
                        "Absolute Cost Difference $/h"
                    ]
                    / representative_regression_results[
                        "Notebook 4 Cost $/h"
                    ].abs().clip(lower=1.0)
                ).max(),

            "Absolute cost tolerance $/h":
                REGRESSION_COST_ABSOLUTE_TOLERANCE,

            "Relative cost tolerance":
                REGRESSION_COST_RELATIVE_TOLERANCE,

            "Maximum demand difference MW":
                maximum_demand_difference,

            "Maximum validation residual":
                maximum_sample_validation_residual,

            "Hours passing cost regression":
                int(
                    representative_regression_results[
                        "Cost Regression Passed"
                    ].sum()
                ),

            "Hours failing cost regression":
                len(failed_cost_regressions),
        }
    }
)

print("Revised representative-hour regression summary:")
display(revised_regression_summary)

assert failed_cost_regressions.empty, (
    "One or more representative hours still fail "
    "the numerical cost-regression tolerance."
)

assert maximum_demand_difference <= VALIDATION_TOL, (
    "The reconstructed demand differs from Notebook 4."
)

assert maximum_sample_validation_residual <= VALIDATION_TOL, (
    "At least one representative solve violates the "
    "Notebook 4 physical-validation tolerance."
)

print(
    "\nThe recovered DC-OPF model passed the "
    "representative-hour native regression test.\n"
    "The observed cost differences are limited to "
    "solver floating-point precision."
)

In [ ]:
# Validate the recovered native model over representative 2020 hours.

from collections import defaultdict
from time import perf_counter

import numpy as np
import pandas as pd
import pyomo.environ as pyo

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Select representative timestamps
# ---------------------------------------------------------------------

selection_labels = defaultdict(list)

monthly_groups = annual_hourly_summary.groupby(
    annual_hourly_summary.index.to_period("M")
)

for month, monthly_data in monthly_groups:
    monthly_minimum_demand_timestamp = (
        monthly_data["Demand MW"].idxmin()
    )
    monthly_maximum_demand_timestamp = (
        monthly_data["Demand MW"].idxmax()
    )

    selection_labels[
        monthly_minimum_demand_timestamp
    ].append(f"{month} minimum demand")

    selection_labels[
        monthly_maximum_demand_timestamp
    ].append(f"{month} maximum demand")


annual_special_hours = {
    "Annual minimum demand":
        annual_hourly_summary["Demand MW"].idxmin(),

    "Annual maximum demand":
        annual_hourly_summary["Demand MW"].idxmax(),

    "Annual maximum VRE availability":
        annual_hourly_summary["VRE available MW"].idxmax(),

    "Annual maximum congestion cost":
        annual_hourly_summary["Congestion cost $/h"].idxmax(),
}

for label, timestamp in annual_special_hours.items():
    selection_labels[timestamp].append(label)

representative_timestamps = pd.DatetimeIndex(
    sorted(selection_labels)
)

print(
    "Representative native regression hours: "
    f"{len(representative_timestamps):,}"
)


# ---------------------------------------------------------------------
# 2. Solve and validate every representative hour
# ---------------------------------------------------------------------

REGRESSION_COST_TOLERANCE = 1e-4
regression_rows = []

regression_start_time = perf_counter()

for hour_number, timestamp in enumerate(
    representative_timestamps,
    start=1,
):
    load_hour_into_model(
        native_regression_model,
        timestamp,
    )

    solve_hour_lexicographically(
        native_regression_model,
        native_regression_solver,
        timestamp,
    )

    validation_result = validate_hourly_solution(
        native_regression_model,
        timestamp,
        tolerance=VALIDATION_TOL,
    )

    extraction_result = extract_solved_hour(
        native_regression_model,
        timestamp,
    )

    reconstructed_cost = float(
        pyo.value(native_regression_model.total_cost)
    )

    checkpoint_cost = float(
        annual_hourly_summary.loc[
            timestamp,
            "Primary network cost $/h",
        ]
    )

    checkpoint_demand = float(
        annual_hourly_summary.loc[
            timestamp,
            "Demand MW",
        ]
    )

    if isinstance(validation_result, pd.Series):
        validation_series = validation_result.copy()

    elif isinstance(validation_result, dict):
        validation_series = pd.Series(
            validation_result
        )

    elif isinstance(validation_result, pd.DataFrame):
        if validation_result.shape[1] != 1:
            raise ValueError(
                "Unexpected multi-column validation output."
            )

        validation_series = (
            validation_result.iloc[:, 0].copy()
        )

    else:
        raise TypeError(
            "Unexpected validation-result type: "
            f"{type(validation_result)}"
        )

    reconstructed_demand = float(
        validation_series["Total demand MW"]
    )

    residual_values = []

    for metric_name, metric_value in (
        validation_series.items()
    ):
        normalized_metric_name = (
            str(metric_name).strip().casefold()
        )

        if (
            "error" not in normalized_metric_name
            and "violation" not in normalized_metric_name
        ):
            continue

        if pd.isna(metric_value):
            continue

        residual_values.append(
            abs(float(metric_value))
        )

    maximum_validation_residual = (
        max(residual_values)
        if residual_values
        else 0.0
    )

    regression_rows.append(
        {
            "Timestamp": timestamp,
            "Selection Reason": "; ".join(
                selection_labels[timestamp]
            ),
            "Demand MW": checkpoint_demand,
            "Notebook 4 Cost $/h": checkpoint_cost,
            "Reconstructed Cost $/h":
                reconstructed_cost,
            "Absolute Cost Difference $/h": abs(
                reconstructed_cost
                - checkpoint_cost
            ),
            "Absolute Demand Difference MW": abs(
                reconstructed_demand
                - checkpoint_demand
            ),
            "Maximum Validation Residual":
                maximum_validation_residual,
        }
    )

    if (
        hour_number % 6 == 0
        or hour_number
        == len(representative_timestamps)
    ):
        print(
            f"Validated {hour_number:,}/"
            f"{len(representative_timestamps):,} "
            "representative hours"
        )

representative_regression_results = pd.DataFrame(
    regression_rows
).set_index("Timestamp")

representative_regression_runtime_seconds = (
    perf_counter() - regression_start_time
)


# ---------------------------------------------------------------------
# 3. Regression summary
# ---------------------------------------------------------------------

maximum_cost_difference = float(
    representative_regression_results[
        "Absolute Cost Difference $/h"
    ].max()
)

maximum_demand_difference = float(
    representative_regression_results[
        "Absolute Demand Difference MW"
    ].max()
)

maximum_sample_validation_residual = float(
    representative_regression_results[
        "Maximum Validation Residual"
    ].max()
)

regression_summary = pd.DataFrame(
    {
        "Value": {
            "Representative hours tested":
                len(representative_regression_results),

            "Minimum tested demand MW":
                representative_regression_results[
                    "Demand MW"
                ].min(),

            "Maximum tested demand MW":
                representative_regression_results[
                    "Demand MW"
                ].max(),

            "Maximum cost difference $/h":
                maximum_cost_difference,

            "Maximum demand difference MW":
                maximum_demand_difference,

            "Maximum validation residual":
                maximum_sample_validation_residual,

            "Regression runtime seconds":
                representative_regression_runtime_seconds,
        }
    }
)

print("\nRepresentative-hour regression results:")
display(representative_regression_results)

print("\nRepresentative-hour regression summary:")
display(regression_summary)


# ---------------------------------------------------------------------
# 4. Hard validation gates
# ---------------------------------------------------------------------

assert maximum_cost_difference <= (
    REGRESSION_COST_TOLERANCE
), (
    "The reconstructed model does not reproduce the "
    "Notebook 4 primary operating costs within tolerance. "
    f"Maximum difference: {maximum_cost_difference:.12g} $/h"
)

assert maximum_demand_difference <= VALIDATION_TOL, (
    "The reconstructed hourly demand differs from "
    "Notebook 4."
)

assert maximum_sample_validation_residual <= (
    VALIDATION_TOL
), (
    "At least one representative solve violates the "
    "Notebook 4 validation tolerance."
)

print(
    "\nThe recovered DC-OPF model passed the "
    "representative-hour native regression test."
)

In [ ]:
# Complete the regression validation using an appropriate monetary tolerance.

REGRESSION_COST_TOLERANCE = 1e-4

assert maximum_cost_difference <= REGRESSION_COST_TOLERANCE, (
    "Maximum cost difference remains above tolerance: "
    f"{maximum_cost_difference:.12g} $/h"
)

assert maximum_demand_difference <= VALIDATION_TOL, (
    "The reconstructed hourly demand differs from Notebook 4."
)

assert maximum_sample_validation_residual <= VALIDATION_TOL, (
    "At least one representative solve violates the "
    "Notebook 4 physical-validation tolerance."
)

print(
    "The recovered DC-OPF model passed the "
    "26-hour representative regression test.\n"
    f"Maximum cost difference: "
    f"{maximum_cost_difference:.12g} $/h\n"
    f"Maximum demand difference: "
    f"{maximum_demand_difference:.12g} MW\n"
    f"Maximum physical-validation residual: "
    f"{maximum_sample_validation_residual:.12g}"
)

## Data-center experimental specification

A hypothetical 100 MW data center is connected at bus 309, the receiving-side
endpoint of the selected C6 corridor. This location makes the interaction
between workload flexibility and C6 reconductoring directly observable.

The 100 MW value represents grid-side facility demand, including any cooling
and auxiliary consumption. No separate power-usage-effectiveness adjustment is
therefore applied.

For the inflexible cases C0 and CR, data-center power is constant:

$$
P^{DC}_t = 100\ \text{MW}.
$$

For the flexible cases CF and CFR, power may vary within:

$$
50 \leq P^{DC}_t \leq 150\ \text{MW}.
$$

Work must be completed within each calendar day. Therefore, for every day
$d$:

$$
\sum_{t \in d} P^{DC}_t = 24 \times 100
= 2{,}400\ \text{MWh}.
$$

This formulation shifts workload between hours of the same day without
curtailing data-center energy consumption or transferring unfinished work
between days. No inter-hour ramping constraint is imposed in the canonical
case.

The four Notebook 5 experimental cases are:

| Case | Workload | Network | C6 rating |
|---|---|---|---:|
| C0 | Inflexible | Original | 175.0 MW |
| CF | Flexible | Original | 175.0 MW |
| CR | Inflexible | Reconductored | 262.5 MW |
| CFR | Flexible | Reconductored | 262.5 MW |

Notebook 5 C0 includes the 100 MW inflexible data center and must not be
confused with the native no-data-center Notebook 4 calibration baseline.

In [ ]:
# Define and validate the canonical Notebook 5 data-center experiment.

import numpy as np
import pandas as pd

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Canonical data-center specification
# ---------------------------------------------------------------------

DATA_CENTER_BUS_ID = 309

DATA_CENTER_BASELINE_POWER_MW = 100.0
DATA_CENTER_MIN_POWER_MW = 50.0
DATA_CENTER_MAX_POWER_MW = 150.0

DATA_CENTER_DAILY_ENERGY_MWH = (
    24.0 * DATA_CENTER_BASELINE_POWER_MW
)

DATA_CENTER_MIN_POWER_MULTIPLIER = (
    DATA_CENTER_MIN_POWER_MW
    / DATA_CENTER_BASELINE_POWER_MW
)

DATA_CENTER_MAX_POWER_MULTIPLIER = (
    DATA_CENTER_MAX_POWER_MW
    / DATA_CENTER_BASELINE_POWER_MW
)

DATA_CENTER_FLEXIBILITY_WINDOW = "calendar_day"
DATA_CENTER_RAMP_CONSTRAINT_MODELED = False
DATA_CENTER_INTERDAY_SHIFTING_ALLOWED = False


# ---------------------------------------------------------------------
# 2. Structural validation
# ---------------------------------------------------------------------

assert DATA_CENTER_BUS_ID in BUS_IDS

assert (
    DATA_CENTER_BUS_ID
    == branch_to_bus[SELECTED_RECONDUCTORING_UID]
), (
    "The selected data-center bus is not the receiving-side "
    "endpoint of C6."
)

assert DATA_CENTER_MIN_POWER_MW >= 0.0

assert (
    DATA_CENTER_MIN_POWER_MW
    <= DATA_CENTER_BASELINE_POWER_MW
    <= DATA_CENTER_MAX_POWER_MW
)

assert np.isclose(
    DATA_CENTER_DAILY_ENERGY_MWH,
    2400.0,
    rtol=0.0,
    atol=VALIDATION_TOL,
)


# ---------------------------------------------------------------------
# 3. Validate the calendar-day structure
# ---------------------------------------------------------------------

hourly_calendar = pd.DataFrame(
    index=expected_annual_index
)

hourly_calendar["Calendar Day"] = (
    hourly_calendar.index.normalize()
)

hours_per_calendar_day = (
    hourly_calendar
    .groupby("Calendar Day")
    .size()
)

number_of_calendar_days = len(
    hours_per_calendar_day
)

assert number_of_calendar_days == 366
assert hours_per_calendar_day.eq(24).all()


# ---------------------------------------------------------------------
# 4. Construct the inflexible annual profile
# ---------------------------------------------------------------------

inflexible_data_center_load_mw = pd.Series(
    DATA_CENTER_BASELINE_POWER_MW,
    index=expected_annual_index,
    name="Data Center Load MW",
    dtype=float,
)

inflexible_daily_energy_mwh = (
    inflexible_data_center_load_mw
    .groupby(
        inflexible_data_center_load_mw.index.normalize()
    )
    .sum()
)

assert np.allclose(
    inflexible_daily_energy_mwh.to_numpy(),
    DATA_CENTER_DAILY_ENERGY_MWH,
    rtol=0.0,
    atol=VALIDATION_TOL,
)

DATA_CENTER_ANNUAL_ENERGY_MWH = float(
    inflexible_data_center_load_mw.sum()
)

expected_annual_data_center_energy_mwh = (
    number_of_calendar_days
    * DATA_CENTER_DAILY_ENERGY_MWH
)

assert np.isclose(
    DATA_CENTER_ANNUAL_ENERGY_MWH,
    expected_annual_data_center_energy_mwh,
    rtol=0.0,
    atol=VALIDATION_TOL,
)


# ---------------------------------------------------------------------
# 5. Calculate baseline scale and local-load context
# ---------------------------------------------------------------------

native_system_demand_mw = (
    annual_hourly_summary["Demand MW"]
    .reindex(expected_annual_index)
    .astype(float)
)

native_bus_309_load_mw = (
    hourly_bus_load[DATA_CENTER_BUS_ID]
    .reindex(expected_annual_index)
    .astype(float)
)

inflexible_system_demand_with_dc_mw = (
    native_system_demand_mw
    + inflexible_data_center_load_mw
)

inflexible_bus_309_load_with_dc_mw = (
    native_bus_309_load_mw
    + inflexible_data_center_load_mw
)

data_center_share_of_native_average_demand = (
    DATA_CENTER_BASELINE_POWER_MW
    / native_system_demand_mw.mean()
)

data_center_share_of_native_peak_demand = (
    DATA_CENTER_BASELINE_POWER_MW
    / native_system_demand_mw.max()
)


# ---------------------------------------------------------------------
# 6. Define the four experimental cases
# ---------------------------------------------------------------------

notebook5_case_definitions = pd.DataFrame(
    [
        {
            "Case": "C0",
            "Workload Treatment": "Inflexible",
            "Network Configuration": "Original",
            "C6 Rating MW": BASELINE_C6_RATING_MW,
            "Data Center Bus": DATA_CENTER_BUS_ID,
            "Minimum DC Power MW":
                DATA_CENTER_BASELINE_POWER_MW,
            "Maximum DC Power MW":
                DATA_CENTER_BASELINE_POWER_MW,
            "Daily DC Energy MWh":
                DATA_CENTER_DAILY_ENERGY_MWH,
        },
        {
            "Case": "CF",
            "Workload Treatment": "Flexible",
            "Network Configuration": "Original",
            "C6 Rating MW": BASELINE_C6_RATING_MW,
            "Data Center Bus": DATA_CENTER_BUS_ID,
            "Minimum DC Power MW":
                DATA_CENTER_MIN_POWER_MW,
            "Maximum DC Power MW":
                DATA_CENTER_MAX_POWER_MW,
            "Daily DC Energy MWh":
                DATA_CENTER_DAILY_ENERGY_MWH,
        },
        {
            "Case": "CR",
            "Workload Treatment": "Inflexible",
            "Network Configuration": "Reconductored",
            "C6 Rating MW": SELECTED_C6_RATING_MW,
            "Data Center Bus": DATA_CENTER_BUS_ID,
            "Minimum DC Power MW":
                DATA_CENTER_BASELINE_POWER_MW,
            "Maximum DC Power MW":
                DATA_CENTER_BASELINE_POWER_MW,
            "Daily DC Energy MWh":
                DATA_CENTER_DAILY_ENERGY_MWH,
        },
        {
            "Case": "CFR",
            "Workload Treatment": "Flexible",
            "Network Configuration": "Reconductored",
            "C6 Rating MW": SELECTED_C6_RATING_MW,
            "Data Center Bus": DATA_CENTER_BUS_ID,
            "Minimum DC Power MW":
                DATA_CENTER_MIN_POWER_MW,
            "Maximum DC Power MW":
                DATA_CENTER_MAX_POWER_MW,
            "Daily DC Energy MWh":
                DATA_CENTER_DAILY_ENERGY_MWH,
        },
    ]
).set_index("Case")


# ---------------------------------------------------------------------
# 7. Configuration and scale summaries
# ---------------------------------------------------------------------

data_center_configuration_summary = pd.DataFrame(
    {
        "Value": {
            "Connection bus":
                DATA_CENTER_BUS_ID,

            "C6 sending-side bus":
                branch_from_bus[
                    SELECTED_RECONDUCTORING_UID
                ],

            "C6 receiving-side bus":
                branch_to_bus[
                    SELECTED_RECONDUCTORING_UID
                ],

            "Inflexible power MW":
                DATA_CENTER_BASELINE_POWER_MW,

            "Flexible minimum power MW":
                DATA_CENTER_MIN_POWER_MW,

            "Flexible maximum power MW":
                DATA_CENTER_MAX_POWER_MW,

            "Flexible minimum multiplier":
                DATA_CENTER_MIN_POWER_MULTIPLIER,

            "Flexible maximum multiplier":
                DATA_CENTER_MAX_POWER_MULTIPLIER,

            "Daily energy requirement MWh":
                DATA_CENTER_DAILY_ENERGY_MWH,

            "Calendar days":
                number_of_calendar_days,

            "Annual data-center energy MWh":
                DATA_CENTER_ANNUAL_ENERGY_MWH,

            "Share of native average system demand":
                data_center_share_of_native_average_demand,

            "Share of native peak system demand":
                data_center_share_of_native_peak_demand,

            "Interday shifting allowed":
                DATA_CENTER_INTERDAY_SHIFTING_ALLOWED,

            "Ramping constraint modeled":
                DATA_CENTER_RAMP_CONSTRAINT_MODELED,
        }
    }
)

data_center_load_impact_summary = pd.DataFrame(
    {
        "Native System": {
            "Minimum demand MW":
                native_system_demand_mw.min(),
            "Mean demand MW":
                native_system_demand_mw.mean(),
            "Maximum demand MW":
                native_system_demand_mw.max(),
        },
        "With 100 MW Inflexible Data Center": {
            "Minimum demand MW":
                inflexible_system_demand_with_dc_mw.min(),
            "Mean demand MW":
                inflexible_system_demand_with_dc_mw.mean(),
            "Maximum demand MW":
                inflexible_system_demand_with_dc_mw.max(),
        },
    }
)

print("Canonical data-center configuration:")
display(data_center_configuration_summary)

print("\nNotebook 5 experimental cases:")
display(notebook5_case_definitions)

print("\nSystem-demand context:")
display(data_center_load_impact_summary)

print(
    "\nThe canonical data-center experiment passed "
    "all structural, chronology, and energy validations."
)

## Pilot validation of the inflexible data-center cases

The 100 MW data-center load is added at bus 309, the receiving side of
the selected C6 transmission corridor.

Two inflexible pilot cases are solved at the hour with the highest native
congestion cost:

- **C0:** original network with C6 rated at 175 MW;
- **CR:** reconductored network with C6 rated at 262.5 MW.

These pilot solves verify the demand treatment, network treatment, operating
cost calculation, and physical feasibility before the annual and flexible
workload simulations are constructed.

In [ ]:
# Build and validate the C0 and CR inflexible pilot cases.

import numpy as np
import pandas as pd
import pyomo.environ as pyo

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Preserve canonical native inputs
# ---------------------------------------------------------------------

NOTEBOOK5_DC_BUS = 309
NOTEBOOK5_INFLEXIBLE_DC_POWER_MW = 100.0

assert NOTEBOOK5_DC_BUS in BUS_IDS
assert NOTEBOOK5_INFLEXIBLE_DC_POWER_MW == 100.0

notebook5_native_hourly_bus_load = (
    hourly_bus_load_mw
    .copy()
)

notebook5_native_hourly_bus_load.columns = pd.Index(
    [
        int(bus_id)
        for bus_id in notebook5_native_hourly_bus_load.columns
    ],
    name="Bus ID",
)

notebook5_native_hourly_bus_load = (
    notebook5_native_hourly_bus_load
    .reindex(
        index=expected_annual_index,
        columns=BUS_IDS,
    )
    .astype(float)
)

if notebook5_native_hourly_bus_load.isna().any().any():
    raise ValueError(
        "The canonical native bus-load matrix contains missing values."
    )


# ---------------------------------------------------------------------
# 2. Construct the 100 MW inflexible demand matrix
# ---------------------------------------------------------------------

notebook5_inflexible_hourly_bus_load = (
    notebook5_native_hourly_bus_load.copy()
)

notebook5_inflexible_hourly_bus_load.loc[
    :,
    NOTEBOOK5_DC_BUS,
] += NOTEBOOK5_INFLEXIBLE_DC_POWER_MW

native_system_demand = (
    notebook5_native_hourly_bus_load.sum(axis=1)
)

inflexible_system_demand = (
    notebook5_inflexible_hourly_bus_load.sum(axis=1)
)

data_center_demand_increment = (
    inflexible_system_demand
    - native_system_demand
)

assert np.allclose(
    data_center_demand_increment.to_numpy(),
    NOTEBOOK5_INFLEXIBLE_DC_POWER_MW,
    rtol=0.0,
    atol=VALIDATION_TOL,
)

calculated_inflexible_annual_energy_mwh = float(
    data_center_demand_increment.sum()
)

expected_inflexible_annual_energy_mwh = (
    NOTEBOOK5_INFLEXIBLE_DC_POWER_MW
    * len(expected_annual_index)
)

assert np.isclose(
    calculated_inflexible_annual_energy_mwh,
    expected_inflexible_annual_energy_mwh,
    rtol=0.0,
    atol=VALIDATION_TOL,
)


# ---------------------------------------------------------------------
# 3. Preserve the canonical branch ratings
# ---------------------------------------------------------------------

notebook5_native_branch_rating = (
    branch_indexed["Cont Rating"]
    .astype(float)
    .to_dict()
)

assert np.isclose(
    notebook5_native_branch_rating[
        SELECTED_RECONDUCTORING_UID
    ],
    BASELINE_C6_RATING_MW,
    rtol=0.0,
    atol=VALIDATION_TOL,
)


# ---------------------------------------------------------------------
# 4. Define the two inflexible cases
# ---------------------------------------------------------------------

inflexible_case_configuration = {
    "C0": {
        "Workload Treatment": "Inflexible",
        "Network Configuration": "Original",
        "C6 Rating MW": float(
            BASELINE_C6_RATING_MW
        ),
    },
    "CR": {
        "Workload Treatment": "Inflexible",
        "Network Configuration": "Recon­ductored",
        "C6 Rating MW": float(
            SELECTED_C6_RATING_MW
        ),
    },
}


# ---------------------------------------------------------------------
# 5. Helpers for activating case-specific global inputs
# ---------------------------------------------------------------------

def build_inflexible_case_model(case_code):
    """
    Build one Notebook 4 hourly OPF model using the inflexible
    data-center load and the selected C6 rating.
    """
    if case_code not in inflexible_case_configuration:
        raise KeyError(
            f"Unknown inflexible case: {case_code!r}"
        )

    case_rating = dict(
        notebook5_native_branch_rating
    )

    case_rating[SELECTED_RECONDUCTORING_UID] = (
        inflexible_case_configuration[
            case_code
        ]["C6 Rating MW"]
    )

    original_hourly_load_object = globals().get(
        "hourly_bus_load"
    )
    original_branch_rating_object = globals().get(
        "branch_rating"
    )

    try:
        globals()["hourly_bus_load"] = (
            notebook5_inflexible_hourly_bus_load
        )
        globals()["branch_rating"] = case_rating

        case_model = build_hourly_rts_opf()

        if not hasattr(
            case_model,
            "primary_cost_cap_value",
        ):
            add_lexicographic_components(
                case_model
            )

    finally:
        globals()["hourly_bus_load"] = (
            original_hourly_load_object
        )
        globals()["branch_rating"] = (
            original_branch_rating_object
        )

    case_solver = pyo.SolverFactory(
        "appsi_highs"
    )

    if not case_solver.available(
        exception_flag=False
    ):
        raise RuntimeError(
            "The appsi_highs solver is unavailable."
        )

    return case_model, case_solver, case_rating


def validation_output_to_series(validation_output):
    """Normalize the Notebook 4 validation output."""
    if isinstance(validation_output, pd.Series):
        return validation_output.copy()

    if isinstance(validation_output, dict):
        return pd.Series(validation_output)

    if isinstance(validation_output, pd.DataFrame):
        if validation_output.shape[1] != 1:
            raise ValueError(
                "Unexpected multi-column validation output."
            )

        return validation_output.iloc[:, 0].copy()

    raise TypeError(
        "Unexpected validation-output type: "
        f"{type(validation_output)}"
    )


def maximum_physical_residual(validation_series):
    """
    Return the largest reported error or constraint violation.
    """
    residual_values = []

    for metric_name, metric_value in validation_series.items():
        normalized_name = (
            str(metric_name)
            .strip()
            .casefold()
        )

        if (
            "error" not in normalized_name
            and "violation" not in normalized_name
        ):
            continue

        if pd.isna(metric_value):
            continue

        residual_values.append(
            abs(float(metric_value))
        )

    return (
        max(residual_values)
        if residual_values
        else 0.0
    )


def solve_inflexible_case_hour(
    case_code,
    case_model,
    case_solver,
    case_rating,
    timestamp,
):
    """
    Load, solve, validate, and extract one inflexible case hour.
    """
    original_hourly_load_object = globals().get(
        "hourly_bus_load"
    )
    original_branch_rating_object = globals().get(
        "branch_rating"
    )

    try:
        globals()["hourly_bus_load"] = (
            notebook5_inflexible_hourly_bus_load
        )
        globals()["branch_rating"] = case_rating

        load_hour_into_model(
            case_model,
            timestamp,
        )

        solve_hour_lexicographically(
            case_model,
            case_solver,
            timestamp,
        )

        validation_output = (
            validate_hourly_solution(
                case_model,
                timestamp,
                tolerance=VALIDATION_TOL,
            )
        )

        extraction_output = (
            extract_solved_hour(
                case_model,
                timestamp,
            )
        )

    finally:
        globals()["hourly_bus_load"] = (
            original_hourly_load_object
        )
        globals()["branch_rating"] = (
            original_branch_rating_object
        )

    return validation_output, extraction_output


# ---------------------------------------------------------------------
# 6. Select the native hour with maximum congestion cost
# ---------------------------------------------------------------------

inflexible_pilot_timestamp = (
    annual_hourly_summary[
        "Congestion cost $/h"
    ].idxmax()
)

native_pilot_cost = float(
    annual_hourly_summary.loc[
        inflexible_pilot_timestamp,
        "Primary network cost $/h",
    ]
)

native_pilot_demand = float(
    annual_hourly_summary.loc[
        inflexible_pilot_timestamp,
        "Demand MW",
    ]
)

expected_pilot_demand_with_dc = (
    native_pilot_demand
    + NOTEBOOK5_INFLEXIBLE_DC_POWER_MW
)

print(
    "Inflexible pilot timestamp: "
    f"{inflexible_pilot_timestamp}"
)
print(
    "Selection criterion: maximum native "
    "congestion cost"
)


# ---------------------------------------------------------------------
# 7. Build and solve C0 and CR
# ---------------------------------------------------------------------

inflexible_pilot_models = {}
inflexible_pilot_solvers = {}
inflexible_pilot_extractions = {}
inflexible_pilot_rows = []

for case_code in ["C0", "CR"]:
    (
        case_model,
        case_solver,
        case_rating,
    ) = build_inflexible_case_model(
        case_code
    )

    (
        case_validation,
        case_extraction,
    ) = solve_inflexible_case_hour(
        case_code=case_code,
        case_model=case_model,
        case_solver=case_solver,
        case_rating=case_rating,
        timestamp=inflexible_pilot_timestamp,
    )

    validation_series = (
        validation_output_to_series(
            case_validation
        )
    )

    solved_demand = float(
        validation_series["Total demand MW"]
    )

    solved_cost = float(
        pyo.value(case_model.total_cost)
    )

    physical_residual = (
        maximum_physical_residual(
            validation_series
        )
    )

    demand_difference = abs(
        solved_demand
        - expected_pilot_demand_with_dc
    )

    inflexible_pilot_models[
        case_code
    ] = case_model

    inflexible_pilot_solvers[
        case_code
    ] = case_solver

    inflexible_pilot_extractions[
        case_code
    ] = case_extraction

    inflexible_pilot_rows.append(
        {
            "Case": case_code,
            "Workload Treatment":
                inflexible_case_configuration[
                    case_code
                ]["Workload Treatment"],
            "Network Configuration":
                inflexible_case_configuration[
                    case_code
                ]["Network Configuration"],
            "C6 Rating MW":
                case_rating[
                    SELECTED_RECONDUCTORING_UID
                ],
            "Native Demand MW":
                native_pilot_demand,
            "Data-Center Demand MW":
                NOTEBOOK5_INFLEXIBLE_DC_POWER_MW,
            "Solved Total Demand MW":
                solved_demand,
            "Demand Difference MW":
                demand_difference,
            "Operating Cost $/h":
                solved_cost,
            "Incremental Cost vs Native $/h":
                solved_cost
                - native_pilot_cost,
            "Maximum Physical Residual":
                physical_residual,
        }
    )


inflexible_pilot_results = (
    pd.DataFrame(
        inflexible_pilot_rows
    )
    .set_index("Case")
)


# ---------------------------------------------------------------------
# 8. Calculate the diagnostic reconductoring effect
# ---------------------------------------------------------------------

pilot_reconductoring_savings_per_hour = float(
    inflexible_pilot_results.loc[
        "C0",
        "Operating Cost $/h",
    ]
    - inflexible_pilot_results.loc[
        "CR",
        "Operating Cost $/h",
    ]
)

inflexible_pilot_summary = pd.DataFrame(
    {
        "Value": {
            "Pilot timestamp":
                inflexible_pilot_timestamp,
            "Native congestion cost $/h":
                annual_hourly_summary.loc[
                    inflexible_pilot_timestamp,
                    "Congestion cost $/h",
                ],
            "Native operating cost $/h":
                native_pilot_cost,
            "Expected demand with DC MW":
                expected_pilot_demand_with_dc,
            "C0 operating cost $/h":
                inflexible_pilot_results.loc[
                    "C0",
                    "Operating Cost $/h",
                ],
            "CR operating cost $/h":
                inflexible_pilot_results.loc[
                    "CR",
                    "Operating Cost $/h",
                ],
            "Pilot reconductoring savings $/h":
                pilot_reconductoring_savings_per_hour,
            "Annual inflexible DC energy MWh":
                calculated_inflexible_annual_energy_mwh,
        }
    }
)


# ---------------------------------------------------------------------
# 9. Display results
# ---------------------------------------------------------------------

print("\nInflexible pilot-case results:")
display(inflexible_pilot_results)

print("\nInflexible pilot summary:")
display(inflexible_pilot_summary)


# ---------------------------------------------------------------------
# 10. Hard validation gates
# ---------------------------------------------------------------------

assert (
    inflexible_pilot_results[
        "Demand Difference MW"
    ].max()
    <= VALIDATION_TOL
), (
    "The 100 MW data-center demand was not represented "
    "correctly in one or more pilot cases."
)

assert (
    inflexible_pilot_results[
        "Maximum Physical Residual"
    ].max()
    <= VALIDATION_TOL
), (
    "At least one inflexible pilot solve violates the "
    "physical-validation tolerance."
)

assert np.isclose(
    inflexible_pilot_results.loc[
        "C0",
        "C6 Rating MW",
    ],
    BASELINE_C6_RATING_MW,
    rtol=0.0,
    atol=VALIDATION_TOL,
)

assert np.isclose(
    inflexible_pilot_results.loc[
        "CR",
        "C6 Rating MW",
    ],
    SELECTED_C6_RATING_MW,
    rtol=0.0,
    atol=VALIDATION_TOL,
)

print(
    "\nThe C0 and CR inflexible pilot cases passed "
    "all demand, network-treatment, and physical validations."
)

In [ ]:
# Audit the exact hourly-model components needed for 24-hour coupling.

import pandas as pd
import pyomo.environ as pyo

from IPython.display import display


audit_model = inflexible_pilot_models["C0"]


# ---------------------------------------------------------------------
# 1. Complete component inventory
# ---------------------------------------------------------------------

component_rows = []

component_types = [
    ("Set", pyo.Set),
    ("Param", pyo.Param),
    ("Var", pyo.Var),
    ("Expression", pyo.Expression),
    ("Constraint", pyo.Constraint),
    ("Objective", pyo.Objective),
    ("Suffix", pyo.Suffix),
]

for component_type_name, component_type in component_types:
    for component in audit_model.component_objects(
        component_type,
        active=None,
        descend_into=True,
    ):
        try:
            component_size = len(component)
        except TypeError:
            component_size = 1

        try:
            dimension = component.dim()
        except Exception:
            dimension = ""

        component_rows.append(
            {
                "Component Type": component_type_name,
                "Component Name": component.name,
                "Indexed": component.is_indexed(),
                "Dimension": dimension,
                "Size": component_size,
                "Active": getattr(
                    component,
                    "active",
                    "",
                ),
                "Mutable": getattr(
                    component,
                    "mutable",
                    "",
                ),
            }
        )

hourly_model_component_inventory = (
    pd.DataFrame(component_rows)
    .sort_values(
        ["Component Type", "Component Name"],
        ignore_index=True,
    )
)

print("Complete C0 hourly-model component inventory:")
display(hourly_model_component_inventory)


# ---------------------------------------------------------------------
# 2. Components directly indexed by data-center bus 309
# ---------------------------------------------------------------------

dc_bus_component_rows = []

for component_type_name, component_type in [
    ("Param", pyo.Param),
    ("Var", pyo.Var),
    ("Expression", pyo.Expression),
    ("Constraint", pyo.Constraint),
]:
    for component in audit_model.component_objects(
        component_type,
        active=None,
        descend_into=True,
    ):
        if not component.is_indexed():
            continue

        try:
            component_data = component[
                NOTEBOOK5_DC_BUS
            ]
        except (KeyError, TypeError):
            continue

        if component_type_name == "Constraint":
            body_text = str(component_data.body)
            lower_text = str(component_data.lower)
            upper_text = str(component_data.upper)
            numeric_value = ""

        else:
            body_text = ""
            lower_text = ""
            upper_text = ""

            try:
                numeric_value = float(
                    pyo.value(component_data)
                )
            except Exception:
                numeric_value = ""

        if len(body_text) > 500:
            body_text = body_text[:497] + "..."

        dc_bus_component_rows.append(
            {
                "Component Type": component_type_name,
                "Component Name": component.name,
                "Value": numeric_value,
                "Lower": lower_text,
                "Upper": upper_text,
                "Body or Expression": body_text,
            }
        )

dc_bus_component_audit = pd.DataFrame(
    dc_bus_component_rows
)

print(
    f"\nComponents indexed directly by bus "
    f"{NOTEBOOK5_DC_BUS}:"
)
display(dc_bus_component_audit)


# ---------------------------------------------------------------------
# 3. Objective audit
# ---------------------------------------------------------------------

objective_rows = []

for objective_data in (
    audit_model.component_data_objects(
        pyo.Objective,
        active=None,
        descend_into=True,
    )
):
    try:
        objective_value = float(
            pyo.value(objective_data.expr)
        )
    except Exception:
        objective_value = np.nan

    expression_text = str(
        objective_data.expr
    )

    if len(expression_text) > 500:
        expression_text = (
            expression_text[:497] + "..."
        )

    objective_rows.append(
        {
            "Objective": objective_data.name,
            "Active": objective_data.active,
            "Sense": str(objective_data.sense),
            "Current Value": objective_value,
            "Expression": expression_text,
        }
    )

hourly_model_objective_audit = pd.DataFrame(
    objective_rows
)

print("\nHourly-model objective audit:")
display(hourly_model_objective_audit)


# ---------------------------------------------------------------------
# 4. Locate the model-source lines relevant to temporal coupling
# ---------------------------------------------------------------------

source_keywords = {
    "build_hourly_rts_opf": [
        "Param",
        "Var",
        "Constraint",
        "Objective",
        "flow",
        "balance",
        "theta",
        "angle",
        "total_cost",
    ],
    "add_lexicographic_components": [
        "cost",
        "objective",
        "curtail",
        "vre",
        "primary",
        "secondary",
        "cap",
    ],
    "load_hour_into_model": [
        "load",
        "demand",
        "pmax",
        "availability",
        "Param",
    ],
}

source_line_rows = []

for function_name, keywords in source_keywords.items():
    function_source = (
        notebook4_function_sources[
            function_name
        ]
    )

    for line_number, source_line in enumerate(
        function_source.splitlines(),
        start=1,
    ):
        normalized_line = source_line.casefold()

        if not any(
            keyword.casefold() in normalized_line
            for keyword in keywords
        ):
            continue

        source_line_rows.append(
            {
                "Function": function_name,
                "Line": line_number,
                "Source": source_line.rstrip(),
            }
        )

relevant_model_source_lines = pd.DataFrame(
    source_line_rows
)

print("\nRelevant lines from the recovered Notebook 4 functions:")
display(relevant_model_source_lines)


print(
    "\nHourly-model structural audit completed. "
    "No model inputs or solutions were changed."
)

## Pilot daily co-optimization of flexible data-center demand

The validated hourly DC-OPF is replicated across the 24 hours of the pilot day.
All original generator, transmission, power-flow, and nodal-balance constraints
are preserved.

At bus 309, the fixed 100 MW data-center demand is replaced by an hourly
decision variable bounded between 50 MW and 150 MW. The sum of hourly
data-center consumption must equal 2,400 MWh, ensuring that flexibility shifts
workload within the day without changing total daily computation.

Two flexible cases are evaluated:

- **CF:** flexible workload with the original 175 MW C6 rating;
- **CFR:** flexible workload with the reconductored 262.5 MW C6 rating.

The model first minimizes daily operating cost and then, without materially
changing that optimum, maximizes variable-renewable dispatch.

In [ ]:
# Build and solve the first 24-hour flexible-workload pilot.

import numpy as np
import pandas as pd
import pyomo.environ as pyo

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Flexible-case configuration
# ---------------------------------------------------------------------

DAILY_DC_MINIMUM_MW = 50.0
DAILY_DC_MAXIMUM_MW = 150.0
DAILY_DC_ENERGY_MWH = 2400.0

DAILY_COST_ABSOLUTE_TOLERANCE = 1e-4
DAILY_COST_RELATIVE_TOLERANCE = 1e-9

flexible_case_configuration = {
    "CF": {
        "Network Configuration": "Original",
        "C6 Rating MW": float(
            BASELINE_C6_RATING_MW
        ),
        "Inflexible Comparator": "C0",
    },
    "CFR": {
        "Network Configuration": "Reconductored",
        "C6 Rating MW": float(
            SELECTED_C6_RATING_MW
        ),
        "Inflexible Comparator": "CR",
    },
}

assert DAILY_DC_MINIMUM_MW <= 100.0
assert DAILY_DC_MAXIMUM_MW >= 100.0

assert np.isclose(
    24.0 * 100.0,
    DAILY_DC_ENERGY_MWH,
    rtol=0.0,
    atol=VALIDATION_TOL,
)


# ---------------------------------------------------------------------
# 2. Select the complete day containing the congestion pilot hour
# ---------------------------------------------------------------------

flexible_pilot_day_start = (
    pd.Timestamp(
        inflexible_pilot_timestamp
    ).normalize()
)

flexible_pilot_timestamps = pd.date_range(
    start=flexible_pilot_day_start,
    periods=24,
    freq="h",
)

if not flexible_pilot_timestamps.isin(
    expected_annual_index
).all():
    raise KeyError(
        "The selected pilot day is not completely represented "
        "in the annual chronology."
    )

assert inflexible_pilot_timestamp in (
    flexible_pilot_timestamps
)

print(
    "Flexible pilot day: "
    f"{flexible_pilot_day_start.date()}"
)
print(
    "Hours included: "
    f"{flexible_pilot_timestamps[0]} through "
    f"{flexible_pilot_timestamps[-1]}"
)


# ---------------------------------------------------------------------
# 3. Build an exact 24-hour copy of the Notebook 4 formulation
# ---------------------------------------------------------------------

def build_daily_flexible_model(
    case_code,
    daily_timestamps,
):
    """
    Replicate the validated hourly model across one day and replace
    the fixed 100 MW demand at bus 309 with a flexible variable.
    """
    if case_code not in flexible_case_configuration:
        raise KeyError(
            f"Unknown flexible case: {case_code!r}"
        )

    daily_timestamps = pd.DatetimeIndex(
        daily_timestamps
    )

    if len(daily_timestamps) != 24:
        raise ValueError(
            "A daily flexible model requires exactly 24 hours."
        )

    case_rating = dict(
        notebook5_native_branch_rating
    )

    case_rating[
        SELECTED_RECONDUCTORING_UID
    ] = flexible_case_configuration[
        case_code
    ]["C6 Rating MW"]

    original_hourly_load_object = globals().get(
        "hourly_bus_load"
    )
    original_branch_rating_object = globals().get(
        "branch_rating"
    )

    daily_model = pyo.ConcreteModel(
        name=f"Notebook5_{case_code}_Daily_OPF"
    )

    daily_model.HOUR = pyo.RangeSet(
        0,
        len(daily_timestamps) - 1,
    )

    daily_model.hour = pyo.Block(
        daily_model.HOUR
    )

    timestamp_by_hour = {
        hour_position: timestamp
        for hour_position, timestamp
        in enumerate(daily_timestamps)
    }

    try:
        globals()["hourly_bus_load"] = (
            notebook5_inflexible_hourly_bus_load
        )
        globals()["branch_rating"] = (
            case_rating
        )

        # The template contains exactly the validated Notebook 4
        # formulation with the selected network rating.
        hourly_template = build_hourly_rts_opf()

        for hour_position in daily_model.HOUR:
            timestamp = timestamp_by_hour[
                int(hour_position)
            ]

            load_hour_into_model(
                hourly_template,
                timestamp,
            )

            hourly_copy = (
                hourly_template.clone()
            )

            daily_model.hour[
                hour_position
            ].transfer_attributes_from(
                hourly_copy
            )

            # A single top-level daily objective will replace the
            # 24 separate hourly objectives.
            daily_model.hour[
                hour_position
            ].total_cost.deactivate()

    finally:
        globals()["hourly_bus_load"] = (
            original_hourly_load_object
        )
        globals()["branch_rating"] = (
            original_branch_rating_object
        )

    daily_model.timestamp_by_hour = (
        timestamp_by_hour
    )
    daily_model.case_code = case_code
    daily_model.case_rating = case_rating

    # Flexible data-center power.
    daily_model.dc_power = pyo.Var(
        daily_model.HOUR,
        domain=pyo.NonNegativeReals,
        bounds=(
            DAILY_DC_MINIMUM_MW,
            DAILY_DC_MAXIMUM_MW,
        ),
        initialize=100.0,
    )

    # Replace only the bus-309 balance equation.
    for hour_position in daily_model.HOUR:
        daily_model.hour[
            hour_position
        ].nodal_balance[
            NOTEBOOK5_DC_BUS
        ].deactivate()

    def flexible_bus_balance_rule(
        model,
        hour_position,
    ):
        original_balance_body = (
            model.hour[
                hour_position
            ].nodal_balance[
                NOTEBOOK5_DC_BUS
            ].body
        )

        # The original demand parameter contains the native load
        # plus 100 MW. Remove that fixed treatment and insert the
        # flexible decision variable.
        return (
            original_balance_body
            + NOTEBOOK5_INFLEXIBLE_DC_POWER_MW
            - model.dc_power[hour_position]
            == 0.0
        )

    daily_model.flexible_bus_balance = (
        pyo.Constraint(
            daily_model.HOUR,
            rule=flexible_bus_balance_rule,
        )
    )

    daily_model.daily_dc_energy = (
        pyo.Constraint(
            expr=sum(
                daily_model.dc_power[
                    hour_position
                ]
                for hour_position
                in daily_model.HOUR
            )
            == DAILY_DC_ENERGY_MWH
        )
    )

    # Total daily operating cost.
    daily_model.daily_cost = pyo.Expression(
        expr=sum(
            daily_model.hour[
                hour_position
            ].total_cost.expr
            for hour_position
            in daily_model.HOUR
        )
    )

    daily_model.total_cost = pyo.Objective(
        expr=daily_model.daily_cost,
        sense=pyo.minimize,
    )

    # Lexicographic cost cap.
    daily_model.primary_cost_cap_value = (
        pyo.Param(
            mutable=True,
            initialize=0.0,
        )
    )

    daily_model.primary_cost_cap = (
        pyo.Constraint(
            expr=(
                daily_model.daily_cost
                <= daily_model.primary_cost_cap_value
            )
        )
    )

    daily_model.primary_cost_cap.deactivate()

    # Secondary objective: maximize total VRE dispatch.
    daily_model.total_vre_dispatch = (
        pyo.Expression(
            expr=sum(
                daily_model.hour[
                    hour_position
                ].generation[
                    generator_uid
                ]
                for hour_position
                in daily_model.HOUR
                for generator_uid
                in variable_renewable_generator_ids
            )
        )
    )

    daily_model.maximum_vre_dispatch = (
        pyo.Objective(
            expr=daily_model.total_vre_dispatch,
            sense=pyo.maximize,
        )
    )

    daily_model.maximum_vre_dispatch.deactivate()

    return daily_model


# ---------------------------------------------------------------------
# 4. Solver and validation helpers
# ---------------------------------------------------------------------

def confirm_optimal_termination(
    solve_result,
    label,
):
    termination_condition = str(
        solve_result.solver.termination_condition
    ).casefold()

    if "optimal" not in termination_condition:
        raise RuntimeError(
            f"{label} did not terminate optimally. "
            f"Termination condition: "
            f"{solve_result.solver.termination_condition}"
        )


def solve_daily_primary_objective(
    model,
    solver,
):
    """Solve only the primary operating-cost objective."""
    model.primary_cost_cap.deactivate()
    model.maximum_vre_dispatch.deactivate()
    model.total_cost.activate()

    solve_result = solver.solve(
        model,
        tee=False,
    )

    confirm_optimal_termination(
        solve_result,
        "Daily primary optimization",
    )

    return float(
        pyo.value(model.daily_cost)
    )


def solve_daily_lexicographically(
    model,
    solver,
):
    """
    Minimize daily cost and then maximize VRE dispatch within
    a numerically negligible cost tolerance.
    """
    primary_optimal_cost = (
        solve_daily_primary_objective(
            model,
            solver,
        )
    )

    cost_tolerance = max(
        DAILY_COST_ABSOLUTE_TOLERANCE,
        DAILY_COST_RELATIVE_TOLERANCE
        * abs(primary_optimal_cost),
    )

    model.primary_cost_cap_value.set_value(
        primary_optimal_cost
        + cost_tolerance
    )

    model.total_cost.deactivate()
    model.primary_cost_cap.activate()
    model.maximum_vre_dispatch.activate()

    secondary_result = solver.solve(
        model,
        tee=False,
    )

    confirm_optimal_termination(
        secondary_result,
        "Daily secondary optimization",
    )

    final_cost = float(
        pyo.value(model.daily_cost)
    )

    total_vre_dispatch = float(
        pyo.value(model.total_vre_dispatch)
    )

    if final_cost > (
        primary_optimal_cost
        + cost_tolerance
        + VALIDATION_TOL
    ):
        raise AssertionError(
            "The secondary optimization exceeded the "
            "primary-cost cap."
        )

    return {
        "Primary Optimal Cost $":
            primary_optimal_cost,
        "Final Lexicographic Cost $":
            final_cost,
        "Cost-Cap Tolerance $":
            cost_tolerance,
        "Total VRE Dispatch MWh":
            total_vre_dispatch,
    }


def maximum_active_constraint_violation(
    model,
):
    """Measure the largest violation across every active constraint."""
    maximum_violation = 0.0
    maximum_constraint_name = ""

    for constraint_data in (
        model.component_data_objects(
            pyo.Constraint,
            active=True,
            descend_into=True,
        )
    ):
        body_value = float(
            pyo.value(constraint_data.body)
        )

        constraint_violation = 0.0

        if constraint_data.lower is not None:
            lower_value = float(
                pyo.value(constraint_data.lower)
            )
            constraint_violation = max(
                constraint_violation,
                lower_value - body_value,
            )

        if constraint_data.upper is not None:
            upper_value = float(
                pyo.value(constraint_data.upper)
            )
            constraint_violation = max(
                constraint_violation,
                body_value - upper_value,
            )

        constraint_violation = max(
            0.0,
            constraint_violation,
        )

        if constraint_violation > maximum_violation:
            maximum_violation = (
                constraint_violation
            )
            maximum_constraint_name = (
                constraint_data.name
            )

    return (
        maximum_violation,
        maximum_constraint_name,
    )


# ---------------------------------------------------------------------
# 5. Build and solve CF and CFR
# ---------------------------------------------------------------------

daily_flexible_models = {}
daily_flexible_solvers = {}
daily_fixed_hourly_costs = {}
daily_flexible_rows = []
daily_schedule_rows = []

for case_code in ["CF", "CFR"]:
    print(
        f"\nBuilding daily model for {case_code}..."
    )

    daily_model = build_daily_flexible_model(
        case_code=case_code,
        daily_timestamps=flexible_pilot_timestamps,
    )

    daily_solver = pyo.SolverFactory(
        "appsi_highs"
    )

    if not daily_solver.available(
        exception_flag=False
    ):
        raise RuntimeError(
            "The appsi_highs solver is unavailable."
        )

    # -------------------------------------------------------------
    # 5A. Solve the same model with DC power fixed at 100 MW.
    # This provides the C0 or CR daily comparator.
    # -------------------------------------------------------------

    for hour_position in daily_model.HOUR:
        daily_model.dc_power[
            hour_position
        ].fix(
            NOTEBOOK5_INFLEXIBLE_DC_POWER_MW
        )

    inflexible_daily_cost = (
        solve_daily_primary_objective(
            daily_model,
            daily_solver,
        )
    )

    fixed_hourly_costs = {}

    for hour_position in daily_model.HOUR:
        timestamp = (
            daily_model.timestamp_by_hour[
                int(hour_position)
            ]
        )

        fixed_hourly_costs[timestamp] = float(
            pyo.value(
                daily_model.hour[
                    hour_position
                ].total_cost.expr
            )
        )

    daily_fixed_hourly_costs[
        case_code
    ] = fixed_hourly_costs

    # -------------------------------------------------------------
    # 5B. Release the hourly DC schedule and solve flexibly.
    # -------------------------------------------------------------

    for hour_position in daily_model.HOUR:
        daily_model.dc_power[
            hour_position
        ].unfix()

    lexicographic_result = (
        solve_daily_lexicographically(
            daily_model,
            daily_solver,
        )
    )

    (
        maximum_constraint_violation,
        maximum_constraint_name,
    ) = maximum_active_constraint_violation(
        daily_model
    )

    dc_schedule = np.array(
        [
            pyo.value(
                daily_model.dc_power[
                    hour_position
                ]
            )
            for hour_position
            in daily_model.HOUR
        ],
        dtype=float,
    )

    calculated_daily_energy = float(
        dc_schedule.sum()
    )

    minimum_bound_violation = float(
        np.maximum(
            DAILY_DC_MINIMUM_MW
            - dc_schedule,
            0.0,
        ).max()
    )

    maximum_bound_violation = float(
        np.maximum(
            dc_schedule
            - DAILY_DC_MAXIMUM_MW,
            0.0,
        ).max()
    )

    flexible_daily_cost = (
        lexicographic_result[
            "Final Lexicographic Cost $"
        ]
    )

    flexibility_savings = (
        inflexible_daily_cost
        - flexible_daily_cost
    )

    daily_flexible_rows.append(
        {
            "Case": case_code,
            "Network Configuration":
                flexible_case_configuration[
                    case_code
                ]["Network Configuration"],
            "C6 Rating MW":
                flexible_case_configuration[
                    case_code
                ]["C6 Rating MW"],
            "Inflexible Comparator":
                flexible_case_configuration[
                    case_code
                ]["Inflexible Comparator"],
            "Inflexible Daily Cost $":
                inflexible_daily_cost,
            "Flexible Primary Optimum $":
                lexicographic_result[
                    "Primary Optimal Cost $"
                ],
            "Flexible Final Cost $":
                flexible_daily_cost,
            "Flexibility Savings $":
                flexibility_savings,
            "Flexibility Savings %":
                (
                    100.0
                    * flexibility_savings
                    / inflexible_daily_cost
                ),
            "Daily DC Energy MWh":
                calculated_daily_energy,
            "Minimum DC Power MW":
                float(dc_schedule.min()),
            "Maximum DC Power MW":
                float(dc_schedule.max()),
            "Hours at 50 MW":
                int(
                    np.isclose(
                        dc_schedule,
                        DAILY_DC_MINIMUM_MW,
                        atol=1e-5,
                    ).sum()
                ),
            "Hours at 150 MW":
                int(
                    np.isclose(
                        dc_schedule,
                        DAILY_DC_MAXIMUM_MW,
                        atol=1e-5,
                    ).sum()
                ),
            "Maximum Constraint Violation":
                maximum_constraint_violation,
            "Maximum-Violation Constraint":
                maximum_constraint_name,
            "Cost-Cap Tolerance $":
                lexicographic_result[
                    "Cost-Cap Tolerance $"
                ],
        }
    )

    # -------------------------------------------------------------
    # 5C. Extract the optimized hourly schedule.
    # -------------------------------------------------------------

    for hour_position in daily_model.HOUR:
        timestamp = (
            daily_model.timestamp_by_hour[
                int(hour_position)
            ]
        )

        hourly_block = daily_model.hour[
            hour_position
        ]

        dc_power = float(
            pyo.value(
                daily_model.dc_power[
                    hour_position
                ]
            )
        )

        native_demand = float(
            notebook5_native_hourly_bus_load.loc[
                timestamp
            ].sum()
        )

        total_generation = float(
            sum(
                pyo.value(
                    hourly_block.generation[
                        generator_uid
                    ]
                )
                for generator_uid
                in GENERATOR_IDS
            )
        )

        available_vre = float(
            hourly_gen_pmax.loc[
                timestamp,
                variable_renewable_generator_ids,
            ].sum()
        )

        dispatched_vre = float(
            sum(
                pyo.value(
                    hourly_block.generation[
                        generator_uid
                    ]
                )
                for generator_uid
                in variable_renewable_generator_ids
            )
        )

        c6_flow = float(
            pyo.value(
                hourly_block.flow[
                    SELECTED_RECONDUCTORING_UID
                ]
            )
        )

        c6_rating = float(
            flexible_case_configuration[
                case_code
            ]["C6 Rating MW"]
        )

        daily_schedule_rows.append(
            {
                "Case": case_code,
                "Timestamp": timestamp,
                "Hour": int(hour_position),
                "DC Power MW": dc_power,
                "Shift from 100 MW": (
                    dc_power
                    - NOTEBOOK5_INFLEXIBLE_DC_POWER_MW
                ),
                "Native Demand MW":
                    native_demand,
                "Total Demand with DC MW":
                    native_demand + dc_power,
                "Total Generation MW":
                    total_generation,
                "System Balance Residual MW":
                    total_generation
                    - native_demand
                    - dc_power,
                "Hourly Operating Cost $":
                    float(
                        pyo.value(
                            hourly_block.total_cost.expr
                        )
                    ),
                "VRE Available MW":
                    available_vre,
                "VRE Dispatched MW":
                    dispatched_vre,
                "VRE Curtailment MW":
                    max(
                        available_vre
                        - dispatched_vre,
                        0.0,
                    ),
                "C6 Flow MW":
                    c6_flow,
                "C6 Rating MW":
                    c6_rating,
                "C6 Loading %":
                    (
                        100.0
                        * abs(c6_flow)
                        / c6_rating
                    ),
            }
        )

    daily_flexible_models[
        case_code
    ] = daily_model

    daily_flexible_solvers[
        case_code
    ] = daily_solver

    print(
        f"{case_code} solved successfully."
    )


# ---------------------------------------------------------------------
# 6. Assemble pilot results
# ---------------------------------------------------------------------

daily_flexible_summary = (
    pd.DataFrame(
        daily_flexible_rows
    )
    .set_index("Case")
)

daily_flexible_schedule = (
    pd.DataFrame(
        daily_schedule_rows
    )
    .set_index(
        ["Case", "Timestamp"]
    )
    .sort_index()
)


# ---------------------------------------------------------------------
# 7. Validate fixed-schedule equivalence at the original pilot hour
# ---------------------------------------------------------------------

fixed_schedule_regression_rows = []

for flexible_case, inflexible_case in [
    ("CF", "C0"),
    ("CFR", "CR"),
]:
    daily_block_cost = (
        daily_fixed_hourly_costs[
            flexible_case
        ][inflexible_pilot_timestamp]
    )

    direct_hourly_cost = float(
        inflexible_pilot_results.loc[
            inflexible_case,
            "Operating Cost $/h",
        ]
    )

    fixed_schedule_regression_rows.append(
        {
            "Flexible Model": flexible_case,
            "Direct Hourly Case": inflexible_case,
            "Timestamp":
                inflexible_pilot_timestamp,
            "Direct Hourly Cost $":
                direct_hourly_cost,
            "Daily-Block Hourly Cost $":
                daily_block_cost,
            "Absolute Difference $":
                abs(
                    daily_block_cost
                    - direct_hourly_cost
                ),
        }
    )

fixed_schedule_regression = pd.DataFrame(
    fixed_schedule_regression_rows
).set_index("Flexible Model")


# ---------------------------------------------------------------------
# 8. Pilot factorial decomposition
# ---------------------------------------------------------------------

pilot_c0_cost = float(
    daily_flexible_summary.loc[
        "CF",
        "Inflexible Daily Cost $",
    ]
)

pilot_cf_cost = float(
    daily_flexible_summary.loc[
        "CF",
        "Flexible Final Cost $",
    ]
)

pilot_cr_cost = float(
    daily_flexible_summary.loc[
        "CFR",
        "Inflexible Daily Cost $",
    ]
)

pilot_cfr_cost = float(
    daily_flexible_summary.loc[
        "CFR",
        "Flexible Final Cost $",
    ]
)

pilot_factorial_effects = pd.DataFrame(
    {
        "Value $": {
            "Flexibility value on original network":
                pilot_c0_cost - pilot_cf_cost,

            "Flexibility value on reconductored network":
                pilot_cr_cost - pilot_cfr_cost,

            "Reconductoring value with inflexible load":
                pilot_c0_cost - pilot_cr_cost,

            "Reconductoring value with flexible load":
                pilot_cf_cost - pilot_cfr_cost,

            "Combined value relative to C0":
                pilot_c0_cost - pilot_cfr_cost,

            "Combined-savings interaction":
                (
                    pilot_cf_cost
                    + pilot_cr_cost
                    - pilot_cfr_cost
                    - pilot_c0_cost
                ),
        }
    }
)


# ---------------------------------------------------------------------
# 9. Display results
# ---------------------------------------------------------------------

print("\nDaily flexible pilot summary:")
display(daily_flexible_summary)

print("\nFixed-schedule regression check:")
display(fixed_schedule_regression)

print("\nPilot factorial effects:")
display(pilot_factorial_effects)

print("\nOptimized hourly schedules:")
display(daily_flexible_schedule)


# ---------------------------------------------------------------------
# 10. Hard validation gates
# ---------------------------------------------------------------------

assert (
    fixed_schedule_regression[
        "Absolute Difference $"
    ].max()
    <= 1e-3
), (
    "The daily block formulation does not reproduce the "
    "validated hourly inflexible model."
)

assert np.allclose(
    daily_flexible_summary[
        "Daily DC Energy MWh"
    ].to_numpy(),
    DAILY_DC_ENERGY_MWH,
    rtol=0.0,
    atol=VALIDATION_TOL,
), (
    "At least one flexible schedule violates the daily "
    "2,400 MWh energy requirement."
)

assert (
    daily_flexible_summary[
        "Minimum DC Power MW"
    ].min()
    >= DAILY_DC_MINIMUM_MW
    - VALIDATION_TOL
)

assert (
    daily_flexible_summary[
        "Maximum DC Power MW"
    ].max()
    <= DAILY_DC_MAXIMUM_MW
    + VALIDATION_TOL
)

assert (
    daily_flexible_summary[
        "Maximum Constraint Violation"
    ].max()
    <= VALIDATION_TOL
), (
    "At least one daily model violates an active physical "
    "or operational constraint."
)

assert (
    daily_flexible_schedule[
        "System Balance Residual MW"
    ].abs().max()
    <= VALIDATION_TOL
), (
    "At least one optimized hour violates total system balance."
)

assert (
    daily_flexible_summary[
        "Flexible Final Cost $"
    ]
    <= (
        daily_flexible_summary[
            "Inflexible Daily Cost $"
        ]
        + daily_flexible_summary[
            "Cost-Cap Tolerance $"
        ]
        + VALIDATION_TOL
    )
).all(), (
    "Flexible operation produced a higher cost than the "
    "feasible 100 MW constant schedule."
)

print(
    "\nThe CF and CFR 24-hour pilot models passed all "
    "energy, power-bound, hourly-equivalence, network, "
    "and physical-feasibility validations."
)

## Representative-day regression of the flexible formulation

The daily co-optimization is now evaluated across representative operating
conditions before the complete annual simulation.

For each month, the sample includes the days with the lowest and highest
average native demand. Additional annual extreme days capture maximum daily
demand, minimum daily demand, maximum renewable availability, maximum
congestion cost, peak hourly demand, and peak hourly congestion.

Both CF and CFR are solved for every selected day. The regression verifies
daily energy conservation, hourly power limits, network feasibility, system
balance, and the economic dominance of the flexible schedule over the feasible
constant 100 MW schedule.

Because the sample deliberately emphasizes extremes, its average effects are
diagnostic and must not be interpreted as annual estimates.

In [ ]:
# Run the flexible model across representative 2020 days.

import gc

from collections import defaultdict
from time import perf_counter

import numpy as np
import pandas as pd
import pyomo.environ as pyo

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Construct daily system-context metrics
# ---------------------------------------------------------------------

representative_daily_context = pd.DataFrame(
    {
        "Mean Demand MW":
            annual_hourly_summary[
                "Demand MW"
            ].resample("D").mean(),

        "Maximum Demand MW":
            annual_hourly_summary[
                "Demand MW"
            ].resample("D").max(),

        "Minimum Demand MW":
            annual_hourly_summary[
                "Demand MW"
            ].resample("D").min(),

        "Daily Demand MWh":
            annual_hourly_summary[
                "Demand MW"
            ].resample("D").sum(),

        "Daily VRE Availability MWh":
            annual_hourly_summary[
                "VRE available MW"
            ].resample("D").sum(),

        "Daily Native Congestion Cost $":
            annual_hourly_summary[
                "Congestion cost $/h"
            ].resample("D").sum(),

        "Maximum Hourly Congestion Cost $/h":
            annual_hourly_summary[
                "Congestion cost $/h"
            ].resample("D").max(),
    }
)

assert len(representative_daily_context) == 366
assert not representative_daily_context.isna().any().any()


# ---------------------------------------------------------------------
# 2. Select monthly and annual representative days
# ---------------------------------------------------------------------

representative_day_labels = defaultdict(list)

monthly_daily_groups = (
    representative_daily_context.groupby(
        representative_daily_context.index.to_period(
            "M"
        )
    )
)

for month, monthly_data in monthly_daily_groups:
    minimum_day = monthly_data[
        "Mean Demand MW"
    ].idxmin()

    maximum_day = monthly_data[
        "Mean Demand MW"
    ].idxmax()

    representative_day_labels[
        minimum_day
    ].append(
        f"{month} minimum mean-demand day"
    )

    representative_day_labels[
        maximum_day
    ].append(
        f"{month} maximum mean-demand day"
    )


annual_daily_extremes = {
    "Annual minimum mean-demand day":
        representative_daily_context[
            "Mean Demand MW"
        ].idxmin(),

    "Annual maximum mean-demand day":
        representative_daily_context[
            "Mean Demand MW"
        ].idxmax(),

    "Annual maximum VRE-availability day":
        representative_daily_context[
            "Daily VRE Availability MWh"
        ].idxmax(),

    "Annual maximum daily-congestion-cost day":
        representative_daily_context[
            "Daily Native Congestion Cost $"
        ].idxmax(),

    "Day containing annual peak hourly demand":
        annual_hourly_summary[
            "Demand MW"
        ].idxmax().normalize(),

    "Day containing annual peak hourly congestion":
        annual_hourly_summary[
            "Congestion cost $/h"
        ].idxmax().normalize(),
}

for label, day in annual_daily_extremes.items():
    representative_day_labels[
        pd.Timestamp(day)
    ].append(label)


representative_days = pd.DatetimeIndex(
    sorted(representative_day_labels)
)

print(
    "Representative days selected: "
    f"{len(representative_days):,}"
)

representative_day_selection = pd.DataFrame(
    [
        {
            "Day": day,
            "Selection Reason": "; ".join(
                representative_day_labels[day]
            ),
            **representative_daily_context.loc[
                day
            ].to_dict(),
        }
        for day in representative_days
    ]
).set_index("Day")

display(representative_day_selection)


# ---------------------------------------------------------------------
# 3. Solve CF and CFR for every representative day
# ---------------------------------------------------------------------

representative_daily_rows = []
representative_schedule_rows = []

representative_start_time = perf_counter()

total_model_runs = (
    len(representative_days)
    * len(flexible_case_configuration)
)

completed_model_runs = 0

for day_number, day in enumerate(
    representative_days,
    start=1,
):
    daily_timestamps = pd.date_range(
        start=day,
        periods=24,
        freq="h",
    )

    for case_code in ["CF", "CFR"]:
        daily_model = build_daily_flexible_model(
            case_code=case_code,
            daily_timestamps=daily_timestamps,
        )

        daily_solver = pyo.SolverFactory(
            "appsi_highs"
        )

        if not daily_solver.available(
            exception_flag=False
        ):
            raise RuntimeError(
                "The appsi_highs solver is unavailable."
            )

        case_rating = float(
            flexible_case_configuration[
                case_code
            ]["C6 Rating MW"]
        )

        # -------------------------------------------------------------
        # 3A. Inflexible comparator: constant 100 MW
        # -------------------------------------------------------------

        for hour_position in daily_model.HOUR:
            daily_model.dc_power[
                hour_position
            ].fix(
                NOTEBOOK5_INFLEXIBLE_DC_POWER_MW
            )

        inflexible_daily_cost = (
            solve_daily_primary_objective(
                daily_model,
                daily_solver,
            )
        )

        daily_vre_available = float(
            hourly_gen_pmax.loc[
                daily_timestamps,
                variable_renewable_generator_ids,
            ].to_numpy().sum()
        )

        inflexible_vre_dispatch = float(
            sum(
                pyo.value(
                    daily_model.hour[
                        hour_position
                    ].generation[
                        generator_uid
                    ]
                )
                for hour_position
                in daily_model.HOUR
                for generator_uid
                in variable_renewable_generator_ids
            )
        )

        inflexible_vre_curtailment = max(
            daily_vre_available
            - inflexible_vre_dispatch,
            0.0,
        )

        inflexible_c6_binding_hours = int(
            sum(
                abs(
                    pyo.value(
                        daily_model.hour[
                            hour_position
                        ].flow[
                            SELECTED_RECONDUCTORING_UID
                        ]
                    )
                )
                >= case_rating - 1e-5
                for hour_position
                in daily_model.HOUR
            )
        )

        # -------------------------------------------------------------
        # 3B. Flexible solution
        # -------------------------------------------------------------

        for hour_position in daily_model.HOUR:
            daily_model.dc_power[
                hour_position
            ].unfix()

        lexicographic_result = (
            solve_daily_lexicographically(
                daily_model,
                daily_solver,
            )
        )

        flexible_daily_cost = float(
            lexicographic_result[
                "Final Lexicographic Cost $"
            ]
        )

        (
            maximum_constraint_violation,
            maximum_constraint_name,
        ) = maximum_active_constraint_violation(
            daily_model
        )

        dc_schedule = np.array(
            [
                pyo.value(
                    daily_model.dc_power[
                        hour_position
                    ]
                )
                for hour_position
                in daily_model.HOUR
            ],
            dtype=float,
        )

        flexible_vre_dispatch = float(
            lexicographic_result[
                "Total VRE Dispatch MWh"
            ]
        )

        flexible_vre_curtailment = max(
            daily_vre_available
            - flexible_vre_dispatch,
            0.0,
        )

        flexible_c6_binding_hours = int(
            sum(
                abs(
                    pyo.value(
                        daily_model.hour[
                            hour_position
                        ].flow[
                            SELECTED_RECONDUCTORING_UID
                        ]
                    )
                )
                >= case_rating - 1e-5
                for hour_position
                in daily_model.HOUR
            )
        )

        hourly_system_balance_residuals = []

        for hour_position in daily_model.HOUR:
            timestamp = (
                daily_model.timestamp_by_hour[
                    int(hour_position)
                ]
            )

            hourly_block = daily_model.hour[
                hour_position
            ]

            dc_power = float(
                pyo.value(
                    daily_model.dc_power[
                        hour_position
                    ]
                )
            )

            native_demand = float(
                notebook5_native_hourly_bus_load.loc[
                    timestamp
                ].sum()
            )

            total_generation = float(
                sum(
                    pyo.value(
                        hourly_block.generation[
                            generator_uid
                        ]
                    )
                    for generator_uid
                    in GENERATOR_IDS
                )
            )

            balance_residual = (
                total_generation
                - native_demand
                - dc_power
            )

            hourly_system_balance_residuals.append(
                abs(balance_residual)
            )

            available_vre = float(
                hourly_gen_pmax.loc[
                    timestamp,
                    variable_renewable_generator_ids,
                ].sum()
            )

            dispatched_vre = float(
                sum(
                    pyo.value(
                        hourly_block.generation[
                            generator_uid
                        ]
                    )
                    for generator_uid
                    in variable_renewable_generator_ids
                )
            )

            c6_flow = float(
                pyo.value(
                    hourly_block.flow[
                        SELECTED_RECONDUCTORING_UID
                    ]
                )
            )

            representative_schedule_rows.append(
                {
                    "Day": day,
                    "Case": case_code,
                    "Timestamp": timestamp,
                    "DC Power MW": dc_power,
                    "Shift from 100 MW":
                        dc_power - 100.0,
                    "Native Demand MW":
                        native_demand,
                    "Hourly Operating Cost $":
                        float(
                            pyo.value(
                                hourly_block.total_cost.expr
                            )
                        ),
                    "VRE Available MW":
                        available_vre,
                    "VRE Dispatched MW":
                        dispatched_vre,
                    "VRE Curtailment MW":
                        max(
                            available_vre
                            - dispatched_vre,
                            0.0,
                        ),
                    "C6 Flow MW":
                        c6_flow,
                    "C6 Loading %":
                        (
                            100.0
                            * abs(c6_flow)
                            / case_rating
                        ),
                    "System Balance Residual MW":
                        balance_residual,
                }
            )

        flexibility_savings = (
            inflexible_daily_cost
            - flexible_daily_cost
        )

        representative_daily_rows.append(
            {
                "Day": day,
                "Case": case_code,
                "Selection Reason": "; ".join(
                    representative_day_labels[day]
                ),
                "Network Configuration":
                    flexible_case_configuration[
                        case_code
                    ]["Network Configuration"],
                "C6 Rating MW":
                    case_rating,
                "Inflexible Comparator":
                    flexible_case_configuration[
                        case_code
                    ]["Inflexible Comparator"],
                "Inflexible Daily Cost $":
                    inflexible_daily_cost,
                "Flexible Primary Optimum $":
                    lexicographic_result[
                        "Primary Optimal Cost $"
                    ],
                "Flexible Final Cost $":
                    flexible_daily_cost,
                "Flexibility Savings $":
                    flexibility_savings,
                "Flexibility Savings %":
                    (
                        100.0
                        * flexibility_savings
                        / inflexible_daily_cost
                    ),
                "Daily DC Energy MWh":
                    float(dc_schedule.sum()),
                "Minimum DC Power MW":
                    float(dc_schedule.min()),
                "Maximum DC Power MW":
                    float(dc_schedule.max()),
                "Hours at 50 MW":
                    int(
                        np.isclose(
                            dc_schedule,
                            DAILY_DC_MINIMUM_MW,
                            atol=1e-5,
                        ).sum()
                    ),
                "Hours at 150 MW":
                    int(
                        np.isclose(
                            dc_schedule,
                            DAILY_DC_MAXIMUM_MW,
                            atol=1e-5,
                        ).sum()
                    ),
                "Inflexible VRE Curtailment MWh":
                    inflexible_vre_curtailment,
                "Flexible VRE Curtailment MWh":
                    flexible_vre_curtailment,
                "VRE Curtailment Reduction MWh":
                    (
                        inflexible_vre_curtailment
                        - flexible_vre_curtailment
                    ),
                "Inflexible C6 Binding Hours":
                    inflexible_c6_binding_hours,
                "Flexible C6 Binding Hours":
                    flexible_c6_binding_hours,
                "Maximum Constraint Violation":
                    maximum_constraint_violation,
                "Maximum-Violation Constraint":
                    maximum_constraint_name,
                "Maximum System Balance Residual MW":
                    max(
                        hourly_system_balance_residuals
                    ),
                "Cost-Cap Tolerance $":
                    lexicographic_result[
                        "Cost-Cap Tolerance $"
                    ],
            }
        )

        completed_model_runs += 1

        if (
            completed_model_runs % 10 == 0
            or completed_model_runs
            == total_model_runs
        ):
            print(
                f"Completed {completed_model_runs:,}/"
                f"{total_model_runs:,} "
                "representative daily models"
            )

        del daily_model
        del daily_solver
        gc.collect()


representative_runtime_seconds = (
    perf_counter()
    - representative_start_time
)


# ---------------------------------------------------------------------
# 4. Assemble representative-day results
# ---------------------------------------------------------------------

representative_daily_results = (
    pd.DataFrame(
        representative_daily_rows
    )
    .set_index(
        ["Day", "Case"]
    )
    .sort_index()
)

representative_flexible_schedules = (
    pd.DataFrame(
        representative_schedule_rows
    )
    .set_index(
        ["Day", "Case", "Timestamp"]
    )
    .sort_index()
)


# ---------------------------------------------------------------------
# 5. Construct the daily 2 × 2 factorial effects
# ---------------------------------------------------------------------

representative_factorial_rows = []

for day in representative_days:
    cf_row = representative_daily_results.loc[
        (day, "CF")
    ]

    cfr_row = representative_daily_results.loc[
        (day, "CFR")
    ]

    c0_cost = float(
        cf_row["Inflexible Daily Cost $"]
    )

    cf_cost = float(
        cf_row["Flexible Final Cost $"]
    )

    cr_cost = float(
        cfr_row["Inflexible Daily Cost $"]
    )

    cfr_cost = float(
        cfr_row["Flexible Final Cost $"]
    )

    representative_factorial_rows.append(
        {
            "Day": day,
            "Selection Reason": "; ".join(
                representative_day_labels[day]
            ),
            "C0 Cost $": c0_cost,
            "CF Cost $": cf_cost,
            "CR Cost $": cr_cost,
            "CFR Cost $": cfr_cost,
            "Flexibility Value, Original Network $":
                c0_cost - cf_cost,
            "Flexibility Value, Reconductored Network $":
                cr_cost - cfr_cost,
            "Reconductoring Value, Inflexible Load $":
                c0_cost - cr_cost,
            "Reconductoring Value, Flexible Load $":
                cf_cost - cfr_cost,
            "Combined Value Relative to C0 $":
                c0_cost - cfr_cost,
            "Combined-Savings Interaction $":
                (
                    cf_cost
                    + cr_cost
                    - cfr_cost
                    - c0_cost
                ),
        }
    )

representative_factorial_effects = (
    pd.DataFrame(
        representative_factorial_rows
    )
    .set_index("Day")
)


# ---------------------------------------------------------------------
# 6. Diagnostic summaries
# ---------------------------------------------------------------------

effect_columns = [
    "Flexibility Value, Original Network $",
    "Flexibility Value, Reconductored Network $",
    "Reconductoring Value, Inflexible Load $",
    "Reconductoring Value, Flexible Load $",
    "Combined Value Relative to C0 $",
    "Combined-Savings Interaction $",
]

representative_effect_distribution = (
    representative_factorial_effects[
        effect_columns
    ]
    .agg(
        [
            "min",
            "median",
            "mean",
            "max",
        ]
    )
    .transpose()
)

representative_validation_summary = pd.DataFrame(
    {
        "Value": {
            "Representative days":
                len(representative_days),

            "Daily models solved":
                len(representative_daily_results),

            "Maximum daily-energy error MWh":
                float(
                    (
                        representative_daily_results[
                            "Daily DC Energy MWh"
                        ]
                        - DAILY_DC_ENERGY_MWH
                    )
                    .abs()
                    .max()
                ),

            "Minimum optimized DC power MW":
                float(
                    representative_daily_results[
                        "Minimum DC Power MW"
                    ].min()
                ),

            "Maximum optimized DC power MW":
                float(
                    representative_daily_results[
                        "Maximum DC Power MW"
                    ].max()
                ),

            "Maximum constraint violation":
                float(
                    representative_daily_results[
                        "Maximum Constraint Violation"
                    ].max()
                ),

            "Maximum system-balance residual MW":
                float(
                    representative_daily_results[
                        "Maximum System Balance Residual MW"
                    ].max()
                ),

            "Minimum flexibility savings $":
                float(
                    representative_daily_results[
                        "Flexibility Savings $"
                    ].min()
                ),

            "Representative runtime seconds":
                representative_runtime_seconds,
        }
    }
)


# ---------------------------------------------------------------------
# 7. Display results
# ---------------------------------------------------------------------

print("\nRepresentative-day validation summary:")
display(representative_validation_summary)

print("\nRepresentative daily model results:")
display(representative_daily_results)

print("\nRepresentative daily factorial effects:")
display(representative_factorial_effects)

print(
    "\nDiagnostic effect distribution "
    "(not an annual estimate):"
)
display(representative_effect_distribution)


# ---------------------------------------------------------------------
# 8. Hard validation gates
# ---------------------------------------------------------------------

maximum_daily_energy_error = float(
    (
        representative_daily_results[
            "Daily DC Energy MWh"
        ]
        - DAILY_DC_ENERGY_MWH
    )
    .abs()
    .max()
)

assert maximum_daily_energy_error <= VALIDATION_TOL

assert (
    representative_daily_results[
        "Minimum DC Power MW"
    ].min()
    >= DAILY_DC_MINIMUM_MW
    - VALIDATION_TOL
)

assert (
    representative_daily_results[
        "Maximum DC Power MW"
    ].max()
    <= DAILY_DC_MAXIMUM_MW
    + VALIDATION_TOL
)

assert (
    representative_daily_results[
        "Maximum Constraint Violation"
    ].max()
    <= VALIDATION_TOL
)

assert (
    representative_daily_results[
        "Maximum System Balance Residual MW"
    ].max()
    <= VALIDATION_TOL
)

assert (
    representative_daily_results[
        "Flexible Final Cost $"
    ]
    <= (
        representative_daily_results[
            "Inflexible Daily Cost $"
        ]
        + representative_daily_results[
            "Cost-Cap Tolerance $"
        ]
        + VALIDATION_TOL
    )
).all()

# The previously validated June 7 pilot must be reproduced.
for case_code in ["CF", "CFR"]:
    repeated_pilot_cost = float(
        representative_daily_results.loc[
            (
                flexible_pilot_day_start,
                case_code,
            ),
            "Flexible Final Cost $",
        ]
    )

    original_pilot_cost = float(
        daily_flexible_summary.loc[
            case_code,
            "Flexible Final Cost $",
        ]
    )

    assert abs(
        repeated_pilot_cost
        - original_pilot_cost
    ) <= 1e-3

print(
    "\nThe flexible formulation passed the complete "
    "representative-day regression test."
)

## Complete annual simulation of the four experimental cases

The validated daily formulation is applied independently to all 366 days of
2020. Each day is solved under both network configurations.

For each network configuration, the model first solves the constant 100 MW
workload and then the flexible 50–150 MW workload with the same 2,400 MWh
daily energy requirement. This produces the four annual cases C0, CF, CR,
and CFR.

Hourly generator dispatch, branch flows, data-center power, and operating
cost are stored in checkpointed disk-backed arrays. This preserves sufficient
detail for subsequent congestion, renewable-curtailment, generation-mix, and
economic analyses without requiring the annual optimization to be repeated.

In [ ]:
# Complete checkpointed annual simulation of C0, CF, CR, and CFR.

import gc
import json

from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import pyomo.environ as pyo

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Annual chronology and case structure
# ---------------------------------------------------------------------

ANNUAL_CASES = [
    "C0",
    "CF",
    "CR",
    "CFR",
]

annual_case_position = {
    case_code: position
    for position, case_code
    in enumerate(ANNUAL_CASES)
}

annual_timestamps = pd.DatetimeIndex(
    expected_annual_index
)

annual_days = pd.DatetimeIndex(
    annual_timestamps.normalize().unique()
)

assert len(annual_timestamps) == 8784
assert len(annual_days) == 366

number_of_cases = len(ANNUAL_CASES)
number_of_hours = len(annual_timestamps)
number_of_days = len(annual_days)
number_of_generators = len(GENERATOR_IDS)
number_of_branches = len(BRANCH_IDS)

annual_network_runs = {
    "Original": {
        "Fixed Case": "C0",
        "Flexible Case": "CF",
        "Flexible Model Code": "CF",
        "C6 Rating MW": float(
            BASELINE_C6_RATING_MW
        ),
    },
    "Reconductored": {
        "Fixed Case": "CR",
        "Flexible Case": "CFR",
        "Flexible Model Code": "CFR",
        "C6 Rating MW": float(
            SELECTED_C6_RATING_MW
        ),
    },
}


# ---------------------------------------------------------------------
# 2. Create the checkpoint directory
# ---------------------------------------------------------------------

ANNUAL_OUTPUT_DIRECTORY = Path(
    "notebook5_annual_outputs"
)

ANNUAL_OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Annual output directory: "
    f"{ANNUAL_OUTPUT_DIRECTORY.resolve()}"
)


# ---------------------------------------------------------------------
# 3. Open or create persistent NumPy arrays
# ---------------------------------------------------------------------

def open_or_create_memmap(
    filename,
    shape,
    dtype,
    fill_value,
):
    """
    Open an existing NumPy memory-mapped array or create and
    initialize it when it does not yet exist.
    """
    path = ANNUAL_OUTPUT_DIRECTORY / filename

    if path.exists():
        array = np.lib.format.open_memmap(
            path,
            mode="r+",
        )

        if tuple(array.shape) != tuple(shape):
            raise ValueError(
                f"Checkpoint shape mismatch for {path.name}: "
                f"found {array.shape}, expected {shape}."
            )

        if np.dtype(array.dtype) != np.dtype(dtype):
            raise ValueError(
                f"Checkpoint dtype mismatch for {path.name}: "
                f"found {array.dtype}, expected {dtype}."
            )

        return array

    array = np.lib.format.open_memmap(
        path,
        mode="w+",
        dtype=dtype,
        shape=shape,
    )

    array[...] = fill_value
    array.flush()

    return array


annual_generation_mw = open_or_create_memmap(
    "annual_generation_mw.npy",
    shape=(
        number_of_cases,
        number_of_hours,
        number_of_generators,
    ),
    dtype=np.float64,
    fill_value=np.nan,
)

annual_branch_flow_mw = open_or_create_memmap(
    "annual_branch_flow_mw.npy",
    shape=(
        number_of_cases,
        number_of_hours,
        number_of_branches,
    ),
    dtype=np.float64,
    fill_value=np.nan,
)

annual_dc_power_mw = open_or_create_memmap(
    "annual_dc_power_mw.npy",
    shape=(
        number_of_cases,
        number_of_hours,
    ),
    dtype=np.float64,
    fill_value=np.nan,
)

annual_operating_cost = open_or_create_memmap(
    "annual_operating_cost.npy",
    shape=(
        number_of_cases,
        number_of_hours,
    ),
    dtype=np.float64,
    fill_value=np.nan,
)

annual_daily_primary_cost = open_or_create_memmap(
    "annual_daily_primary_cost.npy",
    shape=(
        number_of_cases,
        number_of_days,
    ),
    dtype=np.float64,
    fill_value=np.nan,
)

annual_daily_final_cost = open_or_create_memmap(
    "annual_daily_final_cost.npy",
    shape=(
        number_of_cases,
        number_of_days,
    ),
    dtype=np.float64,
    fill_value=np.nan,
)

annual_daily_cost_cap_tolerance = (
    open_or_create_memmap(
        "annual_daily_cost_cap_tolerance.npy",
        shape=(
            number_of_cases,
            number_of_days,
        ),
        dtype=np.float64,
        fill_value=np.nan,
    )
)

annual_daily_constraint_violation = (
    open_or_create_memmap(
        "annual_daily_constraint_violation.npy",
        shape=(
            number_of_cases,
            number_of_days,
        ),
        dtype=np.float64,
        fill_value=np.nan,
    )
)

annual_case_day_completed = (
    open_or_create_memmap(
        "annual_case_day_completed.npy",
        shape=(
            number_of_cases,
            number_of_days,
        ),
        dtype=np.bool_,
        fill_value=False,
    )
)


# ---------------------------------------------------------------------
# 4. Save identifying metadata
# ---------------------------------------------------------------------

annual_metadata = {
    "schema_version": 1,
    "year": 2020,
    "cases": ANNUAL_CASES,
    "hours": number_of_hours,
    "days": number_of_days,
    "generators": number_of_generators,
    "branches": number_of_branches,
    "data_center_bus": NOTEBOOK5_DC_BUS,
    "daily_energy_mwh": DAILY_DC_ENERGY_MWH,
    "minimum_dc_power_mw": DAILY_DC_MINIMUM_MW,
    "maximum_dc_power_mw": DAILY_DC_MAXIMUM_MW,
    "c6_uid": SELECTED_RECONDUCTORING_UID,
    "c6_original_rating_mw":
        BASELINE_C6_RATING_MW,
    "c6_reconductored_rating_mw":
        SELECTED_C6_RATING_MW,
}

with (
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_run_metadata.json"
).open("w", encoding="utf-8") as metadata_file:
    json.dump(
        annual_metadata,
        metadata_file,
        indent=2,
    )

pd.Series(
    GENERATOR_IDS,
    name="GEN UID",
).to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "generator_ids.csv",
    index=False,
)

pd.Series(
    BRANCH_IDS,
    name="Branch UID",
).to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "branch_ids.csv",
    index=False,
)

pd.Series(
    annual_timestamps,
    name="Timestamp",
).to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "timestamps.csv",
    index=False,
)


# ---------------------------------------------------------------------
# 5. Checkpoint and extraction helpers
# ---------------------------------------------------------------------

annual_memmaps = [
    annual_generation_mw,
    annual_branch_flow_mw,
    annual_dc_power_mw,
    annual_operating_cost,
    annual_daily_primary_cost,
    annual_daily_final_cost,
    annual_daily_cost_cap_tolerance,
    annual_daily_constraint_violation,
    annual_case_day_completed,
]


def flush_annual_checkpoints():
    """Write every memory-mapped array to disk."""
    for checkpoint_array in annual_memmaps:
        checkpoint_array.flush()


def extract_and_store_case_day(
    model,
    case_code,
    day_position,
    daily_timestamps,
    primary_cost,
    final_cost,
    cost_cap_tolerance,
    constraint_violation,
):
    """
    Extract one solved case-day and write its hourly and daily
    results into the persistent annual arrays.
    """
    case_position = annual_case_position[
        case_code
    ]

    annual_hour_positions = (
        annual_timestamps.get_indexer(
            daily_timestamps
        )
    )

    if (annual_hour_positions < 0).any():
        raise KeyError(
            "At least one daily timestamp is absent from "
            "the annual chronology."
        )

    local_generation = np.empty(
        (
            24,
            number_of_generators,
        ),
        dtype=float,
    )

    local_branch_flow = np.empty(
        (
            24,
            number_of_branches,
        ),
        dtype=float,
    )

    local_dc_power = np.empty(
        24,
        dtype=float,
    )

    local_operating_cost = np.empty(
        24,
        dtype=float,
    )

    local_system_balance_residual = np.empty(
        24,
        dtype=float,
    )

    for hour_position in model.HOUR:
        local_hour = int(hour_position)

        timestamp = model.timestamp_by_hour[
            local_hour
        ]

        hourly_block = model.hour[
            hour_position
        ]

        local_generation[
            local_hour,
            :,
        ] = [
            pyo.value(
                hourly_block.generation[
                    generator_uid
                ]
            )
            for generator_uid
            in GENERATOR_IDS
        ]

        local_branch_flow[
            local_hour,
            :,
        ] = [
            pyo.value(
                hourly_block.flow[
                    branch_uid
                ]
            )
            for branch_uid
            in BRANCH_IDS
        ]

        local_dc_power[
            local_hour
        ] = float(
            pyo.value(
                model.dc_power[
                    hour_position
                ]
            )
        )

        local_operating_cost[
            local_hour
        ] = float(
            pyo.value(
                hourly_block.total_cost.expr
            )
        )

        native_demand = float(
            notebook5_native_hourly_bus_load.loc[
                timestamp
            ].sum()
        )

        local_system_balance_residual[
            local_hour
        ] = (
            local_generation[
                local_hour,
                :,
            ].sum()
            - native_demand
            - local_dc_power[
                local_hour
            ]
        )

    if not np.isfinite(
        local_generation
    ).all():
        raise ValueError(
            f"Non-finite generation in {case_code} "
            f"on {daily_timestamps[0].date()}."
        )

    if not np.isfinite(
        local_branch_flow
    ).all():
        raise ValueError(
            f"Non-finite branch flow in {case_code} "
            f"on {daily_timestamps[0].date()}."
        )

    daily_energy = float(
        local_dc_power.sum()
    )

    maximum_balance_residual = float(
        np.abs(
            local_system_balance_residual
        ).max()
    )

    if abs(
        daily_energy
        - DAILY_DC_ENERGY_MWH
    ) > VALIDATION_TOL:
        raise AssertionError(
            f"{case_code} daily energy mismatch on "
            f"{daily_timestamps[0].date()}: "
            f"{daily_energy:.12f} MWh."
        )

    if (
        local_dc_power.min()
        < DAILY_DC_MINIMUM_MW
        - VALIDATION_TOL
    ):
        raise AssertionError(
            f"{case_code} violates the minimum DC power."
        )

    if (
        local_dc_power.max()
        > DAILY_DC_MAXIMUM_MW
        + VALIDATION_TOL
    ):
        raise AssertionError(
            f"{case_code} violates the maximum DC power."
        )

    if (
        maximum_balance_residual
        > VALIDATION_TOL
    ):
        raise AssertionError(
            f"{case_code} system-balance residual on "
            f"{daily_timestamps[0].date()}: "
            f"{maximum_balance_residual:.12g} MW."
        )

    if constraint_violation > VALIDATION_TOL:
        raise AssertionError(
            f"{case_code} constraint violation on "
            f"{daily_timestamps[0].date()}: "
            f"{constraint_violation:.12g}."
        )

    hourly_cost_sum = float(
        local_operating_cost.sum()
    )

    if abs(
        hourly_cost_sum
        - final_cost
    ) > max(
        1e-4,
        abs(final_cost) * 1e-9,
    ):
        raise AssertionError(
            f"{case_code} hourly costs do not sum to "
            "the recorded daily cost."
        )

    annual_generation_mw[
        case_position,
        annual_hour_positions,
        :,
    ] = local_generation

    annual_branch_flow_mw[
        case_position,
        annual_hour_positions,
        :,
    ] = local_branch_flow

    annual_dc_power_mw[
        case_position,
        annual_hour_positions,
    ] = local_dc_power

    annual_operating_cost[
        case_position,
        annual_hour_positions,
    ] = local_operating_cost

    annual_daily_primary_cost[
        case_position,
        day_position,
    ] = primary_cost

    annual_daily_final_cost[
        case_position,
        day_position,
    ] = final_cost

    annual_daily_cost_cap_tolerance[
        case_position,
        day_position,
    ] = cost_cap_tolerance

    annual_daily_constraint_violation[
        case_position,
        day_position,
    ] = constraint_violation


# ---------------------------------------------------------------------
# 6. Determine completed work from previous attempts
# ---------------------------------------------------------------------

def network_day_is_complete(
    day_position,
    network_configuration,
):
    fixed_case = (
        network_configuration[
            "Fixed Case"
        ]
    )

    flexible_case = (
        network_configuration[
            "Flexible Case"
        ]
    )

    return bool(
        annual_case_day_completed[
            annual_case_position[fixed_case],
            day_position,
        ]
        and annual_case_day_completed[
            annual_case_position[flexible_case],
            day_position,
        ]
    )


total_network_day_models = (
    number_of_days
    * len(annual_network_runs)
)

initially_completed_models = sum(
    network_day_is_complete(
        day_position,
        network_configuration,
    )
    for day_position in range(
        number_of_days
    )
    for network_configuration
    in annual_network_runs.values()
)

print(
    "Annual network-day models already complete: "
    f"{initially_completed_models:,}/"
    f"{total_network_day_models:,}"
)


# ---------------------------------------------------------------------
# 7. Run the complete annual experiment
# ---------------------------------------------------------------------

annual_run_start_time = perf_counter()
newly_completed_models = 0

for day_position, day in enumerate(
    annual_days
):
    daily_timestamps = pd.date_range(
        start=day,
        periods=24,
        freq="h",
    )

    for (
        network_name,
        network_configuration,
    ) in annual_network_runs.items():

        if network_day_is_complete(
            day_position,
            network_configuration,
        ):
            continue

        fixed_case = network_configuration[
            "Fixed Case"
        ]

        flexible_case = network_configuration[
            "Flexible Case"
        ]

        flexible_model_code = (
            network_configuration[
                "Flexible Model Code"
            ]
        )

        daily_model = build_daily_flexible_model(
            case_code=flexible_model_code,
            daily_timestamps=daily_timestamps,
        )

        daily_solver = pyo.SolverFactory(
            "appsi_highs"
        )

        if not daily_solver.available(
            exception_flag=False
        ):
            raise RuntimeError(
                "The appsi_highs solver is unavailable."
            )

        # -------------------------------------------------------------
        # 7A. Constant 100 MW case: C0 or CR
        # -------------------------------------------------------------

        for hour_position in daily_model.HOUR:
            daily_model.dc_power[
                hour_position
            ].fix(
                NOTEBOOK5_INFLEXIBLE_DC_POWER_MW
            )

        fixed_daily_cost = (
            solve_daily_primary_objective(
                daily_model,
                daily_solver,
            )
        )

        (
            fixed_constraint_violation,
            fixed_constraint_name,
        ) = maximum_active_constraint_violation(
            daily_model
        )

        extract_and_store_case_day(
            model=daily_model,
            case_code=fixed_case,
            day_position=day_position,
            daily_timestamps=daily_timestamps,
            primary_cost=fixed_daily_cost,
            final_cost=fixed_daily_cost,
            cost_cap_tolerance=0.0,
            constraint_violation=(
                fixed_constraint_violation
            ),
        )

        # -------------------------------------------------------------
        # 7B. Flexible case: CF or CFR
        # -------------------------------------------------------------

        for hour_position in daily_model.HOUR:
            daily_model.dc_power[
                hour_position
            ].unfix()

        flexible_solve_result = (
            solve_daily_lexicographically(
                daily_model,
                daily_solver,
            )
        )

        flexible_primary_cost = float(
            flexible_solve_result[
                "Primary Optimal Cost $"
            ]
        )

        flexible_final_cost = float(
            flexible_solve_result[
                "Final Lexicographic Cost $"
            ]
        )

        flexible_cost_tolerance = float(
            flexible_solve_result[
                "Cost-Cap Tolerance $"
            ]
        )

        (
            flexible_constraint_violation,
            flexible_constraint_name,
        ) = maximum_active_constraint_violation(
            daily_model
        )

        if flexible_final_cost > (
            fixed_daily_cost
            + flexible_cost_tolerance
            + VALIDATION_TOL
        ):
            raise AssertionError(
                f"{flexible_case} is more expensive than "
                f"{fixed_case} on {day.date()}."
            )

        extract_and_store_case_day(
            model=daily_model,
            case_code=flexible_case,
            day_position=day_position,
            daily_timestamps=daily_timestamps,
            primary_cost=flexible_primary_cost,
            final_cost=flexible_final_cost,
            cost_cap_tolerance=(
                flexible_cost_tolerance
            ),
            constraint_violation=(
                flexible_constraint_violation
            ),
        )

        # Mark both cases complete only after all extraction and
        # validation steps have succeeded.
        annual_case_day_completed[
            annual_case_position[
                fixed_case
            ],
            day_position,
        ] = True

        annual_case_day_completed[
            annual_case_position[
                flexible_case
            ],
            day_position,
        ] = True

        newly_completed_models += 1

        total_completed_models = (
            initially_completed_models
            + newly_completed_models
        )

        if (
            newly_completed_models % 10 == 0
            or total_completed_models
            == total_network_day_models
        ):
            flush_annual_checkpoints()

            elapsed_seconds = (
                perf_counter()
                - annual_run_start_time
            )

            average_seconds_per_model = (
                elapsed_seconds
                / newly_completed_models
            )

            remaining_models = (
                total_network_day_models
                - total_completed_models
            )

            estimated_remaining_minutes = (
                average_seconds_per_model
                * remaining_models
                / 60.0
            )

            print(
                f"Completed "
                f"{total_completed_models:,}/"
                f"{total_network_day_models:,} "
                "annual network-day models | "
                f"Estimated remaining time: "
                f"{estimated_remaining_minutes:.1f} minutes"
            )

        del daily_model
        del daily_solver
        gc.collect()


flush_annual_checkpoints()

annual_runtime_seconds = (
    perf_counter()
    - annual_run_start_time
)

assert annual_case_day_completed.all()

print(
    "\nAll annual case-days have been solved and "
    "checkpointed successfully."
)


# ---------------------------------------------------------------------
# 8. Build the complete hourly results table
# ---------------------------------------------------------------------

native_demand_hourly = (
    notebook5_native_hourly_bus_load
    .reindex(annual_timestamps)
    .sum(axis=1)
    .to_numpy(dtype=float)
)

generator_position = {
    generator_uid: position
    for position, generator_uid
    in enumerate(GENERATOR_IDS)
}

branch_position = {
    branch_uid: position
    for position, branch_uid
    in enumerate(BRANCH_IDS)
}

vre_generator_positions = [
    generator_position[generator_uid]
    for generator_uid
    in variable_renewable_generator_ids
]

c6_branch_position = branch_position[
    SELECTED_RECONDUCTORING_UID
]

generation_array = np.asarray(
    annual_generation_mw
)

branch_flow_array = np.asarray(
    annual_branch_flow_mw
)

dc_power_array = np.asarray(
    annual_dc_power_mw
)

operating_cost_array = np.asarray(
    annual_operating_cost
)

total_generation_array = (
    generation_array.sum(axis=2)
)

vre_dispatch_array = (
    generation_array[
        :,
        :,
        vre_generator_positions,
    ].sum(axis=2)
)

vre_available_hourly = (
    hourly_gen_pmax
    .reindex(annual_timestamps)
    .loc[
        :,
        variable_renewable_generator_ids,
    ]
    .sum(axis=1)
    .to_numpy(dtype=float)
)

case_c6_ratings = np.array(
    [
        BASELINE_C6_RATING_MW,
        BASELINE_C6_RATING_MW,
        SELECTED_C6_RATING_MW,
        SELECTED_C6_RATING_MW,
    ],
    dtype=float,
)

c6_flow_array = branch_flow_array[
    :,
    :,
    c6_branch_position,
]

system_balance_residual_array = (
    total_generation_array
    - native_demand_hourly[None, :]
    - dc_power_array
)

annual_hourly_index = pd.MultiIndex.from_product(
    [
        ANNUAL_CASES,
        annual_timestamps,
    ],
    names=[
        "Case",
        "Timestamp",
    ],
)

annual_hourly_case_summary = pd.DataFrame(
    {
        "Native Demand MW":
            np.tile(
                native_demand_hourly,
                number_of_cases,
            ),

        "DC Power MW":
            dc_power_array.reshape(-1),

        "Total Demand MW":
            (
                native_demand_hourly[None, :]
                + dc_power_array
            ).reshape(-1),

        "Total Generation MW":
            total_generation_array.reshape(-1),

        "System Balance Residual MW":
            system_balance_residual_array.reshape(
                -1
            ),

        "Operating Cost $/h":
            operating_cost_array.reshape(-1),

        "VRE Available MW":
            np.tile(
                vre_available_hourly,
                number_of_cases,
            ),

        "VRE Dispatched MW":
            vre_dispatch_array.reshape(-1),

        "VRE Curtailment MW":
            np.maximum(
                vre_available_hourly[None, :]
                - vre_dispatch_array,
                0.0,
            ).reshape(-1),

        "C6 Flow MW":
            c6_flow_array.reshape(-1),

        "C6 Rating MW":
            np.repeat(
                case_c6_ratings,
                number_of_hours,
            ),

        "C6 Loading %":
            (
                100.0
                * np.abs(c6_flow_array)
                / case_c6_ratings[:, None]
            ).reshape(-1),
    },
    index=annual_hourly_index,
)


# ---------------------------------------------------------------------
# 9. Annual case summary
# ---------------------------------------------------------------------

annual_case_rows = []

for case_position, case_code in enumerate(
    ANNUAL_CASES
):
    case_hourly = (
        annual_hourly_case_summary.loc[
            case_code
        ]
    )

    annual_case_rows.append(
        {
            "Case": case_code,
            "Network Configuration": (
                "Original"
                if case_code in ["C0", "CF"]
                else "Reconductored"
            ),
            "Workload Treatment": (
                "Inflexible"
                if case_code in ["C0", "CR"]
                else "Flexible"
            ),
            "Annual Operating Cost $":
                case_hourly[
                    "Operating Cost $/h"
                ].sum(),
            "Annual DC Energy MWh":
                case_hourly[
                    "DC Power MW"
                ].sum(),
            "Average DC Power MW":
                case_hourly[
                    "DC Power MW"
                ].mean(),
            "Minimum DC Power MW":
                case_hourly[
                    "DC Power MW"
                ].min(),
            "Maximum DC Power MW":
                case_hourly[
                    "DC Power MW"
                ].max(),
            "Annual VRE Available MWh":
                case_hourly[
                    "VRE Available MW"
                ].sum(),
            "Annual VRE Dispatched MWh":
                case_hourly[
                    "VRE Dispatched MW"
                ].sum(),
            "Annual VRE Curtailment MWh":
                case_hourly[
                    "VRE Curtailment MW"
                ].sum(),
            "VRE Curtailment Rate %":
                (
                    100.0
                    * case_hourly[
                        "VRE Curtailment MW"
                    ].sum()
                    / case_hourly[
                        "VRE Available MW"
                    ].sum()
                ),
            "C6 Binding Hours":
                int(
                    (
                        case_hourly[
                            "C6 Loading %"
                        ]
                        >= 100.0 - 1e-5
                    ).sum()
                ),
            "Maximum C6 Loading %":
                case_hourly[
                    "C6 Loading %"
                ].max(),
            "Maximum System Balance Residual MW":
                case_hourly[
                    "System Balance Residual MW"
                ].abs().max(),
            "Maximum Daily Constraint Violation":
                float(
                    annual_daily_constraint_violation[
                        case_position,
                        :,
                    ].max()
                ),
        }
    )

annual_case_summary = (
    pd.DataFrame(
        annual_case_rows
    )
    .set_index("Case")
)


# ---------------------------------------------------------------------
# 10. Annual factorial effects
# ---------------------------------------------------------------------

annual_costs = (
    annual_case_summary[
        "Annual Operating Cost $"
    ]
)

c0_annual_cost = float(
    annual_costs.loc["C0"]
)
cf_annual_cost = float(
    annual_costs.loc["CF"]
)
cr_annual_cost = float(
    annual_costs.loc["CR"]
)
cfr_annual_cost = float(
    annual_costs.loc["CFR"]
)

annual_factorial_effects = pd.DataFrame(
    {
        "Value $": {
            "Flexibility value, original network":
                c0_annual_cost - cf_annual_cost,

            "Flexibility value, reconductored network":
                cr_annual_cost - cfr_annual_cost,

            "Reconductoring value, inflexible load":
                c0_annual_cost - cr_annual_cost,

            "Reconductoring value, flexible load":
                cf_annual_cost - cfr_annual_cost,

            "Combined value relative to C0":
                c0_annual_cost - cfr_annual_cost,

            "Combined-savings interaction":
                (
                    cf_annual_cost
                    + cr_annual_cost
                    - cfr_annual_cost
                    - c0_annual_cost
                ),
        },
        "Percent of Relevant Baseline": {
            "Flexibility value, original network":
                (
                    100.0
                    * (c0_annual_cost - cf_annual_cost)
                    / c0_annual_cost
                ),

            "Flexibility value, reconductored network":
                (
                    100.0
                    * (cr_annual_cost - cfr_annual_cost)
                    / cr_annual_cost
                ),

            "Reconductoring value, inflexible load":
                (
                    100.0
                    * (c0_annual_cost - cr_annual_cost)
                    / c0_annual_cost
                ),

            "Reconductoring value, flexible load":
                (
                    100.0
                    * (cf_annual_cost - cfr_annual_cost)
                    / cf_annual_cost
                ),

            "Combined value relative to C0":
                (
                    100.0
                    * (c0_annual_cost - cfr_annual_cost)
                    / c0_annual_cost
                ),

            "Combined-savings interaction":
                (
                    100.0
                    * (
                        cf_annual_cost
                        + cr_annual_cost
                        - cfr_annual_cost
                        - c0_annual_cost
                    )
                    / c0_annual_cost
                ),
        },
    }
)


# ---------------------------------------------------------------------
# 11. Construct and save daily summaries
# ---------------------------------------------------------------------

annual_daily_rows = []

for case_position, case_code in enumerate(
    ANNUAL_CASES
):
    for day_position, day in enumerate(
        annual_days
    ):
        hour_start = day_position * 24
        hour_stop = hour_start + 24

        annual_daily_rows.append(
            {
                "Case": case_code,
                "Day": day,
                "Primary Cost $":
                    annual_daily_primary_cost[
                        case_position,
                        day_position,
                    ],
                "Final Cost $":
                    annual_daily_final_cost[
                        case_position,
                        day_position,
                    ],
                "Cost-Cap Tolerance $":
                    annual_daily_cost_cap_tolerance[
                        case_position,
                        day_position,
                    ],
                "Daily DC Energy MWh":
                    annual_dc_power_mw[
                        case_position,
                        hour_start:hour_stop,
                    ].sum(),
                "Minimum DC Power MW":
                    annual_dc_power_mw[
                        case_position,
                        hour_start:hour_stop,
                    ].min(),
                "Maximum DC Power MW":
                    annual_dc_power_mw[
                        case_position,
                        hour_start:hour_stop,
                    ].max(),
                "Maximum Constraint Violation":
                    annual_daily_constraint_violation[
                        case_position,
                        day_position,
                    ],
            }
        )

annual_daily_case_summary = (
    pd.DataFrame(
        annual_daily_rows
    )
    .set_index(
        ["Case", "Day"]
    )
    .sort_index()
)


# ---------------------------------------------------------------------
# 12. Final annual validation
# ---------------------------------------------------------------------

annual_daily_dc_energy = (
    dc_power_array.reshape(
        number_of_cases,
        number_of_days,
        24,
    ).sum(axis=2)
)

maximum_annual_daily_energy_error = float(
    np.abs(
        annual_daily_dc_energy
        - DAILY_DC_ENERGY_MWH
    ).max()
)

maximum_annual_balance_residual = float(
    np.abs(
        system_balance_residual_array
    ).max()
)

maximum_annual_constraint_violation = float(
    annual_daily_constraint_violation.max()
)

assert maximum_annual_daily_energy_error <= (
    VALIDATION_TOL
)

assert dc_power_array.min() >= (
    DAILY_DC_MINIMUM_MW
    - VALIDATION_TOL
)

assert dc_power_array.max() <= (
    DAILY_DC_MAXIMUM_MW
    + VALIDATION_TOL
)

assert maximum_annual_balance_residual <= (
    VALIDATION_TOL
)

assert maximum_annual_constraint_violation <= (
    VALIDATION_TOL
)

assert (
    annual_case_summary.loc[
        "CF",
        "Annual Operating Cost $",
    ]
    <= annual_case_summary.loc[
        "C0",
        "Annual Operating Cost $",
    ]
    + annual_daily_cost_cap_tolerance[
        annual_case_position["CF"],
        :,
    ].sum()
    + VALIDATION_TOL
)

assert (
    annual_case_summary.loc[
        "CFR",
        "Annual Operating Cost $",
    ]
    <= annual_case_summary.loc[
        "CR",
        "Annual Operating Cost $",
    ]
    + annual_daily_cost_cap_tolerance[
        annual_case_position["CFR"],
        :,
    ].sum()
    + VALIDATION_TOL
)


# ---------------------------------------------------------------------
# 13. Save tabular outputs
# ---------------------------------------------------------------------

annual_hourly_case_summary.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_hourly_case_summary.csv"
)

annual_daily_case_summary.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_daily_case_summary.csv"
)

annual_case_summary.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_case_summary.csv"
)

annual_factorial_effects.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_factorial_effects.csv"
)


# ---------------------------------------------------------------------
# 14. Display the first annual results
# ---------------------------------------------------------------------

annual_validation_summary = pd.DataFrame(
    {
        "Value": {
            "Annual cases completed":
                int(
                    annual_case_day_completed.all(
                        axis=1
                    ).sum()
                ),
            "Case-days completed":
                int(
                    annual_case_day_completed.sum()
                ),
            "Maximum daily-energy error MWh":
                maximum_annual_daily_energy_error,
            "Minimum DC power MW":
                float(dc_power_array.min()),
            "Maximum DC power MW":
                float(dc_power_array.max()),
            "Maximum system-balance residual MW":
                maximum_annual_balance_residual,
            "Maximum constraint violation":
                maximum_annual_constraint_violation,
            "New runtime seconds":
                annual_runtime_seconds,
            "Output directory":
                str(
                    ANNUAL_OUTPUT_DIRECTORY.resolve()
                ),
        }
    }
)

print("\nAnnual validation summary:")
display(annual_validation_summary)

print("\nAnnual case summary:")
display(annual_case_summary)

print("\nAnnual factorial effects:")
display(annual_factorial_effects)

print(
    "\nThe complete four-case annual experiment "
    "passed all energy, power-bound, network, and "
    "physical-feasibility validations."
)

## Copper-plate and congestion-cost decomposition

A copper-plate benchmark is calculated for every hour and experimental case.
The benchmark preserves hourly generator availability, generator marginal
costs, and total system demand, but removes all transmission constraints.

Hourly congestion cost is defined as the difference between the
network-constrained DC-OPF cost and the corresponding copper-plate cost.

Before applying the calculation to the four experimental cases, the
vectorized merit-order implementation is validated against the previously
verified native-system copper-plate results from Notebook 4.

In [ ]:
# Calculate hourly copper-plate and congestion costs for all four cases.

import numpy as np
import pandas as pd

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Prepare generator costs and hourly availability in merit order
# ---------------------------------------------------------------------

annual_generator_availability = (
    hourly_gen_pmax
    .reindex(
        index=annual_timestamps,
        columns=GENERATOR_IDS,
    )
    .to_numpy(dtype=float)
)

annual_generator_costs = np.asarray(
    generator_cost_array,
    dtype=float,
)

annual_merit_order = np.asarray(
    merit_order_positions,
    dtype=int,
)

if annual_generator_availability.shape != (
    number_of_hours,
    number_of_generators,
):
    raise ValueError(
        "Unexpected generator-availability matrix shape."
    )

if annual_generator_costs.shape != (
    number_of_generators,
):
    raise ValueError(
        "Unexpected generator-cost vector shape."
    )

availability_in_merit_order = (
    annual_generator_availability[
        :,
        annual_merit_order,
    ]
)

cost_in_merit_order = (
    annual_generator_costs[
        annual_merit_order
    ]
)

cumulative_availability = np.cumsum(
    availability_in_merit_order,
    axis=1,
)

availability_before_generator = (
    cumulative_availability
    - availability_in_merit_order
)

total_available_generation = (
    cumulative_availability[:, -1]
)


# ---------------------------------------------------------------------
# 2. Vectorized copper-plate dispatch function
# ---------------------------------------------------------------------

def calculate_vectorized_copper_plate_cost(
    hourly_demand_mw,
):
    """
    Dispatch available generation in ascending marginal-cost order.

    Each row represents one hour. Transmission constraints are absent,
    while generator availability and total hourly demand are preserved.
    """
    hourly_demand_mw = np.asarray(
        hourly_demand_mw,
        dtype=float,
    )

    if hourly_demand_mw.shape != (
        number_of_hours,
    ):
        raise ValueError(
            "Hourly demand must have one value for each "
            "annual timestamp."
        )

    unserved_energy = np.maximum(
        hourly_demand_mw
        - total_available_generation,
        0.0,
    )

    maximum_unserved_energy = float(
        unserved_energy.max()
    )

    if maximum_unserved_energy > VALIDATION_TOL:
        infeasible_hour_position = int(
            np.argmax(unserved_energy)
        )

        raise RuntimeError(
            "Copper-plate demand exceeds available generation at "
            f"{annual_timestamps[infeasible_hour_position]}. "
            f"Shortfall: {maximum_unserved_energy:.6f} MW."
        )

    remaining_demand_before_generator = (
        hourly_demand_mw[:, None]
        - availability_before_generator
    )

    copper_plate_dispatch = np.minimum(
        availability_in_merit_order,
        np.maximum(
            remaining_demand_before_generator,
            0.0,
        ),
    )

    copper_plate_generation = (
        copper_plate_dispatch.sum(axis=1)
    )

    balance_residual = (
        copper_plate_generation
        - hourly_demand_mw
    )

    maximum_balance_residual = float(
        np.abs(balance_residual).max()
    )

    if maximum_balance_residual > VALIDATION_TOL:
        raise AssertionError(
            "Copper-plate dispatch does not balance demand. "
            f"Maximum residual: "
            f"{maximum_balance_residual:.12g} MW."
        )

    hourly_copper_plate_cost = (
        copper_plate_dispatch
        @ cost_in_merit_order
    )

    return {
        "Cost $/h":
            hourly_copper_plate_cost,
        "Generation MW":
            copper_plate_generation,
        "Maximum Balance Residual MW":
            maximum_balance_residual,
        "Maximum Unserved Energy MW":
            maximum_unserved_energy,
    }


# ---------------------------------------------------------------------
# 3. Validate against the native Notebook 4 copper-plate benchmark
# ---------------------------------------------------------------------

native_copper_plate_checkpoint = (
    annual_hourly_summary[
        "Primary network cost $/h"
    ].reindex(annual_timestamps)
    - annual_hourly_summary[
        "Congestion cost $/h"
    ].reindex(annual_timestamps)
).to_numpy(dtype=float)

native_copper_plate_result = (
    calculate_vectorized_copper_plate_cost(
        native_demand_hourly
    )
)

native_copper_plate_calculated = (
    native_copper_plate_result[
        "Cost $/h"
    ]
)

native_copper_plate_absolute_difference = (
    np.abs(
        native_copper_plate_calculated
        - native_copper_plate_checkpoint
    )
)

native_copper_plate_allowed_difference = (
    1e-4
    + 1e-9
    * np.abs(
        native_copper_plate_checkpoint
    )
)

native_copper_plate_validation = pd.DataFrame(
    {
        "Value": {
            "Hours validated":
                number_of_hours,

            "Maximum absolute cost difference $/h":
                float(
                    native_copper_plate_absolute_difference.max()
                ),

            "Mean absolute cost difference $/h":
                float(
                    native_copper_plate_absolute_difference.mean()
                ),

            "Maximum balance residual MW":
                native_copper_plate_result[
                    "Maximum Balance Residual MW"
                ],

            "Maximum unserved energy MW":
                native_copper_plate_result[
                    "Maximum Unserved Energy MW"
                ],

            "Hours exceeding numerical tolerance":
                int(
                    (
                        native_copper_plate_absolute_difference
                        > native_copper_plate_allowed_difference
                    ).sum()
                ),
        }
    }
)

print("Native copper-plate regression validation:")
display(native_copper_plate_validation)

assert (
    native_copper_plate_absolute_difference
    <= native_copper_plate_allowed_difference
).all(), (
    "The vectorized copper-plate calculation does not reproduce "
    "the validated Notebook 4 benchmark."
)


# ---------------------------------------------------------------------
# 4. Calculate copper-plate costs for C0, CF, CR, and CFR
# ---------------------------------------------------------------------

annual_total_demand_array = (
    native_demand_hourly[None, :]
    + dc_power_array
)

annual_copper_plate_cost = np.empty(
    (
        number_of_cases,
        number_of_hours,
    ),
    dtype=float,
)

annual_copper_plate_balance_residual = (
    np.empty(
        number_of_cases,
        dtype=float,
    )
)

for case_position, case_code in enumerate(
    ANNUAL_CASES
):
    case_copper_plate_result = (
        calculate_vectorized_copper_plate_cost(
            annual_total_demand_array[
                case_position,
                :,
            ]
        )
    )

    annual_copper_plate_cost[
        case_position,
        :,
    ] = case_copper_plate_result[
        "Cost $/h"
    ]

    annual_copper_plate_balance_residual[
        case_position
    ] = case_copper_plate_result[
        "Maximum Balance Residual MW"
    ]


# ---------------------------------------------------------------------
# 5. Calculate hourly congestion cost
# ---------------------------------------------------------------------

raw_annual_congestion_cost = (
    operating_cost_array
    - annual_copper_plate_cost
)

minimum_raw_congestion_cost = float(
    raw_annual_congestion_cost.min()
)

CONGESTION_NUMERICAL_TOLERANCE = 1e-3

if minimum_raw_congestion_cost < (
    -CONGESTION_NUMERICAL_TOLERANCE
):
    minimum_position = np.unravel_index(
        np.argmin(
            raw_annual_congestion_cost
        ),
        raw_annual_congestion_cost.shape,
    )

    raise AssertionError(
        "A network-constrained cost is materially below its "
        "copper-plate benchmark. "
        f"Case: {ANNUAL_CASES[minimum_position[0]]}; "
        f"timestamp: {annual_timestamps[minimum_position[1]]}; "
        f"difference: {minimum_raw_congestion_cost:.12g} $/h."
    )

annual_congestion_cost = np.maximum(
    raw_annual_congestion_cost,
    0.0,
)


# ---------------------------------------------------------------------
# 6. Add the decomposition to the hourly results
# ---------------------------------------------------------------------

annual_hourly_case_summary[
    "Copper-Plate Cost $/h"
] = annual_copper_plate_cost.reshape(-1)

annual_hourly_case_summary[
    "Congestion Cost $/h"
] = annual_congestion_cost.reshape(-1)

annual_hourly_case_summary[
    "Congestion Share %"
] = np.where(
    operating_cost_array.reshape(-1) > 0.0,
    (
        100.0
        * annual_congestion_cost.reshape(-1)
        / operating_cost_array.reshape(-1)
    ),
    0.0,
)


# ---------------------------------------------------------------------
# 7. Extend the annual case summary
# ---------------------------------------------------------------------

annual_copper_plate_cost_by_case = (
    annual_hourly_case_summary[
        "Copper-Plate Cost $/h"
    ]
    .groupby(level="Case")
    .sum()
    .reindex(ANNUAL_CASES)
)

annual_congestion_cost_by_case = (
    annual_hourly_case_summary[
        "Congestion Cost $/h"
    ]
    .groupby(level="Case")
    .sum()
    .reindex(ANNUAL_CASES)
)

annual_case_summary[
    "Annual Copper-Plate Cost $"
] = annual_copper_plate_cost_by_case

annual_case_summary[
    "Annual Congestion Cost $"
] = annual_congestion_cost_by_case

annual_case_summary[
    "Congestion Share of Operating Cost %"
] = (
    100.0
    * annual_case_summary[
        "Annual Congestion Cost $"
    ]
    / annual_case_summary[
        "Annual Operating Cost $"
    ]
)

annual_case_summary[
    "Hours with Positive Congestion Cost"
] = [
    int(
        (
            annual_hourly_case_summary.loc[
                case_code,
                "Congestion Cost $/h",
            ]
            > CONGESTION_NUMERICAL_TOLERANCE
        ).sum()
    )
    for case_code in ANNUAL_CASES
]


# ---------------------------------------------------------------------
# 8. Factorial effects by cost component
# ---------------------------------------------------------------------

def calculate_factorial_effects(
    case_value_series,
):
    """Return the six treatment contrasts for one outcome."""
    case_value_series = (
        case_value_series.reindex(
            ANNUAL_CASES
        )
    )

    c0_value = float(
        case_value_series.loc["C0"]
    )
    cf_value = float(
        case_value_series.loc["CF"]
    )
    cr_value = float(
        case_value_series.loc["CR"]
    )
    cfr_value = float(
        case_value_series.loc["CFR"]
    )

    return {
        "Flexibility value, original network":
            c0_value - cf_value,

        "Flexibility value, reconductored network":
            cr_value - cfr_value,

        "Reconductoring value, inflexible load":
            c0_value - cr_value,

        "Reconductoring value, flexible load":
            cf_value - cfr_value,

        "Combined value relative to C0":
            c0_value - cfr_value,

        "Combined-savings interaction":
            (
                cf_value
                + cr_value
                - cfr_value
                - c0_value
            ),
    }


operating_cost_effects = (
    calculate_factorial_effects(
        annual_case_summary[
            "Annual Operating Cost $"
        ]
    )
)

copper_plate_cost_effects = (
    calculate_factorial_effects(
        annual_case_summary[
            "Annual Copper-Plate Cost $"
        ]
    )
)

congestion_cost_effects = (
    calculate_factorial_effects(
        annual_case_summary[
            "Annual Congestion Cost $"
        ]
    )
)

annual_cost_component_effects = pd.DataFrame(
    {
        "Operating-Cost Effect $":
            operating_cost_effects,

        "Copper-Plate Cost Effect $":
            copper_plate_cost_effects,

        "Congestion-Cost Effect $":
            congestion_cost_effects,
    }
)

annual_cost_component_effects[
    "Decomposition Difference $"
] = (
    annual_cost_component_effects[
        "Operating-Cost Effect $"
    ]
    - (
        annual_cost_component_effects[
            "Copper-Plate Cost Effect $"
        ]
        + annual_cost_component_effects[
            "Congestion-Cost Effect $"
        ]
    )
)


# ---------------------------------------------------------------------
# 9. Decomposition validation
# ---------------------------------------------------------------------

annual_cost_reconstruction_error = (
    annual_case_summary[
        "Annual Operating Cost $"
    ]
    - (
        annual_case_summary[
            "Annual Copper-Plate Cost $"
        ]
        + annual_case_summary[
            "Annual Congestion Cost $"
        ]
    )
)

maximum_case_cost_reconstruction_error = float(
    annual_cost_reconstruction_error.abs().max()
)

maximum_effect_decomposition_error = float(
    annual_cost_component_effects[
        "Decomposition Difference $"
    ].abs().max()
)

assert maximum_case_cost_reconstruction_error <= 1e-3
assert maximum_effect_decomposition_error <= 1e-3


# ---------------------------------------------------------------------
# 10. Save updated outputs
# ---------------------------------------------------------------------

np.save(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_copper_plate_cost.npy",
    annual_copper_plate_cost,
)

np.save(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_congestion_cost.npy",
    annual_congestion_cost,
)

annual_hourly_case_summary.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_hourly_case_summary.csv"
)

annual_case_summary.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_case_summary.csv"
)

annual_cost_component_effects.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_cost_component_effects.csv"
)


# ---------------------------------------------------------------------
# 11. Display results
# ---------------------------------------------------------------------

congestion_decomposition_validation = pd.DataFrame(
    {
        "Value": {
            "Native hours validated":
                number_of_hours,

            "Maximum native copper-plate difference $/h":
                float(
                    native_copper_plate_absolute_difference.max()
                ),

            "Minimum raw experimental congestion cost $/h":
                minimum_raw_congestion_cost,

            "Maximum copper-plate balance residual MW":
                float(
                    annual_copper_plate_balance_residual.max()
                ),

            "Maximum annual-case reconstruction error $":
                maximum_case_cost_reconstruction_error,

            "Maximum factorial-effect decomposition error $":
                maximum_effect_decomposition_error,
        }
    }
)

print("\nCongestion-decomposition validation:")
display(congestion_decomposition_validation)

print("\nAnnual cost decomposition by case:")
display(
    annual_case_summary[
        [
            "Annual Operating Cost $",
            "Annual Copper-Plate Cost $",
            "Annual Congestion Cost $",
            "Congestion Share of Operating Cost %",
            "Hours with Positive Congestion Cost",
            "C6 Binding Hours",
        ]
    ]
)

print("\nTreatment effects separated by cost component:")
display(annual_cost_component_effects)

print(
    "\nThe annual operating costs were successfully "
    "decomposed into copper-plate generation costs and "
    "transmission-congestion costs."
)

## Temporal and physical mechanisms behind the annual results

The annual results are disaggregated by month, day, hour of day, generation
fuel, and transmission branch.

This analysis identifies when each treatment is most valuable, how flexible
workload is scheduled, which generation sources change, and whether relieving
C6 transfers congestion to other parts of the network.

In [ ]:
# Analyze monthly, daily, hourly, generation-mix, and network mechanisms.

import numpy as np
import pandas as pd

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Replace the warning-producing hourly congestion-share calculation
# ---------------------------------------------------------------------

safe_hourly_congestion_share = np.zeros_like(
    operating_cost_array,
    dtype=float,
)

np.divide(
    100.0 * annual_congestion_cost,
    operating_cost_array,
    out=safe_hourly_congestion_share,
    where=operating_cost_array > 0.0,
)

annual_hourly_case_summary[
    "Congestion Share %"
] = safe_hourly_congestion_share.reshape(-1)

assert np.isfinite(
    annual_hourly_case_summary[
        "Congestion Share %"
    ]
).all()


# ---------------------------------------------------------------------
# 2. Prepare a flat hourly analysis table
# ---------------------------------------------------------------------

annual_hourly_analysis = (
    annual_hourly_case_summary
    .reset_index()
)

annual_hourly_analysis["Day"] = (
    annual_hourly_analysis[
        "Timestamp"
    ].dt.normalize()
)

annual_hourly_analysis["Month"] = (
    annual_hourly_analysis[
        "Timestamp"
    ].dt.to_period("M").astype(str)
)

annual_hourly_analysis["Hour of Day"] = (
    annual_hourly_analysis[
        "Timestamp"
    ].dt.hour
)

annual_hourly_analysis[
    "Positive Congestion Cost"
] = (
    annual_hourly_analysis[
        "Congestion Cost $/h"
    ]
    > CONGESTION_NUMERICAL_TOLERANCE
)

annual_hourly_analysis[
    "C6 Binding"
] = (
    annual_hourly_analysis[
        "C6 Loading %"
    ]
    >= 100.0 - 1e-5
)


# ---------------------------------------------------------------------
# 3. Monthly case-level results
# ---------------------------------------------------------------------

monthly_case_results = (
    annual_hourly_analysis
    .groupby(
        ["Month", "Case"],
        sort=True,
    )
    .agg(
        Operating_Cost_Dollars=(
            "Operating Cost $/h",
            "sum",
        ),
        Copper_Plate_Cost_Dollars=(
            "Copper-Plate Cost $/h",
            "sum",
        ),
        Congestion_Cost_Dollars=(
            "Congestion Cost $/h",
            "sum",
        ),
        DC_Energy_MWh=(
            "DC Power MW",
            "sum",
        ),
        VRE_Available_MWh=(
            "VRE Available MW",
            "sum",
        ),
        VRE_Dispatched_MWh=(
            "VRE Dispatched MW",
            "sum",
        ),
        VRE_Curtailment_MWh=(
            "VRE Curtailment MW",
            "sum",
        ),
        Positive_Congestion_Hours=(
            "Positive Congestion Cost",
            "sum",
        ),
        C6_Binding_Hours=(
            "C6 Binding",
            "sum",
        ),
        Mean_DC_Power_MW=(
            "DC Power MW",
            "mean",
        ),
        Minimum_DC_Power_MW=(
            "DC Power MW",
            "min",
        ),
        Maximum_DC_Power_MW=(
            "DC Power MW",
            "max",
        ),
    )
)

monthly_factorial_rows = []

for month in sorted(
    annual_hourly_analysis[
        "Month"
    ].unique()
):
    month_data = monthly_case_results.loc[
        month
    ]

    operating_effects = (
        calculate_factorial_effects(
            month_data[
                "Operating_Cost_Dollars"
            ]
        )
    )

    copper_effects = (
        calculate_factorial_effects(
            month_data[
                "Copper_Plate_Cost_Dollars"
            ]
        )
    )

    congestion_effects = (
        calculate_factorial_effects(
            month_data[
                "Congestion_Cost_Dollars"
            ]
        )
    )

    curtailment_effects = (
        calculate_factorial_effects(
            month_data[
                "VRE_Curtailment_MWh"
            ]
        )
    )

    monthly_factorial_rows.append(
        {
            "Month": month,

            "Flexibility Value, Original $":
                operating_effects[
                    "Flexibility value, original network"
                ],

            "Flexibility Value, Reconductored $":
                operating_effects[
                    "Flexibility value, reconductored network"
                ],

            "Reconductoring Value, Inflexible $":
                operating_effects[
                    "Reconductoring value, inflexible load"
                ],

            "Reconductoring Value, Flexible $":
                operating_effects[
                    "Reconductoring value, flexible load"
                ],

            "Combined Value $":
                operating_effects[
                    "Combined value relative to C0"
                ],

            "Interaction $":
                operating_effects[
                    "Combined-savings interaction"
                ],

            "Combined Copper-Plate Effect $":
                copper_effects[
                    "Combined value relative to C0"
                ],

            "Combined Congestion Effect $":
                congestion_effects[
                    "Combined value relative to C0"
                ],

            "Combined Curtailment Reduction MWh":
                curtailment_effects[
                    "Combined value relative to C0"
                ],
        }
    )

monthly_factorial_effects = (
    pd.DataFrame(
        monthly_factorial_rows
    )
    .set_index("Month")
)


# ---------------------------------------------------------------------
# 4. Daily case-level results and factorial effects
# ---------------------------------------------------------------------

daily_case_mechanisms = (
    annual_hourly_analysis
    .groupby(
        ["Day", "Case"],
        sort=True,
    )
    .agg(
        Operating_Cost_Dollars=(
            "Operating Cost $/h",
            "sum",
        ),
        Copper_Plate_Cost_Dollars=(
            "Copper-Plate Cost $/h",
            "sum",
        ),
        Congestion_Cost_Dollars=(
            "Congestion Cost $/h",
            "sum",
        ),
        VRE_Curtailment_MWh=(
            "VRE Curtailment MW",
            "sum",
        ),
        DC_Energy_MWh=(
            "DC Power MW",
            "sum",
        ),
        Minimum_DC_Power_MW=(
            "DC Power MW",
            "min",
        ),
        Maximum_DC_Power_MW=(
            "DC Power MW",
            "max",
        ),
        C6_Binding_Hours=(
            "C6 Binding",
            "sum",
        ),
        Positive_Congestion_Hours=(
            "Positive Congestion Cost",
            "sum",
        ),
    )
)

daily_factorial_rows = []

for day in annual_days:
    day_data = daily_case_mechanisms.loc[
        day
    ]

    operating_effects = (
        calculate_factorial_effects(
            day_data[
                "Operating_Cost_Dollars"
            ]
        )
    )

    congestion_effects = (
        calculate_factorial_effects(
            day_data[
                "Congestion_Cost_Dollars"
            ]
        )
    )

    curtailment_effects = (
        calculate_factorial_effects(
            day_data[
                "VRE_Curtailment_MWh"
            ]
        )
    )

    daily_factorial_rows.append(
        {
            "Day": day,

            "Flexibility Value, Original $":
                operating_effects[
                    "Flexibility value, original network"
                ],

            "Flexibility Value, Reconductored $":
                operating_effects[
                    "Flexibility value, reconductored network"
                ],

            "Reconductoring Value, Inflexible $":
                operating_effects[
                    "Reconductoring value, inflexible load"
                ],

            "Reconductoring Value, Flexible $":
                operating_effects[
                    "Reconductoring value, flexible load"
                ],

            "Combined Value $":
                operating_effects[
                    "Combined value relative to C0"
                ],

            "Interaction $":
                operating_effects[
                    "Combined-savings interaction"
                ],

            "Combined Congestion Reduction $":
                congestion_effects[
                    "Combined value relative to C0"
                ],

            "Combined Curtailment Reduction MWh":
                curtailment_effects[
                    "Combined value relative to C0"
                ],
        }
    )

daily_factorial_effects = (
    pd.DataFrame(
        daily_factorial_rows
    )
    .set_index("Day")
)

daily_effect_columns = [
    "Flexibility Value, Original $",
    "Flexibility Value, Reconductored $",
    "Reconductoring Value, Inflexible $",
    "Reconductoring Value, Flexible $",
    "Combined Value $",
    "Interaction $",
]

daily_effect_distribution = (
    daily_factorial_effects[
        daily_effect_columns
    ]
    .describe(
        percentiles=[
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
        ]
    )
    .transpose()
)

interaction_sign_summary = pd.DataFrame(
    {
        "Days": {
            "Positive interaction":
                int(
                    (
                        daily_factorial_effects[
                            "Interaction $"
                        ]
                        > 1e-4
                    ).sum()
                ),

            "Approximately zero interaction":
                int(
                    (
                        daily_factorial_effects[
                            "Interaction $"
                        ].abs()
                        <= 1e-4
                    ).sum()
                ),

            "Negative interaction":
                int(
                    (
                        daily_factorial_effects[
                            "Interaction $"
                        ]
                        < -1e-4
                    ).sum()
                ),

            "Positive original-network flexibility value":
                int(
                    (
                        daily_factorial_effects[
                            "Flexibility Value, Original $"
                        ]
                        > 1e-4
                    ).sum()
                ),

            "Positive combined value":
                int(
                    (
                        daily_factorial_effects[
                            "Combined Value $"
                        ]
                        > 1e-4
                    ).sum()
                ),
        }
    }
)


# ---------------------------------------------------------------------
# 5. Hour-of-day flexible-workload behavior
# ---------------------------------------------------------------------

hourly_schedule_rows = []

for case_code in ["CF", "CFR"]:
    case_schedule = (
        annual_hourly_analysis.loc[
            annual_hourly_analysis[
                "Case"
            ].eq(case_code)
        ]
    )

    for hour_of_day, hour_data in (
        case_schedule.groupby(
            "Hour of Day"
        )
    ):
        hourly_dc_values = (
            hour_data[
                "DC Power MW"
            ].to_numpy(dtype=float)
        )

        hourly_schedule_rows.append(
            {
                "Case": case_code,
                "Hour of Day":
                    int(hour_of_day),
                "Mean DC Power MW":
                    float(
                        hourly_dc_values.mean()
                    ),
                "Median DC Power MW":
                    float(
                        np.median(
                            hourly_dc_values
                        )
                    ),
                "Standard Deviation MW":
                    float(
                        hourly_dc_values.std(
                            ddof=0
                        )
                    ),
                "Share at 50 MW %":
                    float(
                        100.0
                        * np.isclose(
                            hourly_dc_values,
                            DAILY_DC_MINIMUM_MW,
                            atol=1e-5,
                        ).mean()
                    ),
                "Share at 150 MW %":
                    float(
                        100.0
                        * np.isclose(
                            hourly_dc_values,
                            DAILY_DC_MAXIMUM_MW,
                            atol=1e-5,
                        ).mean()
                    ),
                "Mean Native Demand MW":
                    float(
                        hour_data[
                            "Native Demand MW"
                        ].mean()
                    ),
                "Mean VRE Availability MW":
                    float(
                        hour_data[
                            "VRE Available MW"
                        ].mean()
                    ),
                "Mean C6 Loading %":
                    float(
                        hour_data[
                            "C6 Loading %"
                        ].mean()
                    ),
            }
        )

average_hourly_workload_profile = (
    pd.DataFrame(
        hourly_schedule_rows
    )
    .set_index(
        ["Case", "Hour of Day"]
    )
    .sort_index()
)


# ---------------------------------------------------------------------
# 6. Annual generation by fuel
# ---------------------------------------------------------------------

generator_fuel_by_position = (
    generator_metadata
    .reindex(GENERATOR_IDS)[
        "Fuel"
    ]
    .astype(str)
    .str.strip()
)

annual_generation_by_fuel_rows = []

for case_position, case_code in enumerate(
    ANNUAL_CASES
):
    for fuel_name in sorted(
        generator_fuel_by_position.unique()
    ):
        fuel_generator_positions = np.flatnonzero(
            generator_fuel_by_position
            .eq(fuel_name)
            .to_numpy()
        )

        annual_fuel_generation = float(
            generation_array[
                case_position,
                :,
                fuel_generator_positions,
            ].sum()
        )

        annual_generation_by_fuel_rows.append(
            {
                "Case": case_code,
                "Fuel": fuel_name,
                "Annual Generation MWh":
                    annual_fuel_generation,
            }
        )

annual_generation_by_fuel = (
    pd.DataFrame(
        annual_generation_by_fuel_rows
    )
    .pivot(
        index="Fuel",
        columns="Case",
        values="Annual Generation MWh",
    )
    .reindex(columns=ANNUAL_CASES)
    .fillna(0.0)
)

generation_change_relative_to_c0 = (
    annual_generation_by_fuel
    .subtract(
        annual_generation_by_fuel[
            "C0"
        ],
        axis=0,
    )
)


# ---------------------------------------------------------------------
# 7. Network-wide branch loading and congestion migration
# ---------------------------------------------------------------------

native_branch_ratings_array = np.array(
    [
        notebook5_native_branch_rating[
            branch_uid
        ]
        for branch_uid in BRANCH_IDS
    ],
    dtype=float,
)

case_branch_ratings = np.tile(
    native_branch_ratings_array[
        None,
        :,
    ],
    (
        number_of_cases,
        1,
    ),
)

case_branch_ratings[
    annual_case_position["CR"],
    c6_branch_position,
] = SELECTED_C6_RATING_MW

case_branch_ratings[
    annual_case_position["CFR"],
    c6_branch_position,
] = SELECTED_C6_RATING_MW

all_branch_loading_percent = (
    100.0
    * np.abs(
        branch_flow_array
    )
    / case_branch_ratings[
        :,
        None,
        :,
    ]
)

branch_binding_hours = (
    all_branch_loading_percent
    >= 100.0 - 1e-5
).sum(axis=1)

branch_maximum_loading = (
    all_branch_loading_percent.max(
        axis=1
    )
)

branch_mean_loading = (
    all_branch_loading_percent.mean(
        axis=1
    )
)

branch_congestion_rows = []

for case_position, case_code in enumerate(
    ANNUAL_CASES
):
    for branch_number, branch_uid in enumerate(
        BRANCH_IDS
    ):
        branch_congestion_rows.append(
            {
                "Case": case_code,
                "Branch UID": branch_uid,
                "From Bus": int(
                    branch_from_bus[
                        branch_uid
                    ]
                ),
                "To Bus": int(
                    branch_to_bus[
                        branch_uid
                    ]
                ),
                "Rating MW": float(
                    case_branch_ratings[
                        case_position,
                        branch_number,
                    ]
                ),
                "Binding Hours": int(
                    branch_binding_hours[
                        case_position,
                        branch_number,
                    ]
                ),
                "Mean Loading %": float(
                    branch_mean_loading[
                        case_position,
                        branch_number,
                    ]
                ),
                "Maximum Loading %": float(
                    branch_maximum_loading[
                        case_position,
                        branch_number,
                    ]
                ),
            }
        )

annual_branch_congestion_summary = (
    pd.DataFrame(
        branch_congestion_rows
    )
    .set_index(
        ["Case", "Branch UID"]
    )
    .sort_index()
)

top_binding_branches = (
    annual_branch_congestion_summary
    .reset_index()
    .sort_values(
        [
            "Case",
            "Binding Hours",
            "Mean Loading %",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
    .groupby(
        "Case",
        group_keys=False,
    )
    .head(10)
    .set_index(
        ["Case", "Branch UID"]
    )
)


# ---------------------------------------------------------------------
# 8. Cross-checks
# ---------------------------------------------------------------------

monthly_cost_reconstruction = (
    monthly_case_results[
        "Operating_Cost_Dollars"
    ]
    .groupby(level="Case")
    .sum()
    .reindex(ANNUAL_CASES)
)

daily_cost_reconstruction = (
    daily_case_mechanisms[
        "Operating_Cost_Dollars"
    ]
    .groupby(level="Case")
    .sum()
    .reindex(ANNUAL_CASES)
)

assert np.allclose(
    monthly_cost_reconstruction,
    annual_case_summary[
        "Annual Operating Cost $"
    ],
    rtol=0.0,
    atol=1e-3,
)

assert np.allclose(
    daily_cost_reconstruction,
    annual_case_summary[
        "Annual Operating Cost $"
    ],
    rtol=0.0,
    atol=1e-3,
)

fuel_generation_totals = (
    annual_generation_by_fuel.sum(
        axis=0
    )
)

model_generation_totals = pd.Series(
    total_generation_array.sum(
        axis=1
    ),
    index=ANNUAL_CASES,
)

assert np.allclose(
    fuel_generation_totals,
    model_generation_totals,
    rtol=0.0,
    atol=1e-3,
)

assert float(
    all_branch_loading_percent.max()
) <= 100.0 + 1e-5


# ---------------------------------------------------------------------
# 9. Save mechanism tables
# ---------------------------------------------------------------------

monthly_case_results.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "monthly_case_results.csv"
)

monthly_factorial_effects.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "monthly_factorial_effects.csv"
)

daily_case_mechanisms.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "daily_case_mechanisms.csv"
)

daily_factorial_effects.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "daily_factorial_effects.csv"
)

average_hourly_workload_profile.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "average_hourly_workload_profile.csv"
)

annual_generation_by_fuel.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_generation_by_fuel.csv"
)

generation_change_relative_to_c0.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "generation_change_relative_to_c0.csv"
)

annual_branch_congestion_summary.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_branch_congestion_summary.csv"
)

annual_hourly_case_summary.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "annual_hourly_case_summary.csv"
)


# ---------------------------------------------------------------------
# 10. Display core mechanism results
# ---------------------------------------------------------------------

print("\nMonthly treatment effects:")
display(monthly_factorial_effects)

print("\nDaily treatment-effect distribution:")
display(daily_effect_distribution)

print("\nDaily interaction and benefit signs:")
display(interaction_sign_summary)

print("\nAverage flexible workload by hour of day:")
display(average_hourly_workload_profile)

print("\nAnnual generation by fuel:")
display(annual_generation_by_fuel)

print("\nGeneration change relative to C0:")
display(generation_change_relative_to_c0)

print("\nMost frequently binding branches in each case:")
display(top_binding_branches)

print(
    "\nThe annual temporal, generation-mix, and "
    "network-congestion mechanism analysis passed "
    "all reconstruction checks."
)

## Daily drivers of flexibility and reconductoring value

Daily treatment effects are compared with native-system demand, renewable
availability, curtailment, C6 loading, and congestion conditions.

Spearman rank correlations are used because treatment benefits are strongly
skewed and may have nonlinear relationships with system conditions. Days are
also classified by native congestion and renewable-availability regimes to
show where each treatment creates the most value.

In [ ]:
# Identify the daily system conditions that drive treatment value.

import numpy as np
import pandas as pd

from IPython.display import display


# ---------------------------------------------------------------------
# 1. Construct native daily operating conditions from C0
# ---------------------------------------------------------------------

c0_hourly_conditions = (
    annual_hourly_analysis.loc[
        annual_hourly_analysis[
            "Case"
        ].eq("C0")
    ]
)

daily_native_context = (
    c0_hourly_conditions
    .groupby(
        "Day",
        sort=True,
    )
    .agg(
        Mean_Native_Demand_MW=(
            "Native Demand MW",
            "mean",
        ),
        Maximum_Native_Demand_MW=(
            "Native Demand MW",
            "max",
        ),
        Minimum_Native_Demand_MW=(
            "Native Demand MW",
            "min",
        ),
        Native_Demand_MWh=(
            "Native Demand MW",
            "sum",
        ),
        VRE_Availability_MWh=(
            "VRE Available MW",
            "sum",
        ),
        C0_VRE_Curtailment_MWh=(
            "VRE Curtailment MW",
            "sum",
        ),
        C0_Operating_Cost_Dollars=(
            "Operating Cost $/h",
            "sum",
        ),
        C0_Congestion_Cost_Dollars=(
            "Congestion Cost $/h",
            "sum",
        ),
        C0_Positive_Congestion_Hours=(
            "Positive Congestion Cost",
            "sum",
        ),
        C0_C6_Binding_Hours=(
            "C6 Binding",
            "sum",
        ),
        C0_Mean_C6_Loading_Percent=(
            "C6 Loading %",
            "mean",
        ),
        C0_Maximum_C6_Loading_Percent=(
            "C6 Loading %",
            "max",
        ),
    )
)

daily_driver_analysis = (
    daily_native_context
    .join(
        daily_factorial_effects,
        how="inner",
    )
)

daily_driver_analysis["Month"] = (
    daily_driver_analysis.index
    .to_period("M")
    .astype(str)
)

assert len(daily_driver_analysis) == 366
assert not daily_driver_analysis.isna().any().any()


# ---------------------------------------------------------------------
# 2. Spearman rank correlations
# ---------------------------------------------------------------------

driver_columns = [
    "Mean_Native_Demand_MW",
    "Maximum_Native_Demand_MW",
    "Native_Demand_MWh",
    "VRE_Availability_MWh",
    "C0_VRE_Curtailment_MWh",
    "C0_Operating_Cost_Dollars",
    "C0_Congestion_Cost_Dollars",
    "C0_Positive_Congestion_Hours",
    "C0_C6_Binding_Hours",
    "C0_Mean_C6_Loading_Percent",
    "C0_Maximum_C6_Loading_Percent",
]

effect_columns = [
    "Flexibility Value, Original $",
    "Flexibility Value, Reconductored $",
    "Reconductoring Value, Inflexible $",
    "Reconductoring Value, Flexible $",
    "Combined Value $",
    "Interaction $",
    "Combined Congestion Reduction $",
    "Combined Curtailment Reduction MWh",
]

daily_driver_correlations = (
    daily_driver_analysis[
        driver_columns
        + effect_columns
    ]
    .corr(method="spearman")
    .loc[
        driver_columns,
        effect_columns,
    ]
)


# ---------------------------------------------------------------------
# 3. Native congestion regimes
# ---------------------------------------------------------------------

daily_driver_analysis[
    "Native Congestion Regime"
] = "No material congestion"

positive_congestion_mask = (
    daily_driver_analysis[
        "C0_Congestion_Cost_Dollars"
    ]
    > CONGESTION_NUMERICAL_TOLERANCE
)

positive_congestion_values = (
    daily_driver_analysis.loc[
        positive_congestion_mask,
        "C0_Congestion_Cost_Dollars",
    ]
)

positive_congestion_rank = (
    positive_congestion_values
    .rank(
        method="first"
    )
)

positive_congestion_regimes = pd.qcut(
    positive_congestion_rank,
    q=4,
    labels=[
        "Low positive congestion",
        "Moderate congestion",
        "High congestion",
        "Extreme congestion",
    ],
)

daily_driver_analysis.loc[
    positive_congestion_mask,
    "Native Congestion Regime",
] = positive_congestion_regimes.astype(str)

congestion_regime_order = [
    "No material congestion",
    "Low positive congestion",
    "Moderate congestion",
    "High congestion",
    "Extreme congestion",
]

daily_driver_analysis[
    "Native Congestion Regime"
] = pd.Categorical(
    daily_driver_analysis[
        "Native Congestion Regime"
    ],
    categories=congestion_regime_order,
    ordered=True,
)

congestion_regime_summary = (
    daily_driver_analysis
    .groupby(
        "Native Congestion Regime",
        observed=True,
    )
    .agg(
        Days=(
            "Combined Value $",
            "size",
        ),
        Mean_C0_Congestion_Cost_Dollars=(
            "C0_Congestion_Cost_Dollars",
            "mean",
        ),
        Mean_Flexibility_Value_Original_Dollars=(
            "Flexibility Value, Original $",
            "mean",
        ),
        Mean_Reconductoring_Value_Inflexible_Dollars=(
            "Reconductoring Value, Inflexible $",
            "mean",
        ),
        Mean_Combined_Value_Dollars=(
            "Combined Value $",
            "mean",
        ),
        Mean_Interaction_Dollars=(
            "Interaction $",
            "mean",
        ),
        Mean_Curtailment_Reduction_MWh=(
            "Combined Curtailment Reduction MWh",
            "mean",
        ),
    )
)


# ---------------------------------------------------------------------
# 4. Renewable-availability regimes
# ---------------------------------------------------------------------

vre_availability_rank = (
    daily_driver_analysis[
        "VRE_Availability_MWh"
    ]
    .rank(
        method="first"
    )
)

daily_driver_analysis[
    "VRE Availability Regime"
] = pd.qcut(
    vre_availability_rank,
    q=4,
    labels=[
        "Lowest VRE quartile",
        "Low-middle VRE quartile",
        "High-middle VRE quartile",
        "Highest VRE quartile",
    ],
)

vre_regime_summary = (
    daily_driver_analysis
    .groupby(
        "VRE Availability Regime",
        observed=True,
    )
    .agg(
        Days=(
            "Combined Value $",
            "size",
        ),
        Mean_VRE_Availability_MWh=(
            "VRE_Availability_MWh",
            "mean",
        ),
        Mean_C0_Curtailment_MWh=(
            "C0_VRE_Curtailment_MWh",
            "mean",
        ),
        Mean_Flexibility_Value_Original_Dollars=(
            "Flexibility Value, Original $",
            "mean",
        ),
        Mean_Reconductoring_Value_Inflexible_Dollars=(
            "Reconductoring Value, Inflexible $",
            "mean",
        ),
        Mean_Combined_Value_Dollars=(
            "Combined Value $",
            "mean",
        ),
        Mean_Curtailment_Reduction_MWh=(
            "Combined Curtailment Reduction MWh",
            "mean",
        ),
    )
)


# ---------------------------------------------------------------------
# 5. Interaction-sign regimes
# ---------------------------------------------------------------------

interaction_tolerance = 1e-4

daily_driver_analysis[
    "Interaction Classification"
] = np.select(
    [
        daily_driver_analysis[
            "Interaction $"
        ] > interaction_tolerance,

        daily_driver_analysis[
            "Interaction $"
        ] < -interaction_tolerance,
    ],
    [
        "Positive interaction",
        "Negative interaction",
    ],
    default="Approximately zero",
)

interaction_driver_summary = (
    daily_driver_analysis
    .groupby(
        "Interaction Classification"
    )
    .agg(
        Days=(
            "Interaction $",
            "size",
        ),
        Mean_Interaction_Dollars=(
            "Interaction $",
            "mean",
        ),
        Mean_Native_Demand_MW=(
            "Mean_Native_Demand_MW",
            "mean",
        ),
        Mean_VRE_Availability_MWh=(
            "VRE_Availability_MWh",
            "mean",
        ),
        Mean_C0_Curtailment_MWh=(
            "C0_VRE_Curtailment_MWh",
            "mean",
        ),
        Mean_C0_Congestion_Cost_Dollars=(
            "C0_Congestion_Cost_Dollars",
            "mean",
        ),
        Mean_C0_C6_Binding_Hours=(
            "C0_C6_Binding_Hours",
            "mean",
        ),
        Mean_Flexibility_Value_Original_Dollars=(
            "Flexibility Value, Original $",
            "mean",
        ),
        Mean_Reconductoring_Value_Inflexible_Dollars=(
            "Reconductoring Value, Inflexible $",
            "mean",
        ),
    )
)


# ---------------------------------------------------------------------
# 6. Extreme-value days
# ---------------------------------------------------------------------

highest_combined_value_days = (
    daily_driver_analysis
    .nlargest(
        10,
        "Combined Value $",
    )[
        [
            "Combined Value $",
            "Flexibility Value, Original $",
            "Reconductoring Value, Inflexible $",
            "Interaction $",
            "C0_Congestion_Cost_Dollars",
            "C0_VRE_Curtailment_MWh",
            "C0_C6_Binding_Hours",
            "VRE_Availability_MWh",
        ]
    ]
)

most_negative_interaction_days = (
    daily_driver_analysis
    .nsmallest(
        10,
        "Interaction $",
    )[
        [
            "Interaction $",
            "Combined Value $",
            "Flexibility Value, Original $",
            "Reconductoring Value, Inflexible $",
            "C0_Congestion_Cost_Dollars",
            "C0_VRE_Curtailment_MWh",
            "C0_C6_Binding_Hours",
        ]
    ]
)

most_positive_interaction_days = (
    daily_driver_analysis
    .nlargest(
        10,
        "Interaction $",
    )[
        [
            "Interaction $",
            "Combined Value $",
            "Flexibility Value, Original $",
            "Reconductoring Value, Inflexible $",
            "C0_Congestion_Cost_Dollars",
            "C0_VRE_Curtailment_MWh",
            "C0_C6_Binding_Hours",
        ]
    ]
)


# ---------------------------------------------------------------------
# 7. Validate daily-to-annual treatment totals
# ---------------------------------------------------------------------

daily_to_annual_mapping = {
    "Flexibility Value, Original $":
        "Flexibility value, original network",

    "Flexibility Value, Reconductored $":
        "Flexibility value, reconductored network",

    "Reconductoring Value, Inflexible $":
        "Reconductoring value, inflexible load",

    "Reconductoring Value, Flexible $":
        "Reconductoring value, flexible load",

    "Combined Value $":
        "Combined value relative to C0",

    "Interaction $":
        "Combined-savings interaction",
}

daily_annual_reconstruction_rows = []

for (
    daily_column,
    annual_effect_name,
) in daily_to_annual_mapping.items():
    daily_total = float(
        daily_driver_analysis[
            daily_column
        ].sum()
    )

    annual_total = float(
        annual_factorial_effects.loc[
            annual_effect_name,
            "Value $",
        ]
    )

    daily_annual_reconstruction_rows.append(
        {
            "Effect": annual_effect_name,
            "Daily Sum $": daily_total,
            "Annual Effect $": annual_total,
            "Absolute Difference $": abs(
                daily_total
                - annual_total
            ),
        }
    )

daily_annual_reconstruction = (
    pd.DataFrame(
        daily_annual_reconstruction_rows
    )
    .set_index("Effect")
)

assert (
    daily_annual_reconstruction[
        "Absolute Difference $"
    ].max()
    <= 1e-3
)


# ---------------------------------------------------------------------
# 8. Save driver-analysis tables
# ---------------------------------------------------------------------

daily_driver_analysis.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "daily_driver_analysis.csv"
)

daily_driver_correlations.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "daily_driver_correlations.csv"
)

congestion_regime_summary.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "congestion_regime_summary.csv"
)

vre_regime_summary.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "vre_regime_summary.csv"
)

interaction_driver_summary.to_csv(
    ANNUAL_OUTPUT_DIRECTORY
    / "interaction_driver_summary.csv"
)


# ---------------------------------------------------------------------
# 9. Display results
# ---------------------------------------------------------------------

print("\nSpearman correlations: daily drivers versus treatment effects")
display(daily_driver_correlations)

print("\nTreatment value by native congestion regime:")
display(congestion_regime_summary)

print("\nTreatment value by VRE-availability regime:")
display(vre_regime_summary)

print("\nSystem conditions by interaction classification:")
display(interaction_driver_summary)

print("\nTen days with the greatest combined value:")
display(highest_combined_value_days)

print("\nTen most negative-interaction days:")
display(most_negative_interaction_days)

print("\nTen most positive-interaction days:")
display(most_positive_interaction_days)

print("\nDaily-to-annual reconstruction:")
display(daily_annual_reconstruction)

print(
    "\nThe daily driver analysis passed all annual "
    "reconstruction checks."
)

## Baseline experimental conclusions

The complete 2020 experiment demonstrates that both temporal workload
flexibility and targeted transmission reconductoring reduce system operating
costs while preserving the data center's annual energy consumption.

Relative to the original network with a constant 100 MW workload, daily
workload flexibility reduces annual operating cost by approximately
$1.88 million, while increasing renewable utilization by 53.3 GWh.
Reconductoring C6 from 175 MW to 262.5 MW reduces annual operating cost by
approximately $6.87 million and renewable curtailment by 282.4 GWh.

Applying both treatments produces the lowest annual operating cost, with
approximately $8.47 million in gross annual savings and 326.0 GWh less
renewable curtailment relative to the untreated case. Approximately 86% of
the combined economic benefit is attributable to reduced transmission-
congestion cost, while 14% reflects improved temporal generation dispatch.

The combined benefit is smaller than the sum of the treatments' independent
benefits by approximately $274,000. This negative interaction indicates that
workload flexibility and reconductoring are generally partial substitutes:
both alleviate overlapping congestion and renewable-integration problems.
Nevertheless, positive interaction occurs during a smaller set of severely
congested, high-renewable-curtailment days.

Flexible demand follows renewable availability rather than simply minimizing
system demand. Average data-center power approaches 150 MW during solar-rich
morning and midday hours and falls toward 50 MW during the evening peak.
Under the combined treatment, annual wind generation increases by
approximately 315.7 GWh and solar generation by 10.4 GWh, while natural-gas
generation decreases by approximately 296.7 GWh.

Reconductoring substantially relieves C6, but congestion migrates toward other
corridors, particularly CA-1, CB-1, C29, and C2. This demonstrates that a
single transmission upgrade can reduce total congestion cost without
eliminating network constraints.

Daily treatment value is strongly associated with native congestion, C6
binding, renewable availability, and renewable curtailment. It is negatively
associated with native demand, indicating that the treatments are most
valuable during renewable-rich and transmission-constrained conditions rather
than necessarily during peak-load periods.

These results represent gross operating-cost benefits under a linear DC-OPF
with perfect daily workload foresight, no interday workload shifting, no
generator commitment or ramping constraints, and no assigned capital cost for
reconductoring or implementation cost for workload flexibility. Sensitivity
analysis is therefore required before drawing broader investment conclusions.